In [ ]:
"""
================================================================================
CELL 0 — PACKAGE IMPORTS / INSTALLS
Run this first, before Cell 1 (data) and Cell 2 (Q6 model).
torch/pandas/numpy/scikit-learn are already present on Colab/Kaggle by default
-- only pennylane (quantum circuit correctness verification) and kagglehub
(dataset download) typically need installing.
================================================================================
"""

import subprocess, sys

def pip_install(*pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    except subprocess.CalledProcessError:
        # some environments (externally-managed Python) refuse a bare install -- retry with the override.
        # Harmless on Colab/Kaggle, where this branch is never needed.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs], check=True)

pip_install("pennylane", "kagglehub")

# sanity import check -- fails loudly here rather than mid-run later
import math, os, glob, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pennylane as qml
import kagglehub

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("pennylane:", qml.__version__)
print("pandas:", pd.__version__, "| numpy:", np.__version__)
print("All packages imported OK.")

torch: 2.11.0+cpu | cuda available: False
pennylane: 0.45.1
pandas: 2.2.3 | numpy: 2.1.3
All packages imported OK.


In [ ]:
"""
================================================================================
CELL 1 — DATA ACQUISITION + PROCESSING
Same methodology as all prior cells (Kaggle download, phi-unit detection,
target construction, standardization, graph construction) with ONE change:
SPLIT is now 20% train / 20% val / 60% test -- the 60% that was previously
discarded as "unused" is now the test set instead. Nothing is thrown away.
================================================================================
"""

import os, math, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# ==============================================================================
# 0. CONFIGURATION  (unchanged from all prior cells)
# ==============================================================================
SEED = 42 # 1 60 100 120
USE_PT_FLOOR = True
PHI_UNITS = "auto"

KAGGLE_DATASET = "prateekagnihotri/cms-trigger-dataset"
LOCAL_FALLBACK_DIR = "/content"

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

N_NODES = 4
N_FEATS = 7
D_NODE = 4
N_QUBITS = 4
N_EDGES = N_NODES * (N_NODES - 1)
QDIM = 2 ** N_QUBITS
assert D_NODE == N_QUBITS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ==============================================================================
# 1. DATA ACQUISITION  (unchanged)
# ==============================================================================
STATIONS = [0, 2, 3, 4]
STATION_FEATS = ["Phi", "Theta", "BendingAngle", "TimeInfo", "RingNumber", "Front", "Mask"]
REQUIRED_STATION_COLS = [f"{feat}_{st}" for st in STATIONS for feat in STATION_FEATS]
REQUIRED_COLS = REQUIRED_STATION_COLS + ["q/pt"]

def find_csv_with_required_columns(root_dir, required_cols, max_files=200):
    for path in glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True)[:max_files]:
        try:
            header = pd.read_csv(path, nrows=0).columns.tolist()
            if all(c in header for c in required_cols):
                return path
        except Exception:
            continue
    return None

def download_dataset():
    try:
        import kagglehub
        path = kagglehub.dataset_download(KAGGLE_DATASET)
        print("kagglehub download path:", path)
        found = find_csv_with_required_columns(path, REQUIRED_COLS)
        if found:
            return found
        any_csv = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
        if any_csv:
            return any_csv[0]
    except Exception as e:
        print("kagglehub path unavailable or failed:", repr(e))
    try:
        import kaggle
        dl_dir = "/content/cms_trigger_dataset"
        os.makedirs(dl_dir, exist_ok=True)
        kaggle.api.dataset_download_files(KAGGLE_DATASET, path=dl_dir, unzip=True)
        found = find_csv_with_required_columns(dl_dir, REQUIRED_COLS)
        if found:
            return found
    except Exception as e:
        print("kaggle API path unavailable or failed:", repr(e))
    found = find_csv_with_required_columns(LOCAL_FALLBACK_DIR, REQUIRED_COLS)
    if found:
        return found
    raise FileNotFoundError(f"Could not locate {KAGGLE_DATASET} CSV automatically.")

csv_path = download_dataset()
print("Discovered CSV file:", csv_path)
df_raw = pd.read_csv(csv_path)
missing = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")
print("Original dataset size:", len(df_raw))

# ==============================================================================
# 2. PHI UNIT DETECTION + TARGET CONSTRUCTION  (unchanged)
# ==============================================================================
_phi_cols = [f"Phi_{st}" for st in STATIONS]
_phi_vals = pd.to_numeric(df_raw[_phi_cols].stack(), errors="coerce")
_phi_vals = _phi_vals[np.isfinite(_phi_vals)]
_phi_span = float(_phi_vals.max() - _phi_vals.min())
if PHI_UNITS == "auto":
    PHI_IS_RADIANS = _phi_span <= 2 * math.pi + 1e-6
elif PHI_UNITS == "radians":
    PHI_IS_RADIANS = True
else:
    PHI_IS_RADIANS = False
print("Phi treated as:", "radians (wrapped)" if PHI_IS_RADIANS else "integer bins (plain diff)")

df = df_raw.copy()
with np.errstate(divide="ignore", invalid="ignore"):
    df["pT"] = np.abs(1.0 / df["q/pt"])

invalid_mask = (
    df["pT"].isna() | np.isinf(df["pT"]) | (df["pT"] <= 0)
    | df[REQUIRED_STATION_COLS].isna().any(axis=1)
    | np.isinf(df[REQUIRED_STATION_COLS]).any(axis=1)
)
df = df.loc[~invalid_mask].reset_index(drop=True)
print("Remaining rows after cleaning:", len(df))

# ==============================================================================
# 3. SPLIT — CHANGED: 20% train / 20% val / 60% test, nothing discarded
# ==============================================================================
N_total = len(df)
train_df, _rest1 = train_test_split(df, train_size=0.20, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(_rest1, train_size=0.25, random_state=SEED, shuffle=True)
# _rest1 is 80% of total; 0.25 of that = 20% of total -> val. The remaining 75% of
# _rest1 (=60% of total) is test_df in full -- no further split, no unused portion.
print(f"Total rows: {N_total} | Train: {len(train_df)} ({100*len(train_df)/N_total:.1f}%) "
      f"| Val: {len(val_df)} ({100*len(val_df)/N_total:.1f}%) "
      f"| Test: {len(test_df)} ({100*len(test_df)/N_total:.1f}%)")
assert len(train_df) + len(val_df) + len(test_df) == N_total, "split must cover the full dataset now"

# ==============================================================================
# 4. PREPROCESSING  (unchanged)
# ==============================================================================
def station_cols(st):
    return [f"{feat}_{st}" for feat in STATION_FEATS]

def build_X(d):
    return np.stack([d[station_cols(st)].values for st in STATIONS], axis=1).astype(np.float32)

X_train_raw = build_X(train_df)
X_val_raw   = build_X(val_df)
X_test_raw  = build_X(test_df)

X_mu = X_train_raw.mean(axis=0, keepdims=True)
X_sigma = np.maximum(X_train_raw.std(axis=0, keepdims=True), 1e-6)

def standardize(x):
    return (x - X_mu) / X_sigma

X_train, X_val, X_test = standardize(X_train_raw), standardize(X_val_raw), standardize(X_test_raw)

PHI_IDX = STATION_FEATS.index("Phi")
BA_IDX = STATION_FEATS.index("BendingAngle")
phi_train, ba_train = X_train_raw[:, :, PHI_IDX], X_train_raw[:, :, BA_IDX]
phi_val, ba_val     = X_val_raw[:, :, PHI_IDX], X_val_raw[:, :, BA_IDX]
phi_test, ba_test   = X_test_raw[:, :, PHI_IDX], X_test_raw[:, :, BA_IDX]

y_train = np.log1p(train_df["pT"].values.astype(np.float32))
y_val   = np.log1p(val_df["pT"].values.astype(np.float32))
y_test  = np.log1p(test_df["pT"].values.astype(np.float32))
pT_train = train_df["pT"].values.astype(np.float32)
pT_test  = test_df["pT"].values.astype(np.float32)

Y_MEAN = float(y_train.mean())
Y_STD  = float(y_train.std() + 1e-8)
PT_FLOOR_TRAIN = float(train_df["pT"].min())
PT_CLAMP_LO = PT_FLOOR_TRAIN if USE_PT_FLOOR else 0.0
print(f"Target log(1+pT): train mean={Y_MEAN:.4f} std={Y_STD:.4f}")

# ==============================================================================
# 5. GRAPH CONSTRUCTION  (unchanged)
# ==============================================================================
EDGES = [(i, j) for i in range(N_NODES) for j in range(N_NODES) if j != i]
edge_recv = torch.tensor([e[0] for e in EDGES], dtype=torch.long)
edge_send = torch.tensor([e[1] for e in EDGES], dtype=torch.long)
assert len(EDGES) == 12

def compute_r(phi, ba, recv, send):
    phi_i, phi_j = phi[:, recv], phi[:, send]
    ba_i, ba_j = ba[:, recv], ba[:, send]
    if PHI_IS_RADIANS:
        dphi = np.arctan2(np.sin(phi_j - phi_i), np.cos(phi_j - phi_i))
    else:
        dphi = phi_j - phi_i
    return np.stack([np.abs(ba_j - ba_i), np.abs(dphi)], axis=-1)

r_train_np = compute_r(phi_train, ba_train, edge_recv.numpy(), edge_send.numpy())
r_mean = torch.tensor(r_train_np.reshape(-1, 2).mean(axis=0), dtype=torch.float32)
r_std  = torch.tensor(np.maximum(r_train_np.reshape(-1, 2).std(axis=0), 1e-6), dtype=torch.float32)
r_mean_d, r_std_d = r_mean.to(DEVICE), r_std.to(DEVICE)

print("\nData ready: X_train/X_val/X_test, phi_*/ba_* , y_*, pT_train/pT_test, "
      "edge_recv/edge_send, r_mean/r_std (+ _d device copies), Y_MEAN/Y_STD, PT_CLAMP_LO.")

Device: cpu


100%|██████████| 50.6M/50.6M [00:01<00:00, 41.5MB/s]

Extracting files...


kagglehub download path: /root/.cache/kagglehub/datasets/prateekagnihotri/cms-trigger-dataset/versions/1
Discovered CSV file: /root/.cache/kagglehub/datasets/prateekagnihotri/cms-trigger-dataset/versions/1/CMS_trigger.csv
Original dataset size: 1179356
Phi treated as: integer bins (plain diff)
Remaining rows after cleaning: 1179356
Total rows: 1179356 | Train: 235871 (20.0%) | Val: 235871 (20.0%) | Test: 707614 (60.0%)
Target log(1+pT): train mean=2.0636 std=0.9417

Data ready: X_train/X_val/X_test, phi_*/ba_* , y_*, pT_train/pT_test, edge_recv/edge_send, r_mean/r_std (+ _d device copies), Y_MEAN/Y_STD, PT_CLAMP_LO.


In [ ]:
"""
================================================================================
CELL 3 -- PI-QGAT, 144-parameter model (quantum interaction block)          [FAST JAX BACKEND, closed-form circuit]
Needs: Cell 0 (installs) and Cell 1 (data). Independent of Cell 2.

WHAT THIS CELL DOES
  * Trains the Physics-Informed Quantum Graph Attention Network (PI-QGAT) described below, 5 seeds x 50 epochs, prints
    every epoch, restores the best-validation checkpoint, then evaluates on the 60% test split.
  * Reports log1p(pT) metrics (R2, MAE, MSE, RMSE), pT metrics in GeV (MAE, MSE, RMSE; no R2), a per-pT-class table
    (0-2, 2-4, 4-6, 6-10, 10-20, 20-50, 50-100, >=100 GeV), convergence statistics, training time and INFERENCE time.
  * Verifies the fast circuit against PennyLane before training.

MODEL (144 trainable parameters)
   node_proj 7->4 (32) | pair_proj 8->4 (36) | physics_prior 2->1 (3) | head 4->6->1 (37)
   quantum block: 16 circuit angles + entangle_gate1 4->4 (20, makes the RZZ angles data dependent) = 36
   The old model had 132 parameters; its 2nd RZZ layer (entangle_gate2, 20 params) is a diagonal gate before a
   Z-basis measurement and provably cannot change <Z>, so it was dead weight and is removed (same function).
   The head 4->6->1 (+32 params vs the original 4->1 head) is the only architectural change vs the original cell.

SPEED / SIZE MEASURES (none changes the math)
   whole epoch in one XLA call (lax.scan), exact real-arithmetic 4-qubit simulator, one fused 32x32 matrix per
   variational layer, top-half product for the real input state, inference-time constant folding of the two layer
   matrices ("folded" path, checked equal to the training graph), float16 weight-storage check (144 params = 288 bytes).

NOTE ON RUN TIME: 5 seeds x 50 epochs. Reduce SEEDS for a quick look (e.g. SEEDS = [1]).
================================================================================
"""
SEEDS = [SEED]
VARIANTS_TO_RUN = ["quantum"]
RESULTS_FILE = "piqgat_results_quantum.json"

# ==============================================================================
# 0. IMPORTS  (Cell 0 already installed pennylane / kagglehub; JAX ships with Colab/Kaggle)
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree
import pennylane as qml

# ---- Cell 1 must have been run: these names come from it --------------------------------------------------------
for _n in ["X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES", "QDIM"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12 and QDIM == 16

# ==============================================================================
# 1. HYPER-PARAMETERS  (training recipe identical to the original PI-QGAT cell)
# ==============================================================================
EPOCHS = 50                     # all 50 epochs are run and printed (no early stop unless EARLY_STOPPING=True)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3            # Adam, constant
CLASSICAL_WEIGHT_DECAY = 3e-5   # L2 (added to the gradient, torch-Adam semantics) on every classical weight
QUANTUM_WEIGHT_DECAY = 0.0      # none on the 16 circuit angles
GRAD_CLIP_NORM = 1.0            # global-norm clipping, applied before weight decay
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True       # network predicts standardized log1p(pT); output = y*Y_STD + Y_MEAN
LAMBDA_PHYSICS_ATTN = 0.1       # weight of the physics prior in the attention logit
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6                 # regression head 4 -> 6 -> 1 (tanh)
GAMMA_ESCAPE = True             # nudge RZ angles stuck at sin(gamma)~0 (quantum variant only), as in the original
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EVAL_BATCH_SIZE = 65536         # inference chunk size (no gradients -> same numbers as any other chunk size)
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]       # GeV classes
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
DEVICE_MODE = "auto"            # "auto" = JAX default device | "cpu" = force CPU | "gpu" = force GPU.
                                # Batch-64 training steps are tiny: on many machines CPU is as fast or faster than GPU for TRAINING.
                                # If sec/epoch looks slow and the banner says GPU/TPU, try "cpu" (and vice versa).
if DEVICE_MODE == "cpu":
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
elif DEVICE_MODE == "gpu":
    jax.config.update("jax_default_device", jax.devices("gpu")[0])
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 96)
print(f"Backend: JAX {jax.__version__} | running on {RUN_DEVICE} (all devices: {jax.devices()})   |   seeds {SEEDS}   |   variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} rows   |   epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}, "
      f"wd {CLASSICAL_WEIGHT_DECAY}, clip {GRAD_CLIP_NORM}, SmoothL1 beta {SMOOTH_L1_BETA}")
print("=" * 96)

# ==============================================================================
# 2. FAST EXACT 4-QUBIT SIMULATOR (real arithmetic, batched)
#    state = (re, im) of 16 amplitudes.  circuit per edge:
#      RY(theta_k) --> [RZ(gamma) RY(omega)]_layer0 --> ring IsingZZ(alpha_i(theta)) --> RY(theta_k) (data re-upload)
#      --> [RZ(gamma) RY(omega)]_layer1 --> <Z_k>
#    The variational layer is ONE 16x16 unitary built from the 8 angles of that layer (32x32 real matrix).
#    A 2nd IsingZZ layer would be diagonal in Z and cannot change <Z>: it is omitted (exactly equal output; the 20
#    parameters that used to drive it had zero gradient).  Verified against PennyLane below.
# ==============================================================================
_idx16 = np.arange(QDIM)
_ZZMAT = np.stack([(1 - 2 * ((_idx16 >> (N_QUBITS - 1 - a)) & 1)) * (1 - 2 * ((_idx16 >> (N_QUBITS - 1 - b)) & 1))
                   for a, b in RING_PAIRS]).astype(np.float32)                                   # (4,16)
_ZPL = np.stack([1 - 2 * ((_idx16 >> (N_QUBITS - 1 - k)) & 1) for k in range(N_QUBITS)], 1).astype(np.float32)   # (16,4)

def _var_M_sv(w):
    """w: (4,2) = (omega,gamma) per qubit -> 32x32 real matrix [[Vr^T, Vi^T],[-Vi^T, Vr^T]] of the layer unitary V."""
    om, gam = w[:, 0] / 2, w[:, 1] / 2
    co, si, cg, sg = jnp.cos(om), jnp.sin(om), jnp.cos(gam), jnp.sin(gam)
    gre = jnp.stack([jnp.stack([co * cg, -si * cg], -1), jnp.stack([si * cg, co * cg], -1)], -2)
    gim = jnp.stack([jnp.stack([-co * sg, -si * sg], -1), jnp.stack([-si * sg, co * sg], -1)], -2)
    Vr, Vi = gre[0], gim[0]
    for k in range(1, N_QUBITS):
        Vr, Vi = (jnp.kron(Vr, gre[k]) - jnp.kron(Vi, gim[k]), jnp.kron(Vr, gim[k]) + jnp.kron(Vi, gre[k]))
    return jnp.block([[Vr.T, Vi.T], [-Vi.T, Vr.T]])

def _ry_layer(a, c, s):
    N = a.shape[1]
    for k in range(N_QUBITS):
        r = a.reshape(2, N, 2 ** k, 2, 2 ** (N_QUBITS - 1 - k))
        s0, s1 = r[:, :, :, 0, :], r[:, :, :, 1, :]
        ck, sk = c[None, :, k, None, None], s[None, :, k, None, None]
        a = jnp.stack([ck * s0 - sk * s1, sk * s0 + ck * s1], axis=3).reshape(2, N, QDIM)
    return a

def circuit_core_statevector(theta, M0, M1, a1):
    """REFERENCE ONLY (old 16-amplitude path). theta (N,4) angles, M0/M1 layer matrices, a1 (N,4) RZZ angles -> <Z_k> (N,4)."""
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, N_QUBITS):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(theta.shape[0], -1)
    out = psi @ M0[:QDIM]                                   # input state is real -> only the top half of M0 is needed
    re, im = out[:, :QDIM], out[:, QDIM:]
    ph = -0.5 * (a1 @ jnp.asarray(_ZZMAT, a1.dtype))        # ring IsingZZ = diagonal phase
    cp, sp = jnp.cos(ph), jnp.sin(ph)
    re, im = re * cp - im * sp, re * sp + im * cp
    a = _ry_layer(jnp.stack([re, im]), c, s)                # data re-upload (RY is real)
    out = jnp.concatenate([a[0], a[1]], -1) @ M1
    re, im = out[:, :QDIM], out[:, QDIM:]
    return (re * re + im * im) @ jnp.asarray(_ZPL, re.dtype)

# ------------------------------------------------------------------------------------------------------------------
# FAST PATH: closed-form (light-cone / reduced-density-matrix) evaluation of the SAME circuit.  Exact, not approximate.
#   Every gate except the ring IsingZZ is single-qubit, so the state before the IsingZZ layer is a product state and the
#   IsingZZ layer is diagonal.  Tracing out the other qubits, the reduced state of qubit k after the ring is its own
#   Bloch vector (x, y, z) with the coherence w = x - i*y multiplied by  prod_{j in nbr(k)} (cos a_kj - i <Z_j> sin a_kj)
#   (z unchanged).  The remaining gates on qubit k are single-qubit rotations, and <Z_k> only depends on the reduced
#   state of k, so the whole circuit reduces to 3-D Bloch-vector rotations: O(n_qubits) elementwise ops per edge instead
#   of 16-amplitude complex state vectors and 32x32 matrix products.  This is the light-cone / Heisenberg-picture idea of
#   Pauli propagation (Rudolph et al., arXiv:2505.21606) applied exactly to this depth-1-entangler circuit.
#   Checked against the state-vector simulator AND PennyLane in verify_against_pennylane().
# ------------------------------------------------------------------------------------------------------------------
def _layer(w):
    """Variational-layer parameters (4,2) = (omega, gamma) per qubit. The closed form uses the angles directly."""
    return w

def circuit_core(theta, W0, W1, a1):
    """theta (N,4) angles, W0/W1 (4,2) layer angles, a1 (N,4) RZZ angles (edge i joins qubits i, i+1 mod 4) -> <Z_k> (N,4)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    # RY(theta) |0>  ->  RZ(gamma0)  ->  RY(omega0)
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    # ring IsingZZ: coherence of qubit k picks up (cos a - i z_j sin a) from both ring neighbours j
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)          # <Z> of qubit k+1 and k-1
    fr, fi = ca, -z_next * sa                                                  # edge k     = (k, k+1)
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)        # edge k-1   = (k-1, k)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr                                  # (x - i y) * h
    x, y = wr, -wi
    # data re-upload RY(theta) -> RZ(gamma1) -> RY(omega1) -> <Z>
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL  (variants share everything except the "interaction block" that produces z_exp)
#   h      = node_proj(x)                                  7 -> 4        per station (node)
#   edge   : theta = pi*tanh(pair_proj([h_i, h_j]))        8 -> 4        angles for the interaction block
#   z_exp  = interaction block(theta)                      4 -> 4
#              quantum     : 4-qubit re-uploading ring circuit; 16 angles + entangle_gate1 (4->4, makes the RZZ angles)
#              classical16 : tanh(W theta), W is 4x4 (16 params), no entangle_gate1
#              noquantum   : z_exp = 0 (block and pair_proj removed)
#   P      = physics_prior(|d bending|, |d phi|)           2 -> 1
#   alpha  = softmax_j( mean_k z_exp + lambda * P )        attention over the 3 incoming edges
#   m_i    = sum_j alpha_ij (h_j + z_exp_ij);  h'_i = h_i + m_i;  g = mean_i h'_i
#   y_hat  = Y_STD * head(g) + Y_MEAN,   head = Linear(4,6)-tanh-Linear(6,1)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b) if bias else jnp.zeros((n_out,), jnp.float32)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same keys in every variant -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if variant != "noquantum":
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
    if variant == "quantum":
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    if variant == "classical16":
        bnd = 1.0 / math.sqrt(N_QUBITS)
        P["cW"] = jax.random.uniform(ks[6], (N_QUBITS, N_QUBITS), jnp.float32, -bnd, bnd)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, variant, x, phi, ba, M01=None):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    if variant == "noquantum":
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
        s = LAMBDA_PHYSICS_ATTN * Pp
    else:
        u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        if variant == "quantum":
            a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
            M0, M1 = M01 if M01 is not None else (_layer(P["qw"][0]), _layer(P["qw"][1]))
            z = circuit_core(tf, M0, M1, a1)
        else:
            z = jnp.tanh(tf @ P["cW"].T)
        z = z.reshape(B, N_EDGES, N_QUBITS)
        s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

GROUPS = {"node": "node_proj 7->4", "pair": "pair_proj 8->4", "phys": "physics_prior 2->1", "head1": "head 4->6",
          "head2": "head 6->1", "gate1": "entangle_gate1 4->4 (RZZ angles)", "qw": "circuit angles (2x4x2)",
          "cW": "classical block W 4x4"}
def count_table(P):
    rows = [(GROUPS[k], int(sum(a.size for a in jax.tree_util.tree_leaves(v)))) for k, v in P.items()]
    return rows, sum(n for _, n in rows)

# ==============================================================================
# 4. TRAIN STEP / EPOCH / PREDICT  (one compiled call per epoch via lax.scan)
# ==============================================================================
class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        self.wd_vec = ravel_pytree(wd)[0]
        unravel, wd_vec = self.unravel, self.wd_vec

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), variant, x, p_, b_), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                               # torch Adam L2 decay
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, M01, x, phi, ba):
            return forward(unravel(w), variant, x, phi, ba, M01)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

    def fold(self, w):
        """Inference-time constant folding: precompute the two layer matrices once (quantum variant only)."""
        if self.variant != "quantum":
            return None
        P = self.unravel(w)
        return (_layer(P["qw"][0]), _layer(P["qw"][1]))

# data on device, once
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, M01=None, bs=EVAL_BATCH_SIZE):
    outs = [C.pred(w, M01, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)]
    out = jnp.concatenate(outs); out.block_until_ready()
    return np.asarray(out)

# ==============================================================================
# 5. METRICS  (log scale: R2/MAE/MSE/RMSE.  pT in GeV: MAE/MSE/RMSE only.  per-GeV-class table)
# ==============================================================================
def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

# ==============================================================================
# 6. TRAIN ONE (variant, seed)
# ==============================================================================
def train_one(C, seed):
    variant = C.variant
    P0 = init_params(variant, seed)
    w, _ = ravel_pytree(P0); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, bad, escapes = [], float("inf"), w, 0, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and variant == "quantum":
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        pv = predict_log(C, w, Xva, pva, bva)
        vm, _ = split_metrics(pv, y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep, bad = vm["loss"], jnp.array(w), ep, 0; flag = " *best"
        else:
            bad += 1
        print(f"[{variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s (train {t_tr:.1f} + eval {sec - t_tr:.1f}){flag}"
              + ("   (epoch 1 includes one-time XLA compile)" if ep == 1 and seed == SEEDS[0] else ""), flush=True)
        if EARLY_STOPPING and bad >= EARLY_STOPPING_PATIENCE:
            print(f"   early stop at epoch {ep}"); break
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

# ==============================================================================
# 7. TEST: metrics + per-class table + INFERENCE TIMING
# ==============================================================================
def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def timing_block(C, w, M01):
    n = Xte.shape[0]
    def full_pass(Mx):
        outs = [C.pred(w, Mx, Xte[i:i + EVAL_BATCH_SIZE], pte[i:i + EVAL_BATCH_SIZE], bte[i:i + EVAL_BATCH_SIZE])
                for i in range(0, n, EVAL_BATCH_SIZE)]
        jnp.concatenate(outs).block_until_ready()
    out = {}
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        full_pass(Mx)                                           # warm-up / compile
        tt = _med(lambda: full_pass(Mx), 3)
        out[name] = dict(test_pass_s=tt, us_per_row=1e6 * tt / n, rows_per_s=n / tt)
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        lat = {}
        for b in (1, 64, 1024):
            f = lambda: C.pred(w, Mx, Xte[:b], pte[:b], bte[:b]).block_until_ready()
            f(); lat[b] = 1e3 * _med(f, 200 if b == 1 else 60)
        out["latency_" + name] = lat
    return out

def evaluate_test(C, tr, seed):
    w = tr["w"]; M01 = C.fold(w)
    pred = predict_log(C, w, Xte, pte, bte, M01)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    if M01 is not None:                                          # folded path must equal the training graph
        m["fold_maxdiff"] = float(np.abs(pred - predict_log(C, w, Xte, pte, bte, None)).max())
    w16 = w.astype(jnp.float16).astype(jnp.float32)              # storage-precision check: weights rounded to float16
    m16, _ = split_metrics(predict_log(C, w16, Xte, pte, bte, C.fold(w16)), y_test, pT_test64)
    m["fp16_dR2_log"] = m16["r2_log"] - m["r2_log"]; m["fp16_dMAE_log"] = m16["mae_log"] - m["mae_log"]
    tm = timing_block(C, w, M01)
    return m, bins, tm

# ==============================================================================
# 8. PENNYLANE VERIFICATION (quantum variant only)
# ==============================================================================
def verify_against_pennylane():
    print("-" * 96 + "\nVERIFY: closed-form JAX circuit vs PennyLane default.qubit (full circuit incl. the 2nd IsingZZ layer, float64)\n" + "-" * 96)
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def ref(theta, w, a1, a2):
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[a, b])
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    rs = np.random.default_rng(7)
    th = rs.normal(size=(64, 4)) * 1.5; w = rs.normal(size=(2, 4, 2)) * 2
    a1 = rs.uniform(0, math.pi, (64, 4)); a2 = rs.uniform(0, math.pi, (64, 4))
    zr = np.stack([np.asarray(o) for o in ref(th, w, a1, a2)], -1)
    zj = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), _layer(jnp.asarray(w[0], jnp.float32)),
                                 _layer(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    zsv = np.asarray(circuit_core_statevector(jnp.asarray(th, jnp.float32), _var_M_sv(jnp.asarray(w[0], jnp.float32)),
                                              _var_M_sv(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    print(f"  max|PennyLane - old state-vector JAX| = {np.abs(zr - zsv).max():.2e}")
    zr0 = np.stack([np.asarray(o) for o in ref(th, w, a1, np.zeros_like(a2))], -1)
    d = np.abs(zr - zj).max()
    print(f"  max|PennyLane - FAST closed-form JAX| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
    print(f"  max|PennyLane(with 2nd RZZ layer) - PennyLane(2nd layer angles = 0)| = {np.abs(zr - zr0).max():.2e}  -> 2nd layer is inert")
    assert d < 5e-5, "JAX circuit disagrees with PennyLane"
    print("  -> PASS\n")

def print_circuit():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def c(th, w, a1):
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    print("Quantum interaction block (per edge; theta = pi*tanh(pair_proj), RZZ angles = pi*sigmoid(entangle_gate1(theta))):")
    print(qml.draw(c)(np.zeros(4), np.zeros((2, 4, 2)), np.zeros(4)))

# ==============================================================================
# 9. RUN
# ==============================================================================
ALL = {}
for variant in VARIANTS_TO_RUN:
    P_demo = init_params(variant, 0)
    rows, total = count_table(P_demo)
    print("\n" + "#" * 96 + f"\n# VARIANT: {variant}   |   {total} trainable parameters\n" + "#" * 96)
    for nm, n in rows:
        print(f"   {nm:<38}{n:>6}")
    print(f"   {'TOTAL':<38}{total:>6}")
    if variant == "quantum":
        verify_against_pennylane(); print_circuit()
    C = Compiled(variant)
    res = []
    for seed in SEEDS:
        print(f"\n--- {variant} | seed {seed} ---")
        tr = train_one(C, seed)
        m, bins, tm = evaluate_test(C, tr, seed)
        h = tr["hist"]
        print(f"   >> best epoch {tr['best_ep']} (val loss {h[tr['best_ep']-1]['val_loss']:.4f}) | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f} | TEST pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.2f}  RMSE {m['rmse_pt']:.3f}")
        print(f"   >> inference (full {len(X_test)}-row test pass, data on device): "
              + "  ".join(f"{k}: {v['test_pass_s']*1e3:.0f} ms = {v['us_per_row']:.2f} us/row ({v['rows_per_s']/1e3:.0f}k rows/s)"
                          for k, v in tm.items() if not k.startswith("latency")))
        print(f"   >> latency ms @ batch 1/64/1024: " + "   ".join(k[8:] + " " + " / ".join(f"{x:.2f}" for x in v.values()) for k, v in tm.items() if k.startswith("latency"))
              + f"   | fp16-weights dR2_log {m['fp16_dR2_log']:+.5f}" + (f" | folded-vs-graph max|d| {m['fold_maxdiff']:.1e}" if 'fold_maxdiff' in m else ""))
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_{variant}_seed{seed}.npz"), w=np.asarray(tr["w"]),
                 w_fp16=np.asarray(tr["w"].astype(jnp.float16)))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        test=m, bins=bins, timing={k: ({str(b): x for b, x in v.items()} if k.startswith("latency") else v) for k, v in tm.items()},
                        hist=h))
    ALL[variant] = res

# ==============================================================================
# 10. AGGREGATE REPORT  (mean +- std over seeds)
# ==============================================================================
def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

def report(variant, res):
    print("\n" + "=" * 96 + f"\nFINAL REPORT  |  {variant}  |  {res[0]['n_par']} params  |  {len(res)} seeds {[r['seed'] for r in res]}  |  best-val checkpoint\n" + "=" * 96)
    g = lambda key: [r["test"][key] for r in res]
    print("TEST SET, log1p(pT) scale:")
    for lab, key in [("R2", "r2_log"), ("MAE", "mae_log"), ("MSE", "mse_log"), ("RMSE", "rmse_log")]:
        print(f"   {lab:<6}{ms(g(key), '{:.5f}')}")
    print("TEST SET, pT in GeV (R2 intentionally omitted):")
    for lab, key, f in [("MAE (GeV)", "mae_pt", "{:.4f}"), ("MSE (GeV^2)", "mse_pt", "{:.3f}"), ("RMSE (GeV)", "rmse_pt", "{:.4f}")]:
        print(f"   {lab:<12}{ms(g(key), f)}")
    print(f"   predictions clamped at pT floor: {ms(g('n_clamped'), '{:.0f}')} rows")
    print("\nCONVERGENCE (val R2_log mean +- std at epoch) and TRAINING TIME:")
    for ep in sorted({1, 2, 5, 10, 25, EPOCHS}):
        if ep <= min(len(r["hist"]) for r in res):
            print(f"   epoch {ep:>2}: {ms([r['hist'][ep-1]['val_r2_log'] for r in res])}   (val loss {ms([r['hist'][ep-1]['val_loss'] for r in res], '{:.4f}')})")
    e2 = [next((h['ep'] for h in r['hist'] if h['val_loss'] <= 1.02 * min(x['val_loss'] for x in r['hist'])), None) for r in res]
    print(f"   epochs to reach within 2% of own best val loss: {ms(e2, '{:.1f}')}   | best epoch: {ms([r['best_ep'] for r in res], '{:.1f}')}")
    print(f"   sec/epoch (median over epochs, incl. val pass): {ms([np.median([h['sec'] for h in r['hist']]) for r in res], '{:.2f}')}"
          f"   | total train: {ms([r['train_total_s'] for r in res], '{:.0f}')} s")
    print("\nINFERENCE TIMING (test pass over all test rows, data already on device, median of 3):")
    for k in res[0]["timing"]:
        if k.startswith("latency"): continue
        print(f"   {k:<7} path: {ms([r['timing'][k]['test_pass_s'] * 1e3 for r in res], '{:.0f}')} ms/pass   "
              f"{ms([r['timing'][k]['us_per_row'] for r in res], '{:.2f}')} us/row   {ms([r['timing'][k]['rows_per_s'] / 1e3 for r in res], '{:.0f}')}k rows/s")
    for k in res[0]["timing"]:
        if k.startswith("latency"):
            print(f"   latency ms, {k[8:]} path (batch 1 / 64 / 1024): " + " / ".join(ms([r['timing'][k][b] for r in res], '{:.2f}') for b in ("1", "64", "1024")))
    print(f"   model size: {res[0]['n_par']} params = {4 * res[0]['n_par']} bytes float32, {2 * res[0]['n_par']} bytes float16 "
          f"(fp16-rounded weights change test R2_log by {ms(g('fp16_dR2_log'), '{:+.5f}')})")
    print("\nPER pT CLASS (test, mean over seeds; +- is std of MAE_pT):")
    print(f"{'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>16}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
    for i, b0 in enumerate(res[0]["bins"]):
        col = lambda k: [r["bins"][i][k] for r in res]
        s = np.std(col("mae_pt"), ddof=1) if len(res) > 1 else 0.0
        print(f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}{b0['frac']:>6.1f}%{np.mean(col('mae_log')):>9.4f}"
              f"{np.mean(col('mae_pt')):>10.3f}+-{s:<5.3f}{np.mean(col('mse_pt')):>13.2f}{np.mean(col('rmse_pt')):>10.3f}"
              f"{np.mean(col('med_rel')):>10.3f}{np.mean(col('bias')):>8.3f}{np.mean(col('sse_share')):>6.1f}%")

for v in VARIANTS_TO_RUN:
    report(v, ALL[v])

out_path = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump({v: ALL[v] for v in ALL}, open(out_path, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {out_path}")

Backend: JAX 0.11.1 | running on cpu:0 (all devices: [CpuDevice(id=0)])   |   seeds [42]   |   variants ['quantum']
Train 235871 | Val 235871 | Test 707614 rows   |   epochs 50, batch 64, lr 0.001, wd 3e-05, clip 1.0, SmoothL1 beta 1.0

################################################################################################
# VARIANT: quantum   |   144 trainable parameters
################################################################################################
   node_proj 7->4                            32
   physics_prior 2->1                         3
   head 4->6                                 30
   head 6->1                                  7
   pair_proj 8->4                            36
   entangle_gate1 4->4 (RZZ angles)          20
   circuit angles (2x4x2)                    16
   TOTAL                                    144
------------------------------------------------------------------------------------------------
VERIFY: closed-form JAX circuit vs Pen

#Ablation Studies

In [ ]:
"""
================================================================================
CELL 4 -- QUANTUM-vs-CLASSICAL ABLATION (same architecture, only the interaction block changes)      [FAST JAX BACKEND, closed-form circuit]
Needs: Cell 0 and Cell 1. Independent of Cell 2 and Cell 3 (if Cell 3 saved its results file, they are loaded for the
comparison table at the end; otherwise set RERUN_QUANTUM_REFERENCE = True to retrain the quantum model here).

VARIANTS (everything else -- node_proj, pair_proj, physics prior, attention, message passing, 4->6->1 head, loss,
optimizer, seeds, batch order, init of the shared layers -- is identical to Cell 3)
  classical36 : PARAMETER-MATCHED control. The whole quantum block (16 circuit angles + 20 entangle_gate1 params = 36) is
                replaced by a 36-parameter classical block  z_exp = tanh( W2 tanh(W1 theta + b1) ),  W1, W2 in R^{4x4}, b1 in R^4
                (same 4-in / 4-out interface, same [-1,1] output range). Total 144 params = identical to the quantum model.
  classical16 : the 16 circuit angles are replaced by a 16-parameter classical map  z_exp = tanh(W theta), W in R^{4x4}.
                entangle_gate1 (the 20 RZZ-angle parameters) is removed as requested, so this model has 124 params
                (quantum model: 144 = 124 + 20 RZZ-gate params).
  noquantum   : the whole quantum block is removed (z_exp = 0, so the message is h_j and the attention logit is the physics
                prior only). pair_proj feeds only the block, so it is removed too: 72 params.

HOW TO READ IT: quantum (144) vs classical36 (144) is the equal-parameter test of the circuit itself. classical16 (124) and
noquantum (72) show how much the interaction block and the RZZ generator contribute at all. IMPORTANT: use the same SEEDS here
as in Cell 3 -- the paired comparison only uses seeds present in both.
================================================================================
"""
SEEDS = [42]                              # must match Cell 3 (paired comparison); e.g. [1, 2, 3, 4, 5]
RERUN_QUANTUM_REFERENCE = False          # True -> also retrain the 144-param quantum model inside this cell
VARIANTS_TO_RUN = ["classical36", "classical16", "noquantum"] + (["quantum"] if RERUN_QUANTUM_REFERENCE else [])
RESULTS_FILE = "piqgat_results_ablation.json"

# ==============================================================================
# 0. IMPORTS  (Cell 0 already installed pennylane / kagglehub; JAX ships with Colab/Kaggle)
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree
import pennylane as qml

# ---- Cell 1 must have been run: these names come from it --------------------------------------------------------
for _n in ["X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES", "QDIM"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12 and QDIM == 16

# ==============================================================================
# 1. HYPER-PARAMETERS  (training recipe identical to the original PI-QGAT cell)
# ==============================================================================
EPOCHS = 50                     # all 50 epochs are run and printed (no early stop unless EARLY_STOPPING=True)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3            # Adam, constant
CLASSICAL_WEIGHT_DECAY = 3e-5   # L2 (added to the gradient, torch-Adam semantics) on every classical weight
QUANTUM_WEIGHT_DECAY = 0.0      # none on the 16 circuit angles
GRAD_CLIP_NORM = 1.0            # global-norm clipping, applied before weight decay
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True       # network predicts standardized log1p(pT); output = y*Y_STD + Y_MEAN
LAMBDA_PHYSICS_ATTN = 0.1       # weight of the physics prior in the attention logit
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6                 # regression head 4 -> 6 -> 1 (tanh)
GAMMA_ESCAPE = True             # nudge RZ angles stuck at sin(gamma)~0 (quantum variant only), as in the original
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EVAL_BATCH_SIZE = 65536         # inference chunk size (no gradients -> same numbers as any other chunk size)
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]       # GeV classes
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
DEVICE_MODE = "auto"            # "auto" = JAX default device | "cpu" = force CPU | "gpu" = force GPU.
                                # Batch-64 training steps are tiny: on many machines CPU is as fast or faster than GPU for TRAINING.
                                # If sec/epoch looks slow and the banner says GPU/TPU, try "cpu" (and vice versa).
if DEVICE_MODE == "cpu":
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
elif DEVICE_MODE == "gpu":
    jax.config.update("jax_default_device", jax.devices("gpu")[0])
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 96)
print(f"Backend: JAX {jax.__version__} | running on {RUN_DEVICE} (all devices: {jax.devices()})   |   seeds {SEEDS}   |   variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} rows   |   epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}, "
      f"wd {CLASSICAL_WEIGHT_DECAY}, clip {GRAD_CLIP_NORM}, SmoothL1 beta {SMOOTH_L1_BETA}")
print("=" * 96)

# ==============================================================================
# 2. FAST EXACT 4-QUBIT SIMULATOR (real arithmetic, batched)
#    state = (re, im) of 16 amplitudes.  circuit per edge:
#      RY(theta_k) --> [RZ(gamma) RY(omega)]_layer0 --> ring IsingZZ(alpha_i(theta)) --> RY(theta_k) (data re-upload)
#      --> [RZ(gamma) RY(omega)]_layer1 --> <Z_k>
#    The variational layer is ONE 16x16 unitary built from the 8 angles of that layer (32x32 real matrix).
#    A 2nd IsingZZ layer would be diagonal in Z and cannot change <Z>: it is omitted (exactly equal output; the 20
#    parameters that used to drive it had zero gradient).  Verified against PennyLane below.
# ==============================================================================
_idx16 = np.arange(QDIM)
_ZZMAT = np.stack([(1 - 2 * ((_idx16 >> (N_QUBITS - 1 - a)) & 1)) * (1 - 2 * ((_idx16 >> (N_QUBITS - 1 - b)) & 1))
                   for a, b in RING_PAIRS]).astype(np.float32)                                   # (4,16)
_ZPL = np.stack([1 - 2 * ((_idx16 >> (N_QUBITS - 1 - k)) & 1) for k in range(N_QUBITS)], 1).astype(np.float32)   # (16,4)

def _var_M_sv(w):
    """w: (4,2) = (omega,gamma) per qubit -> 32x32 real matrix [[Vr^T, Vi^T],[-Vi^T, Vr^T]] of the layer unitary V."""
    om, gam = w[:, 0] / 2, w[:, 1] / 2
    co, si, cg, sg = jnp.cos(om), jnp.sin(om), jnp.cos(gam), jnp.sin(gam)
    gre = jnp.stack([jnp.stack([co * cg, -si * cg], -1), jnp.stack([si * cg, co * cg], -1)], -2)
    gim = jnp.stack([jnp.stack([-co * sg, -si * sg], -1), jnp.stack([-si * sg, co * sg], -1)], -2)
    Vr, Vi = gre[0], gim[0]
    for k in range(1, N_QUBITS):
        Vr, Vi = (jnp.kron(Vr, gre[k]) - jnp.kron(Vi, gim[k]), jnp.kron(Vr, gim[k]) + jnp.kron(Vi, gre[k]))
    return jnp.block([[Vr.T, Vi.T], [-Vi.T, Vr.T]])

def _ry_layer(a, c, s):
    N = a.shape[1]
    for k in range(N_QUBITS):
        r = a.reshape(2, N, 2 ** k, 2, 2 ** (N_QUBITS - 1 - k))
        s0, s1 = r[:, :, :, 0, :], r[:, :, :, 1, :]
        ck, sk = c[None, :, k, None, None], s[None, :, k, None, None]
        a = jnp.stack([ck * s0 - sk * s1, sk * s0 + ck * s1], axis=3).reshape(2, N, QDIM)
    return a

def circuit_core_statevector(theta, M0, M1, a1):
    """REFERENCE ONLY (old 16-amplitude path). theta (N,4) angles, M0/M1 layer matrices, a1 (N,4) RZZ angles -> <Z_k> (N,4)."""
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, N_QUBITS):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(theta.shape[0], -1)
    out = psi @ M0[:QDIM]                                   # input state is real -> only the top half of M0 is needed
    re, im = out[:, :QDIM], out[:, QDIM:]
    ph = -0.5 * (a1 @ jnp.asarray(_ZZMAT, a1.dtype))        # ring IsingZZ = diagonal phase
    cp, sp = jnp.cos(ph), jnp.sin(ph)
    re, im = re * cp - im * sp, re * sp + im * cp
    a = _ry_layer(jnp.stack([re, im]), c, s)                # data re-upload (RY is real)
    out = jnp.concatenate([a[0], a[1]], -1) @ M1
    re, im = out[:, :QDIM], out[:, QDIM:]
    return (re * re + im * im) @ jnp.asarray(_ZPL, re.dtype)

# ------------------------------------------------------------------------------------------------------------------
# FAST PATH: closed-form (light-cone / reduced-density-matrix) evaluation of the SAME circuit.  Exact, not approximate.
#   Every gate except the ring IsingZZ is single-qubit, so the state before the IsingZZ layer is a product state and the
#   IsingZZ layer is diagonal.  Tracing out the other qubits, the reduced state of qubit k after the ring is its own
#   Bloch vector (x, y, z) with the coherence w = x - i*y multiplied by  prod_{j in nbr(k)} (cos a_kj - i <Z_j> sin a_kj)
#   (z unchanged).  The remaining gates on qubit k are single-qubit rotations, and <Z_k> only depends on the reduced
#   state of k, so the whole circuit reduces to 3-D Bloch-vector rotations: O(n_qubits) elementwise ops per edge instead
#   of 16-amplitude complex state vectors and 32x32 matrix products.  This is the light-cone / Heisenberg-picture idea of
#   Pauli propagation (Rudolph et al., arXiv:2505.21606) applied exactly to this depth-1-entangler circuit.
#   Checked against the state-vector simulator AND PennyLane in verify_against_pennylane().
# ------------------------------------------------------------------------------------------------------------------
def _layer(w):
    """Variational-layer parameters (4,2) = (omega, gamma) per qubit. The closed form uses the angles directly."""
    return w

def circuit_core(theta, W0, W1, a1):
    """theta (N,4) angles, W0/W1 (4,2) layer angles, a1 (N,4) RZZ angles (edge i joins qubits i, i+1 mod 4) -> <Z_k> (N,4)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    # RY(theta) |0>  ->  RZ(gamma0)  ->  RY(omega0)
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    # ring IsingZZ: coherence of qubit k picks up (cos a - i z_j sin a) from both ring neighbours j
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)          # <Z> of qubit k+1 and k-1
    fr, fi = ca, -z_next * sa                                                  # edge k     = (k, k+1)
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)        # edge k-1   = (k-1, k)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr                                  # (x - i y) * h
    x, y = wr, -wi
    # data re-upload RY(theta) -> RZ(gamma1) -> RY(omega1) -> <Z>
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL  (variants share everything except the "interaction block" that produces z_exp)
#   h      = node_proj(x)                                  7 -> 4        per station (node)
#   edge   : theta = pi*tanh(pair_proj([h_i, h_j]))        8 -> 4        angles for the interaction block
#   z_exp  = interaction block(theta)                      4 -> 4
#              quantum     : 4-qubit re-uploading ring circuit; 16 angles + entangle_gate1 (4->4, sets the RZZ angles)
#              classical36 : tanh(W2 tanh(W1 theta + b1)), 16+4+16 = 36 params (matches the quantum block's 36)
#              classical16 : tanh(W theta), W is 4x4 (16 params), no entangle_gate1
#              noquantum   : z_exp = 0 (block and pair_proj removed)
#   P      = physics_prior(|d bending|, |d phi|)           2 -> 1
#   alpha  = softmax_j( mean_k z_exp + lambda * P )        attention over the 3 incoming edges
#   m_i    = sum_j alpha_ij (h_j + z_exp_ij);  h'_i = h_i + m_i;  g = mean_i h'_i
#   y_hat  = Y_STD * head(g) + Y_MEAN,   head = Linear(4,6)-tanh-Linear(6,1)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b) if bias else jnp.zeros((n_out,), jnp.float32)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same keys in every variant -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if variant != "noquantum":
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
    if variant == "quantum":
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    if variant == "classical36":
        l1 = _init_lin(ks[7], N_QUBITS, N_QUBITS); l2 = _init_lin(jax.random.fold_in(ks[7], 1), N_QUBITS, N_QUBITS, bias=False)
        P["c36"] = {"W1": l1["W"], "b1": l1["b"], "W2": l2["W"]}
    if variant == "classical16":
        bnd = 1.0 / math.sqrt(N_QUBITS)
        P["cW"] = jax.random.uniform(ks[6], (N_QUBITS, N_QUBITS), jnp.float32, -bnd, bnd)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, variant, x, phi, ba, M01=None):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    if variant == "noquantum":
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
        s = LAMBDA_PHYSICS_ATTN * Pp
    else:
        u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        if variant == "quantum":
            a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
            M0, M1 = M01 if M01 is not None else (_layer(P["qw"][0]), _layer(P["qw"][1]))
            z = circuit_core(tf, M0, M1, a1)
        elif variant == "classical36":
            c = P["c36"]
            z = jnp.tanh(jnp.tanh(tf @ c["W1"].T + c["b1"]) @ c["W2"].T)
        else:
            z = jnp.tanh(tf @ P["cW"].T)
        z = z.reshape(B, N_EDGES, N_QUBITS)
        s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

GROUPS = {"node": "node_proj 7->4", "pair": "pair_proj 8->4", "phys": "physics_prior 2->1", "head1": "head 4->6",
          "head2": "head 6->1", "gate1": "entangle_gate1 4->4 (RZZ angles)", "qw": "circuit angles (2x4x2)",
          "cW": "classical block W 4x4", "c36": "classical block 4-4-4 (W1,b1,W2)"}
def count_table(P):
    rows = [(GROUPS[k], int(sum(a.size for a in jax.tree_util.tree_leaves(v)))) for k, v in P.items()]
    return rows, sum(n for _, n in rows)

# ==============================================================================
# 4. TRAIN STEP / EPOCH / PREDICT  (one compiled call per epoch via lax.scan)
# ==============================================================================
class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        self.wd_vec = ravel_pytree(wd)[0]
        unravel, wd_vec = self.unravel, self.wd_vec

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), variant, x, p_, b_), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                               # torch Adam L2 decay
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, M01, x, phi, ba):
            return forward(unravel(w), variant, x, phi, ba, M01)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

    def fold(self, w):
        """Inference-time constant folding: precompute the two layer matrices once (quantum variant only)."""
        if self.variant != "quantum":
            return None
        P = self.unravel(w)
        return (_layer(P["qw"][0]), _layer(P["qw"][1]))

# data on device, once
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, M01=None, bs=EVAL_BATCH_SIZE):
    outs = [C.pred(w, M01, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)]
    out = jnp.concatenate(outs); out.block_until_ready()
    return np.asarray(out)

# ==============================================================================
# 5. METRICS  (log scale: R2/MAE/MSE/RMSE.  pT in GeV: MAE/MSE/RMSE only.  per-GeV-class table)
# ==============================================================================
def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

# ==============================================================================
# 6. TRAIN ONE (variant, seed)
# ==============================================================================
def train_one(C, seed):
    variant = C.variant
    P0 = init_params(variant, seed)
    w, _ = ravel_pytree(P0); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, bad, escapes = [], float("inf"), w, 0, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and variant == "quantum":
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        pv = predict_log(C, w, Xva, pva, bva)
        vm, _ = split_metrics(pv, y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep, bad = vm["loss"], jnp.array(w), ep, 0; flag = " *best"
        else:
            bad += 1
        print(f"[{variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s (train {t_tr:.1f} + eval {sec - t_tr:.1f}){flag}"
              + ("   (epoch 1 includes one-time XLA compile)" if ep == 1 and seed == SEEDS[0] else ""), flush=True)
        if EARLY_STOPPING and bad >= EARLY_STOPPING_PATIENCE:
            print(f"   early stop at epoch {ep}"); break
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

# ==============================================================================
# 7. TEST: metrics + per-class table + INFERENCE TIMING
# ==============================================================================
def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def timing_block(C, w, M01):
    n = Xte.shape[0]
    def full_pass(Mx):
        outs = [C.pred(w, Mx, Xte[i:i + EVAL_BATCH_SIZE], pte[i:i + EVAL_BATCH_SIZE], bte[i:i + EVAL_BATCH_SIZE])
                for i in range(0, n, EVAL_BATCH_SIZE)]
        jnp.concatenate(outs).block_until_ready()
    out = {}
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        full_pass(Mx)                                           # warm-up / compile
        tt = _med(lambda: full_pass(Mx), 3)
        out[name] = dict(test_pass_s=tt, us_per_row=1e6 * tt / n, rows_per_s=n / tt)
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        lat = {}
        for b in (1, 64, 1024):
            f = lambda: C.pred(w, Mx, Xte[:b], pte[:b], bte[:b]).block_until_ready()
            f(); lat[b] = 1e3 * _med(f, 200 if b == 1 else 60)
        out["latency_" + name] = lat
    return out

def evaluate_test(C, tr, seed):
    w = tr["w"]; M01 = C.fold(w)
    pred = predict_log(C, w, Xte, pte, bte, M01)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    if M01 is not None:                                          # folded path must equal the training graph
        m["fold_maxdiff"] = float(np.abs(pred - predict_log(C, w, Xte, pte, bte, None)).max())
    w16 = w.astype(jnp.float16).astype(jnp.float32)              # storage-precision check: weights rounded to float16
    m16, _ = split_metrics(predict_log(C, w16, Xte, pte, bte, C.fold(w16)), y_test, pT_test64)
    m["fp16_dR2_log"] = m16["r2_log"] - m["r2_log"]; m["fp16_dMAE_log"] = m16["mae_log"] - m["mae_log"]
    tm = timing_block(C, w, M01)
    return m, bins, tm

# ==============================================================================
# 8. PENNYLANE VERIFICATION (quantum variant only)
# ==============================================================================
def verify_against_pennylane():
    print("-" * 96 + "\nVERIFY: closed-form JAX circuit vs PennyLane default.qubit (full circuit incl. the 2nd IsingZZ layer, float64)\n" + "-" * 96)
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def ref(theta, w, a1, a2):
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[a, b])
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    rs = np.random.default_rng(7)
    th = rs.normal(size=(64, 4)) * 1.5; w = rs.normal(size=(2, 4, 2)) * 2
    a1 = rs.uniform(0, math.pi, (64, 4)); a2 = rs.uniform(0, math.pi, (64, 4))
    zr = np.stack([np.asarray(o) for o in ref(th, w, a1, a2)], -1)
    zj = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), _layer(jnp.asarray(w[0], jnp.float32)),
                                 _layer(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    zsv = np.asarray(circuit_core_statevector(jnp.asarray(th, jnp.float32), _var_M_sv(jnp.asarray(w[0], jnp.float32)),
                                              _var_M_sv(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    print(f"  max|PennyLane - old state-vector JAX| = {np.abs(zr - zsv).max():.2e}")
    zr0 = np.stack([np.asarray(o) for o in ref(th, w, a1, np.zeros_like(a2))], -1)
    d = np.abs(zr - zj).max()
    print(f"  max|PennyLane - FAST closed-form JAX| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
    print(f"  max|PennyLane(with 2nd RZZ layer) - PennyLane(2nd layer angles = 0)| = {np.abs(zr - zr0).max():.2e}  -> 2nd layer is inert")
    assert d < 5e-5, "JAX circuit disagrees with PennyLane"
    print("  -> PASS\n")

def print_circuit():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def c(th, w, a1):
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    print("Quantum interaction block (per edge; theta = pi*tanh(pair_proj), RZZ angles = pi*sigmoid(entangle_gate1(theta))):")
    print(qml.draw(c)(np.zeros(4), np.zeros((2, 4, 2)), np.zeros(4)))

# ==============================================================================
# 9. RUN
# ==============================================================================
ALL = {}
for variant in VARIANTS_TO_RUN:
    P_demo = init_params(variant, 0)
    rows, total = count_table(P_demo)
    print("\n" + "#" * 96 + f"\n# VARIANT: {variant}   |   {total} trainable parameters\n" + "#" * 96)
    for nm, n in rows:
        print(f"   {nm:<38}{n:>6}")
    print(f"   {'TOTAL':<38}{total:>6}")
    if variant == "quantum":
        verify_against_pennylane(); print_circuit()
    C = Compiled(variant)
    res = []
    for seed in SEEDS:
        print(f"\n--- {variant} | seed {seed} ---")
        tr = train_one(C, seed)
        m, bins, tm = evaluate_test(C, tr, seed)
        h = tr["hist"]
        print(f"   >> best epoch {tr['best_ep']} (val loss {h[tr['best_ep']-1]['val_loss']:.4f}) | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f} | TEST pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.2f}  RMSE {m['rmse_pt']:.3f}")
        print(f"   >> inference (full {len(X_test)}-row test pass, data on device): "
              + "  ".join(f"{k}: {v['test_pass_s']*1e3:.0f} ms = {v['us_per_row']:.2f} us/row ({v['rows_per_s']/1e3:.0f}k rows/s)"
                          for k, v in tm.items() if not k.startswith("latency")))
        print(f"   >> latency ms @ batch 1/64/1024: " + "   ".join(k[8:] + " " + " / ".join(f"{x:.2f}" for x in v.values()) for k, v in tm.items() if k.startswith("latency"))
              + f"   | fp16-weights dR2_log {m['fp16_dR2_log']:+.5f}" + (f" | folded-vs-graph max|d| {m['fold_maxdiff']:.1e}" if 'fold_maxdiff' in m else ""))
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_{variant}_seed{seed}.npz"), w=np.asarray(tr["w"]),
                 w_fp16=np.asarray(tr["w"].astype(jnp.float16)))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        test=m, bins=bins, timing={k: ({str(b): x for b, x in v.items()} if k.startswith("latency") else v) for k, v in tm.items()},
                        hist=h))
    ALL[variant] = res

# ==============================================================================
# 10. AGGREGATE REPORT  (mean +- std over seeds)
# ==============================================================================
def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

def report(variant, res):
    print("\n" + "=" * 96 + f"\nFINAL REPORT  |  {variant}  |  {res[0]['n_par']} params  |  {len(res)} seeds {[r['seed'] for r in res]}  |  best-val checkpoint\n" + "=" * 96)
    g = lambda key: [r["test"][key] for r in res]
    print("TEST SET, log1p(pT) scale:")
    for lab, key in [("R2", "r2_log"), ("MAE", "mae_log"), ("MSE", "mse_log"), ("RMSE", "rmse_log")]:
        print(f"   {lab:<6}{ms(g(key), '{:.5f}')}")
    print("TEST SET, pT in GeV (R2 intentionally omitted):")
    for lab, key, f in [("MAE (GeV)", "mae_pt", "{:.4f}"), ("MSE (GeV^2)", "mse_pt", "{:.3f}"), ("RMSE (GeV)", "rmse_pt", "{:.4f}")]:
        print(f"   {lab:<12}{ms(g(key), f)}")
    print(f"   predictions clamped at pT floor: {ms(g('n_clamped'), '{:.0f}')} rows")
    print("\nCONVERGENCE (val R2_log mean +- std at epoch) and TRAINING TIME:")
    for ep in sorted({1, 2, 5, 10, 25, EPOCHS}):
        if ep <= min(len(r["hist"]) for r in res):
            print(f"   epoch {ep:>2}: {ms([r['hist'][ep-1]['val_r2_log'] for r in res])}   (val loss {ms([r['hist'][ep-1]['val_loss'] for r in res], '{:.4f}')})")
    e2 = [next((h['ep'] for h in r['hist'] if h['val_loss'] <= 1.02 * min(x['val_loss'] for x in r['hist'])), None) for r in res]
    print(f"   epochs to reach within 2% of own best val loss: {ms(e2, '{:.1f}')}   | best epoch: {ms([r['best_ep'] for r in res], '{:.1f}')}")
    print(f"   sec/epoch (median over epochs, incl. val pass): {ms([np.median([h['sec'] for h in r['hist']]) for r in res], '{:.2f}')}"
          f"   | total train: {ms([r['train_total_s'] for r in res], '{:.0f}')} s")
    print("\nINFERENCE TIMING (test pass over all test rows, data already on device, median of 3):")
    for k in res[0]["timing"]:
        if k.startswith("latency"): continue
        print(f"   {k:<7} path: {ms([r['timing'][k]['test_pass_s'] * 1e3 for r in res], '{:.0f}')} ms/pass   "
              f"{ms([r['timing'][k]['us_per_row'] for r in res], '{:.2f}')} us/row   {ms([r['timing'][k]['rows_per_s'] / 1e3 for r in res], '{:.0f}')}k rows/s")
    for k in res[0]["timing"]:
        if k.startswith("latency"):
            print(f"   latency ms, {k[8:]} path (batch 1 / 64 / 1024): " + " / ".join(ms([r['timing'][k][b] for r in res], '{:.2f}') for b in ("1", "64", "1024")))
    print(f"   model size: {res[0]['n_par']} params = {4 * res[0]['n_par']} bytes float32, {2 * res[0]['n_par']} bytes float16 "
          f"(fp16-rounded weights change test R2_log by {ms(g('fp16_dR2_log'), '{:+.5f}')})")
    print("\nPER pT CLASS (test, mean over seeds; +- is std of MAE_pT):")
    print(f"{'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>16}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
    for i, b0 in enumerate(res[0]["bins"]):
        col = lambda k: [r["bins"][i][k] for r in res]
        s = np.std(col("mae_pt"), ddof=1) if len(res) > 1 else 0.0
        print(f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}{b0['frac']:>6.1f}%{np.mean(col('mae_log')):>9.4f}"
              f"{np.mean(col('mae_pt')):>10.3f}+-{s:<5.3f}{np.mean(col('mse_pt')):>13.2f}{np.mean(col('rmse_pt')):>10.3f}"
              f"{np.mean(col('med_rel')):>10.3f}{np.mean(col('bias')):>8.3f}{np.mean(col('sse_share')):>6.1f}%")

for v in VARIANTS_TO_RUN:
    report(v, ALL[v])

out_path = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump({v: ALL[v] for v in ALL}, open(out_path, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {out_path}")

# ==============================================================================
# 11. QUANTUM vs CLASSICAL COMPARISON  (paired by seed: same init for shared layers, same batch order)
# ==============================================================================
REF = ALL.get("quantum")
if REF is None:
    _p = os.path.join(RESULTS_DIR, "piqgat_results_quantum.json")
    if os.path.exists(_p):
        REF = json.load(open(_p))["quantum"]; print(f"\n(loaded quantum reference from {_p})")
    else:
        print("\nNo quantum reference found (run Cell 3 or set RERUN_QUANTUM_REFERENCE=True) -> comparison skipped.")
if REF is not None:
    CMP = {"quantum (144)": REF, **{f"{v} ({ALL[v][0]['n_par']})": ALL[v] for v in ALL if v != "quantum"}}
    print("\n" + "=" * 130 + "\nCOMPARISON (test set, mean +- std over seeds)\n" + "=" * 130)
    print(f"{'variant (params)':<22}{'R2_log':>20}{'MAE_log':>20}{'MAE_pT GeV':>20}{'MSE_pT':>20}{'us/row':>12}{'sec/epoch':>11}{'ep->2%':>8}")
    for nm, rs in CMP.items():
        e2 = [next((h['ep'] for h in r['hist'] if h['val_loss'] <= 1.02 * min(x['val_loss'] for x in r['hist'])), np.nan) for r in rs]
        print(f"{nm:<22}{ms([r['test']['r2_log'] for r in rs], '{:.4f}'):>20}{ms([r['test']['mae_log'] for r in rs], '{:.4f}'):>20}"
              f"{ms([r['test']['mae_pt'] for r in rs], '{:.3f}'):>20}{ms([r['test']['mse_pt'] for r in rs], '{:.2f}'):>20}"
              f"{np.mean([r['timing']['graph']['us_per_row'] for r in rs]):>12.2f}"
              f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.nanmean(e2):>8.1f}")
    qs = {r["seed"]: r for r in REF}
    print("\nPAIRED DIFFERENCE  (quantum minus variant; for R2 positive = quantum better, for errors negative = quantum better):")
    for v in ALL:
        if v == "quantum": continue
        common = [r["seed"] for r in ALL[v] if r["seed"] in qs]
        vs = {r["seed"]: r for r in ALL[v]}
        for lab, key in [("dR2_log", "r2_log"), ("dMAE_log", "mae_log"), ("dMAE_pT", "mae_pt")]:
            d = np.array([qs[s]["test"][key] - vs[s]["test"][key] for s in common])
            wins = int((d > 0).sum()) if key == "r2_log" else int((d < 0).sum())
            print(f"   quantum - {v:<12} {lab:<9} {d.mean():+.5f} +- {d.std(ddof=1) if len(d) > 1 else 0:.5f}   quantum better in {wins}/{len(d)} seeds")
    print("\nA gap is evidence only if its mean clearly exceeds the std and the sign is the same in (nearly) all seeds.")

Backend: JAX 0.11.1 | running on cpu:0 (all devices: [CpuDevice(id=0)])   |   seeds [42]   |   variants ['classical36', 'classical16', 'noquantum']
Train 235871 | Val 235871 | Test 707614 rows   |   epochs 50, batch 64, lr 0.001, wd 3e-05, clip 1.0, SmoothL1 beta 1.0

################################################################################################
# VARIANT: classical36   |   144 trainable parameters
################################################################################################
   node_proj 7->4                            32
   physics_prior 2->1                         3
   head 4->6                                 30
   head 6->1                                  7
   pair_proj 8->4                            36
   classical block 4-4-4 (W1,b1,W2)          36
   TOTAL                                    144

--- classical36 | seed 42 ---
[classical36 | seed 42] ep  1/50  train 0.2288  val loss 0.1675  R2_log 0.4552  MAE_log 0.3828  MAE_pT 15.217 GeV   

In [ ]:
"""
================================================================================
CELL 5 -- DISTIL THE QUANTUM INTERACTION BLOCK INTO CLASSICAL PARAMETERS                         [FAST JAX BACKEND, closed-form circuit]
Needs: Cell 0, Cell 1 and Cell 3 (Cell 3 must have saved piqgat_quantum_seed{S}.npz = the trained 144-param teacher).

TEACHER  the trained PI-QGAT: node_proj, pair_proj, physics prior, head are classical; the 4-qubit circuit + entangle_gate1
         (36 params) maps theta (4 angles per edge) -> z = <Z_k> (4 values per edge).
STUDENT  the SAME network with the 36-param quantum block replaced by a classical MLP  z = tanh(W2 tanh(W1 theta + b1)),
         hidden width H.  H = 4 -> 36 block params -> 144 total (equal budget).  H = 8/16/32 -> 180/252/396 total: "how much
         classical capacity does it take to absorb the circuit?".  No circuit is evaluated at inference.

STAGE A  block-level, LABEL-FREE hint fit (FitNets-style / "classical surrogate"):  min_c  E || block_c(theta) - z_teacher(theta) ||^2
         over every edge of every training event, theta from the TEACHER's own trunk.  Result = "zero-shot swap": drop the student
         block in, change nothing else, no label ever seen.  Measures pure REPRESENTABILITY of the circuit by H hidden units.
STAGE B  prediction-level KD fine-tune of ALL student parameters (small lr):
             L = LAM_Y * SmoothL1(y_s, y_true) + LAM_T * SmoothL1(y_s, y_teacher) + LAM_Z * MSE(z_s, z_teacher(theta_student))
         (regression KD: no temperature; the teacher's log-pT output is the "soft label"; the hint term keeps the block faithful
         on the thetas the student's own trunk produces).
CONTROL  same budget, same warm-started trunk, random block, TRUE LABELS ONLY (LAM_T = LAM_Z = 0).  If distilled > control, the
         gain is the teacher's knowledge, not just the warm start.   Cell 4's classical36 (from scratch) is printed if its
         results file exists.
READ-OUT students beating/matching the teacher at the same 144 params => the circuit's value was a better OPTIMISATION PATH, not a
         better function class.  Students lagging => the circuit expresses something 36 tanh units cannot (checked by the H sweep).
================================================================================
"""
SEEDS = [42]                              # teacher seeds to distil (need piqgat_quantum_seed{S}.npz from Cell 3)
VARIANTS_TO_RUN = ["distilled"]
H_LIST = [4, 8, 16, 32]                   # student hidden widths; H = 4 is the 144-param, equal-budget student
HEADLINE_H = 4
A_EPOCHS, A_BATCH, A_LR = 40, 8192, 3e-3  # stage A (label-free block fit)
B_EPOCHS, B_LR = 20, 3e-4                 # stage B (KD fine-tune), batch = BATCH_SIZE (64), same clip / wd as Cell 3
LAM_Y, LAM_T, LAM_Z = 0.5, 0.5, 0.5       # true-label / teacher-output / block-hint weights
RUN_CONTROL = True                        # headline-size student, true labels only, warm-started trunk, random block
RESULTS_FILE = "piqgat_results_distill.json"
# ==============================================================================
# 0. IMPORTS  (Cell 0 already installed pennylane / kagglehub; JAX ships with Colab/Kaggle)
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree
import pennylane as qml

# ---- Cell 1 must have been run: these names come from it --------------------------------------------------------
for _n in ["X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES", "QDIM"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12 and QDIM == 16

# ==============================================================================
# 1. HYPER-PARAMETERS  (training recipe identical to the original PI-QGAT cell)
# ==============================================================================
EPOCHS = 50                     # all 50 epochs are run and printed (no early stop unless EARLY_STOPPING=True)
BATCH_SIZE = 64
LEARNING_RATE = 1e-3            # Adam, constant
CLASSICAL_WEIGHT_DECAY = 3e-5   # L2 (added to the gradient, torch-Adam semantics) on every classical weight
QUANTUM_WEIGHT_DECAY = 0.0      # none on the 16 circuit angles
GRAD_CLIP_NORM = 1.0            # global-norm clipping, applied before weight decay
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True       # network predicts standardized log1p(pT); output = y*Y_STD + Y_MEAN
LAMBDA_PHYSICS_ATTN = 0.1       # weight of the physics prior in the attention logit
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6                 # regression head 4 -> 6 -> 1 (tanh)
GAMMA_ESCAPE = True             # nudge RZ angles stuck at sin(gamma)~0 (quantum variant only), as in the original
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EARLY_STOPPING = False
EARLY_STOPPING_PATIENCE = 12
EVAL_BATCH_SIZE = 65536         # inference chunk size (no gradients -> same numbers as any other chunk size)
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]       # GeV classes
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
DEVICE_MODE = "auto"            # "auto" = JAX default device | "cpu" = force CPU | "gpu" = force GPU.
                                # Batch-64 training steps are tiny: on many machines CPU is as fast or faster than GPU for TRAINING.
                                # If sec/epoch looks slow and the banner says GPU/TPU, try "cpu" (and vice versa).
if DEVICE_MODE == "cpu":
    jax.config.update("jax_default_device", jax.devices("cpu")[0])
elif DEVICE_MODE == "gpu":
    jax.config.update("jax_default_device", jax.devices("gpu")[0])
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 96)
print(f"Backend: JAX {jax.__version__} | running on {RUN_DEVICE} (all devices: {jax.devices()})   |   seeds {SEEDS}   |   variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} rows   |   epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}, "
      f"wd {CLASSICAL_WEIGHT_DECAY}, clip {GRAD_CLIP_NORM}, SmoothL1 beta {SMOOTH_L1_BETA}")
print("=" * 96)

# ==============================================================================
# 2. FAST EXACT 4-QUBIT SIMULATOR (real arithmetic, batched)
#    state = (re, im) of 16 amplitudes.  circuit per edge:
#      RY(theta_k) --> [RZ(gamma) RY(omega)]_layer0 --> ring IsingZZ(alpha_i(theta)) --> RY(theta_k) (data re-upload)
#      --> [RZ(gamma) RY(omega)]_layer1 --> <Z_k>
#    The variational layer is ONE 16x16 unitary built from the 8 angles of that layer (32x32 real matrix).
#    A 2nd IsingZZ layer would be diagonal in Z and cannot change <Z>: it is omitted (exactly equal output; the 20
#    parameters that used to drive it had zero gradient).  Verified against PennyLane below.
# ==============================================================================
_idx16 = np.arange(QDIM)
_ZZMAT = np.stack([(1 - 2 * ((_idx16 >> (N_QUBITS - 1 - a)) & 1)) * (1 - 2 * ((_idx16 >> (N_QUBITS - 1 - b)) & 1))
                   for a, b in RING_PAIRS]).astype(np.float32)                                   # (4,16)
_ZPL = np.stack([1 - 2 * ((_idx16 >> (N_QUBITS - 1 - k)) & 1) for k in range(N_QUBITS)], 1).astype(np.float32)   # (16,4)

def _var_M_sv(w):
    """w: (4,2) = (omega,gamma) per qubit -> 32x32 real matrix [[Vr^T, Vi^T],[-Vi^T, Vr^T]] of the layer unitary V."""
    om, gam = w[:, 0] / 2, w[:, 1] / 2
    co, si, cg, sg = jnp.cos(om), jnp.sin(om), jnp.cos(gam), jnp.sin(gam)
    gre = jnp.stack([jnp.stack([co * cg, -si * cg], -1), jnp.stack([si * cg, co * cg], -1)], -2)
    gim = jnp.stack([jnp.stack([-co * sg, -si * sg], -1), jnp.stack([-si * sg, co * sg], -1)], -2)
    Vr, Vi = gre[0], gim[0]
    for k in range(1, N_QUBITS):
        Vr, Vi = (jnp.kron(Vr, gre[k]) - jnp.kron(Vi, gim[k]), jnp.kron(Vr, gim[k]) + jnp.kron(Vi, gre[k]))
    return jnp.block([[Vr.T, Vi.T], [-Vi.T, Vr.T]])

def _ry_layer(a, c, s):
    N = a.shape[1]
    for k in range(N_QUBITS):
        r = a.reshape(2, N, 2 ** k, 2, 2 ** (N_QUBITS - 1 - k))
        s0, s1 = r[:, :, :, 0, :], r[:, :, :, 1, :]
        ck, sk = c[None, :, k, None, None], s[None, :, k, None, None]
        a = jnp.stack([ck * s0 - sk * s1, sk * s0 + ck * s1], axis=3).reshape(2, N, QDIM)
    return a

def circuit_core_statevector(theta, M0, M1, a1):
    """REFERENCE ONLY (old 16-amplitude path). theta (N,4) angles, M0/M1 layer matrices, a1 (N,4) RZZ angles -> <Z_k> (N,4)."""
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, N_QUBITS):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(theta.shape[0], -1)
    out = psi @ M0[:QDIM]                                   # input state is real -> only the top half of M0 is needed
    re, im = out[:, :QDIM], out[:, QDIM:]
    ph = -0.5 * (a1 @ jnp.asarray(_ZZMAT, a1.dtype))        # ring IsingZZ = diagonal phase
    cp, sp = jnp.cos(ph), jnp.sin(ph)
    re, im = re * cp - im * sp, re * sp + im * cp
    a = _ry_layer(jnp.stack([re, im]), c, s)                # data re-upload (RY is real)
    out = jnp.concatenate([a[0], a[1]], -1) @ M1
    re, im = out[:, :QDIM], out[:, QDIM:]
    return (re * re + im * im) @ jnp.asarray(_ZPL, re.dtype)

# ------------------------------------------------------------------------------------------------------------------
# FAST PATH: closed-form (light-cone / reduced-density-matrix) evaluation of the SAME circuit.  Exact, not approximate.
#   Every gate except the ring IsingZZ is single-qubit, so the state before the IsingZZ layer is a product state and the
#   IsingZZ layer is diagonal.  Tracing out the other qubits, the reduced state of qubit k after the ring is its own
#   Bloch vector (x, y, z) with the coherence w = x - i*y multiplied by  prod_{j in nbr(k)} (cos a_kj - i <Z_j> sin a_kj)
#   (z unchanged).  The remaining gates on qubit k are single-qubit rotations, and <Z_k> only depends on the reduced
#   state of k, so the whole circuit reduces to 3-D Bloch-vector rotations: O(n_qubits) elementwise ops per edge instead
#   of 16-amplitude complex state vectors and 32x32 matrix products.  This is the light-cone / Heisenberg-picture idea of
#   Pauli propagation (Rudolph et al., arXiv:2505.21606) applied exactly to this depth-1-entangler circuit.
#   Checked against the state-vector simulator AND PennyLane in verify_against_pennylane().
# ------------------------------------------------------------------------------------------------------------------
def _layer(w):
    """Variational-layer parameters (4,2) = (omega, gamma) per qubit. The closed form uses the angles directly."""
    return w

def circuit_core(theta, W0, W1, a1):
    """theta (N,4) angles, W0/W1 (4,2) layer angles, a1 (N,4) RZZ angles (edge i joins qubits i, i+1 mod 4) -> <Z_k> (N,4)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    # RY(theta) |0>  ->  RZ(gamma0)  ->  RY(omega0)
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    # ring IsingZZ: coherence of qubit k picks up (cos a - i z_j sin a) from both ring neighbours j
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)          # <Z> of qubit k+1 and k-1
    fr, fi = ca, -z_next * sa                                                  # edge k     = (k, k+1)
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)        # edge k-1   = (k-1, k)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr                                  # (x - i y) * h
    x, y = wr, -wi
    # data re-upload RY(theta) -> RZ(gamma1) -> RY(omega1) -> <Z>
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL  (variants share everything except the "interaction block" that produces z_exp)
#   h      = node_proj(x)                                  7 -> 4        per station (node)
#   edge   : theta = pi*tanh(pair_proj([h_i, h_j]))        8 -> 4        angles for the interaction block
#   z_exp  = interaction block(theta)                      4 -> 4
#              quantum     : 4-qubit re-uploading ring circuit; 16 angles + entangle_gate1 (4->4, sets the RZZ angles)
#              classical16 : tanh(W theta), W is 4x4 (16 params), no entangle_gate1
#              noquantum   : z_exp = 0 (block and pair_proj removed)
#   P      = physics_prior(|d bending|, |d phi|)           2 -> 1
#   alpha  = softmax_j( mean_k z_exp + lambda * P )        attention over the 3 incoming edges
#   m_i    = sum_j alpha_ij (h_j + z_exp_ij);  h'_i = h_i + m_i;  g = mean_i h'_i
#   y_hat  = Y_STD * head(g) + Y_MEAN,   head = Linear(4,6)-tanh-Linear(6,1)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b) if bias else jnp.zeros((n_out,), jnp.float32)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same keys in every variant -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if variant != "noquantum":
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
    if variant == "quantum":
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    if variant == "classical16":
        bnd = 1.0 / math.sqrt(N_QUBITS)
        P["cW"] = jax.random.uniform(ks[6], (N_QUBITS, N_QUBITS), jnp.float32, -bnd, bnd)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, variant, x, phi, ba, M01=None):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    if variant == "noquantum":
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
        s = LAMBDA_PHYSICS_ATTN * Pp
    else:
        u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        if variant == "quantum":
            a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
            M0, M1 = M01 if M01 is not None else (_layer(P["qw"][0]), _layer(P["qw"][1]))
            z = circuit_core(tf, M0, M1, a1)
        else:
            z = jnp.tanh(tf @ P["cW"].T)
        z = z.reshape(B, N_EDGES, N_QUBITS)
        s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

GROUPS = {"node": "node_proj 7->4", "pair": "pair_proj 8->4", "phys": "physics_prior 2->1", "head1": "head 4->6",
          "head2": "head 6->1", "gate1": "entangle_gate1 4->4 (RZZ angles)", "qw": "circuit angles (2x4x2)",
          "cW": "classical block W 4x4"}
def count_table(P):
    rows = [(GROUPS[k], int(sum(a.size for a in jax.tree_util.tree_leaves(v)))) for k, v in P.items()]
    return rows, sum(n for _, n in rows)

# ==============================================================================
# 4. TRAIN STEP / EPOCH / PREDICT  (one compiled call per epoch via lax.scan)
# ==============================================================================
class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        self.wd_vec = ravel_pytree(wd)[0]
        unravel, wd_vec = self.unravel, self.wd_vec

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), variant, x, p_, b_), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                               # torch Adam L2 decay
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, M01, x, phi, ba):
            return forward(unravel(w), variant, x, phi, ba, M01)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

    def fold(self, w):
        """Inference-time constant folding: precompute the two layer matrices once (quantum variant only)."""
        if self.variant != "quantum":
            return None
        P = self.unravel(w)
        return (_layer(P["qw"][0]), _layer(P["qw"][1]))

# data on device, once
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, M01=None, bs=EVAL_BATCH_SIZE):
    outs = [C.pred(w, M01, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)]
    out = jnp.concatenate(outs); out.block_until_ready()
    return np.asarray(out)

# ==============================================================================
# 5. METRICS  (log scale: R2/MAE/MSE/RMSE.  pT in GeV: MAE/MSE/RMSE only.  per-GeV-class table)
# ==============================================================================
def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

# ==============================================================================
# 6. TRAIN ONE (variant, seed)
# ==============================================================================
def train_one(C, seed):
    variant = C.variant
    P0 = init_params(variant, seed)
    w, _ = ravel_pytree(P0); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, bad, escapes = [], float("inf"), w, 0, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and variant == "quantum":
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        pv = predict_log(C, w, Xva, pva, bva)
        vm, _ = split_metrics(pv, y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep, bad = vm["loss"], jnp.array(w), ep, 0; flag = " *best"
        else:
            bad += 1
        print(f"[{variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes one-time XLA compile)" if ep == 1 and seed == SEEDS[0] else ""), flush=True)
        if EARLY_STOPPING and bad >= EARLY_STOPPING_PATIENCE:
            print(f"   early stop at epoch {ep}"); break
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

# ==============================================================================
# 7. TEST: metrics + per-class table + INFERENCE TIMING
# ==============================================================================
def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def timing_block(C, w, M01):
    n = Xte.shape[0]
    def full_pass(Mx):
        outs = [C.pred(w, Mx, Xte[i:i + EVAL_BATCH_SIZE], pte[i:i + EVAL_BATCH_SIZE], bte[i:i + EVAL_BATCH_SIZE])
                for i in range(0, n, EVAL_BATCH_SIZE)]
        jnp.concatenate(outs).block_until_ready()
    out = {}
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        full_pass(Mx)                                           # warm-up / compile
        tt = _med(lambda: full_pass(Mx), 3)
        out[name] = dict(test_pass_s=tt, us_per_row=1e6 * tt / n, rows_per_s=n / tt)
    for name, Mx in ([("graph", None), ("folded", M01)] if M01 is not None else [("graph", None)]):
        lat = {}
        for b in (1, 64, 1024):
            f = lambda: C.pred(w, Mx, Xte[:b], pte[:b], bte[:b]).block_until_ready()
            f(); lat[b] = 1e3 * _med(f, 200 if b == 1 else 60)
        out["latency_" + name] = lat
    return out

def evaluate_test(C, tr, seed):
    w = tr["w"]; M01 = C.fold(w)
    pred = predict_log(C, w, Xte, pte, bte, M01)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    if M01 is not None:                                          # folded path must equal the training graph
        m["fold_maxdiff"] = float(np.abs(pred - predict_log(C, w, Xte, pte, bte, None)).max())
    w16 = w.astype(jnp.float16).astype(jnp.float32)              # storage-precision check: weights rounded to float16
    m16, _ = split_metrics(predict_log(C, w16, Xte, pte, bte, C.fold(w16)), y_test, pT_test64)
    m["fp16_dR2_log"] = m16["r2_log"] - m["r2_log"]; m["fp16_dMAE_log"] = m16["mae_log"] - m["mae_log"]
    tm = timing_block(C, w, M01)
    return m, bins, tm

# ==============================================================================
# 8. PENNYLANE VERIFICATION (quantum variant only)
# ==============================================================================
def verify_against_pennylane():
    print("-" * 96 + "\nVERIFY: closed-form JAX circuit vs PennyLane default.qubit (full circuit incl. the 2nd IsingZZ layer, float64)\n" + "-" * 96)
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def ref(theta, w, a1, a2):
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(theta[:, k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[a, b])
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    rs = np.random.default_rng(7)
    th = rs.normal(size=(64, 4)) * 1.5; w = rs.normal(size=(2, 4, 2)) * 2
    a1 = rs.uniform(0, math.pi, (64, 4)); a2 = rs.uniform(0, math.pi, (64, 4))
    zr = np.stack([np.asarray(o) for o in ref(th, w, a1, a2)], -1)
    zj = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), _layer(jnp.asarray(w[0], jnp.float32)),
                                 _layer(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    zsv = np.asarray(circuit_core_statevector(jnp.asarray(th, jnp.float32), _var_M_sv(jnp.asarray(w[0], jnp.float32)),
                                              _var_M_sv(jnp.asarray(w[1], jnp.float32)), jnp.asarray(a1, jnp.float32)))
    print(f"  max|PennyLane - old state-vector JAX| = {np.abs(zr - zsv).max():.2e}")
    zr0 = np.stack([np.asarray(o) for o in ref(th, w, a1, np.zeros_like(a2))], -1)
    d = np.abs(zr - zj).max()
    print(f"  max|PennyLane - FAST closed-form JAX| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
    print(f"  max|PennyLane(with 2nd RZZ layer) - PennyLane(2nd layer angles = 0)| = {np.abs(zr - zr0).max():.2e}  -> 2nd layer is inert")
    assert d < 5e-5, "JAX circuit disagrees with PennyLane"
    print("  -> PASS\n")

def print_circuit():
    dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev)
    def c(th, w, a1):
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[0, k, 1], wires=k); qml.RY(w[0, k, 0], wires=k)
        for i, (a, b) in enumerate(RING_PAIRS): qml.IsingZZ(a1[i], wires=[a, b])
        for k in range(N_QUBITS): qml.RY(th[k], wires=k)
        for k in range(N_QUBITS): qml.RZ(w[1, k, 1], wires=k); qml.RY(w[1, k, 0], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(N_QUBITS)]
    print("Quantum interaction block (per edge; theta = pi*tanh(pair_proj), RZZ angles = pi*sigmoid(entangle_gate1(theta))):")
    print(qml.draw(c)(np.zeros(4), np.zeros((2, 4, 2)), np.zeros(4)))

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 11. DISTILLATION
# ==============================================================================
print("\n" + "#" * 96 + "\n# CELL 5: distilling the quantum block into classical parameters\n" + "#" * 96)
Ct = Compiled("quantum")
_sl1j = _sl1

def tf_from(P, x):
    h = _lin(P["node"], x)
    u = _lin(P["pair"], jnp.concatenate([h[:, RECV], h[:, SEND]], -1))
    return (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)

@jax.jit
def teacher_block(Pt, M0, M1, tf):
    a1 = jax.nn.sigmoid(_lin(Pt["gate1"], tf)) * math.pi
    return circuit_core(tf, M0, M1, a1)

def student_block(c, tf):
    return jnp.tanh(jnp.tanh(tf @ c["W1"].T + c["b1"]) @ c["W2"].T)

def forward_s(P, x, phi, ba):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x); h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, N_QUBITS)
    z = student_block(P["c"], tf).reshape(B, N_EDGES, N_QUBITS)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), tf, z

def student_init(Pt, H, seed, random_block=True):
    ks = jax.random.split(jax.random.PRNGKey(10_000 + seed), 2)
    l1, l2 = _init_lin(ks[0], N_QUBITS, H), _init_lin(ks[1], H, N_QUBITS)
    P = {k: Pt[k] for k in ("node", "pair", "phys", "head1", "head2")}      # trunk + head warm-started from the teacher
    P["c"] = {"W1": l1["W"], "b1": l1["b"], "W2": l2["W"]}
    return P

def n_params(P): return int(sum(a.size for a in jax.tree_util.tree_leaves(P)))

def chunked(fn, *arrs, bs=EVAL_BATCH_SIZE):
    return jnp.concatenate([fn(*[a[i:i + bs] for a in arrs]) for i in range(0, arrs[0].shape[0], bs)])

def make_adam_epoch(loss_fn, wd):
    @jax.jit
    def epoch(w, mm, vv, t, lr, batches):
        def step(carry, b):
            w, mm, vv, t = carry
            l, g = jax.value_and_grad(lambda ww: loss_fn(ww, *b))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - lr * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), l
        (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), batches)
        return w, mm, vv, t, ls.mean()
    return epoch

def np_forward(Pn, x, phi, ba):
    """Pure-NumPy student inference (no JAX, no PennyLane): this is the deployable model."""
    lin = lambda p, a: a @ p["W"].T + p["b"]
    h = lin(Pn["node"], x); h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = np.arctan2(np.sin(d), np.cos(d)) if PHI_IS_RADIANS else d
    r = np.stack([np.abs(ba[:, SEND] - ba[:, RECV]), np.abs(dphi)], -1)
    Pp = lin(Pn["phys"], (r - np.asarray(RM)) / np.asarray(RS))[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * np.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * np.tanh(lin(Pn["pair"], np.concatenate([h_i, h_j], -1)))).reshape(-1, N_QUBITS)
    c = Pn["c"]; z = np.tanh(np.tanh(tf @ c["W1"].T + c["b1"]) @ c["W2"].T).reshape(len(x), N_EDGES, N_QUBITS)
    s = (z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp).reshape(len(x), N_NODES, N_NODES - 1)
    e = np.exp(s - s.max(-1, keepdims=True)); alpha = e / e.sum(-1, keepdims=True)
    m = (alpha[..., None] * (h_j + z).reshape(len(x), N_NODES, N_NODES - 1, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = (np.tanh(g @ Pn["head1"]["W"].T + Pn["head1"]["b"]) @ Pn["head2"]["W"].T + Pn["head2"]["b"])[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def fidelity(zs, zt):
    zs, zt = np.asarray(zs, np.float64), np.asarray(zt, np.float64)
    return dict(z_r2=float(1 - ((zs - zt) ** 2).sum() / ((zt - zt.mean(0)) ** 2).sum()), z_rmse=float(np.sqrt(((zs - zt) ** 2).mean())),
                z_maxerr=float(np.abs(zs - zt).max()))

def eval_student(pred_fn, w, Xs, ps, bs_, y_log, pt_true, pt_teacher_log=None):
    p = np.asarray(chunked(lambda a, b, c: pred_fn(w, a, b, c), Xs, ps, bs_))
    m, pt_pred = split_metrics(p, y_log, pt_true)
    if pt_teacher_log is not None:
        m["agree_r2_vs_teacher"] = float(1 - ((p - pt_teacher_log) ** 2).sum() / ((pt_teacher_log - pt_teacher_log.mean()) ** 2).sum())
        m["agree_mae_vs_teacher"] = float(np.abs(p - pt_teacher_log).mean())
    return m, p, pt_pred

def distil_one(seed):
    f = os.path.join(RESULTS_DIR, f"piqgat_quantum_seed{seed}.npz")
    assert os.path.isfile(f), f"{f} not found -- run Cell 3 with this seed first"
    wt = jnp.asarray(np.load(f)["w"]); Pt = Ct.unravel(wt); M01 = Ct.fold(wt)
    # ---- teacher reference ----
    yT_tr = np.asarray(predict_log(Ct, wt, Xtr, ptr, btr, M01)); yT_va = predict_log(Ct, wt, Xva, pva, bva, M01)
    yT_te = predict_log(Ct, wt, Xte, pte, bte, M01)
    mT, ptT = split_metrics(yT_te, y_test, pT_test64); binsT = bin_stats(pT_test64, ptT, yT_te, y_test)
    print(f"\n[teacher seed {seed}] {n_params(Pt)} params | TEST R2_log {mT['r2_log']:.4f}  MAE_log {mT['mae_log']:.4f}  MAE_pT {mT['mae_pt']:.3f} GeV")
    # ---- theta / z_teacher on every edge (teacher trunk), train + val ----
    tfj = jax.jit(tf_from)
    TF = chunked(lambda x: tfj(Pt, x), Xtr); ZT = chunked(lambda t: teacher_block(Pt, M01[0], M01[1], t), TF)
    TFv = chunked(lambda x: tfj(Pt, x), Xva); ZTv = chunked(lambda t: teacher_block(Pt, M01[0], M01[1], t), TFv)
    print(f"   edge samples for the block fit: {TF.shape[0]:,} train / {TFv.shape[0]:,} val   (z std per qubit: "
          f"{np.round(np.asarray(ZT.std(0)), 3).tolist()})")
    out = dict(seed=seed, teacher=dict(test=mT, bins=binsT, n_par=n_params(Pt)), students={})
    rng = np.random.default_rng(seed)
    plans = [(H, "distilled") for H in H_LIST] + ([(HEADLINE_H, "control")] if RUN_CONTROL else [])
    for H, kind in plans:
        P0 = student_init(Pt, H, seed); npar = n_params(P0); name = f"{kind}_H{H}"
        print(f"\n--- student {name}: {npar} params (block {n_params(P0['c'])}) ---")
        wS, unr = ravel_pytree(P0)
        pred_s = jax.jit(lambda w, x, p, b: forward_s(unr(w), x, p, b)[0])
        rec = dict(H=H, kind=kind, n_par=npar)
        if kind == "distilled":
            # ---------------- STAGE A: label-free block fit ----------------
            c0 = P0["c"]; wc, unr_c = ravel_pytree(c0)
            epA = make_adam_epoch(lambda w, t, z: jnp.mean((student_block(unr_c(w), t) - z) ** 2), 0.0)
            mm, vv, tt = jnp.zeros_like(wc), jnp.zeros_like(wc), jnp.float32(0.0)
            nbA = TF.shape[0] // A_BATCH; tA = time.time()
            for ep in range(1, A_EPOCHS + 1):
                perm = jnp.asarray(rng.permutation(TF.shape[0]))[:nbA * A_BATCH]
                lr = A_LR * (1 - 0.9 * (ep - 1) / max(A_EPOCHS - 1, 1))
                wc, mm, vv, tt, l = epA(wc, mm, vv, tt, lr, (TF[perm].reshape(nbA, A_BATCH, N_QUBITS), ZT[perm].reshape(nbA, A_BATCH, N_QUBITS)))
                if ep in (1, 5, 10, 20, A_EPOCHS):
                    fv = fidelity(student_block(unr_c(wc), TFv), ZTv)
                    print(f"   [A] ep {ep:2d}/{A_EPOCHS}  block-fit MSE {float(l):.5f} | val z-R2 {fv['z_r2']:.4f}  z-RMSE {fv['z_rmse']:.4f}  max|dz| {fv['z_maxerr']:.3f}", flush=True)
            P0 = dict(P0); P0["c"] = unr_c(wc); wS, unr = ravel_pytree(P0)
            pred_s = jax.jit(lambda w, x, p, b: forward_s(unr(w), x, p, b)[0])
            rec["stageA_secs"] = time.time() - tA; rec["fid_val"] = fidelity(student_block(P0["c"], TFv), ZTv)
            mA, _, _ = eval_student(pred_s, wS, Xte, pte, bte, y_test, pT_test64, yT_te)
            rec["zero_shot_test"] = mA
            print(f"   [A] ZERO-SHOT swap (no labels, nothing else changed): TEST R2_log {mA['r2_log']:.4f}  MAE_log {mA['mae_log']:.4f}  "
                  f"| agreement with teacher R2 {mA['agree_r2_vs_teacher']:.4f}")
        # ---------------- STAGE B: KD fine-tune (all params) ----------------
        lam_y, lam_t, lam_z = (LAM_Y, LAM_T, LAM_Z) if kind == "distilled" else (1.0, 0.0, 0.0)
        def loss_b(w, x, p, b, y, yT):
            ys, tf, zs = forward_s(unr(w), x, p, b)
            l = lam_y * _sl1j(ys, y)
            if lam_t: l = l + lam_t * _sl1j(ys, yT)
            if lam_z:
                zt = jax.lax.stop_gradient(teacher_block(Pt, M01[0], M01[1], tf))
                l = l + lam_z * jnp.mean((zs.reshape(-1, N_QUBITS) - zt) ** 2)
            return l
        epB = make_adam_epoch(loss_b, CLASSICAL_WEIGHT_DECAY)
        mm, vv, tt = jnp.zeros_like(wS), jnp.zeros_like(wS), jnp.float32(0.0)
        nfull = NB * BATCH_SIZE; shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
        yT_trj = jnp.asarray(yT_tr, jnp.float32)
        pv0 = np.asarray(chunked(lambda a, b, c: pred_s(wS, a, b, c), Xva, pva, bva))
        best, best_w, best_ep, hist = split_metrics(pv0, y_val, pT_val)[0]["loss"], wS, 0, []
        tB = time.time()
        for ep in range(1, B_EPOCHS + 1):
            t0 = time.time(); perm = jnp.asarray(rng.permutation(N_TR))
            wS, mm, vv, tt, tl = epB(wS, mm, vv, tt, B_LR, (shp(Xtr[perm]), shp(ptr[perm]), shp(btr[perm]), shp(ytr[perm]), shp(yT_trj[perm])))
            pv = np.asarray(chunked(lambda a, b, c: pred_s(wS, a, b, c), Xva, pva, bva))
            vm, _ = split_metrics(pv, y_val, pT_val)
            flag = ""
            if vm["loss"] < best - 1e-6: best, best_w, best_ep, flag = vm["loss"], jnp.array(wS), ep, " *best"
            hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], sec=time.time() - t0))
            print(f"   [B {name}] ep {ep:2d}/{B_EPOCHS}  loss {float(tl):.4f}  val loss {vm['loss']:.4f}  R2_log {vm['r2_log']:.4f}  {time.time()-t0:4.1f}s{flag}", flush=True)
        # (best_ep 0 = the stage-A / warm-start state itself was best)
        wS = best_w
        m, p, pt_pred = eval_student(pred_s, wS, Xte, pte, bte, y_test, pT_test64, yT_te)
        rec.update(best_ep=best_ep, test=m, bins=bin_stats(pT_test64, pt_pred, p, y_test), hist=hist, stageB_secs=time.time() - tB)
        print(f"   >> {name} | best ep {best_ep} | TEST R2_log {m['r2_log']:.4f}  MAE_log {m['mae_log']:.4f}  MAE_pT {m['mae_pt']:.3f} GeV "
              f"| vs teacher: dR2_log {m['r2_log']-mT['r2_log']:+.4f}  agreement R2 {m['agree_r2_vs_teacher']:.4f}")
        # timing: full test pass
        full = lambda: chunked(lambda a, b, c: pred_s(wS, a, b, c), Xte, pte, bte).block_until_ready()
        full(); rec["us_per_row"] = 1e6 * _med(full, 3) / Xte.shape[0]
        # pure-NumPy deployable check
        Pn = jax.tree_util.tree_map(lambda a: np.asarray(a, np.float64), unr(wS))
        npred = np_forward(Pn, np.asarray(Xte[:20000], np.float64), np.asarray(pte[:20000], np.float64), np.asarray(bte[:20000], np.float64))
        rec["numpy_maxdiff"] = float(np.abs(npred - p[:20000]).max())
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_student_{name}_seed{seed}.npz"), w=np.asarray(wS), w_fp16=np.asarray(wS.astype(jnp.float16)))
        out["students"][name] = rec
    # teacher timing for the table
    Cf = lambda: chunked(lambda a, b, c: Ct.pred(wt, M01, a, b, c), Xte, pte, bte).block_until_ready()
    Cf(); out["teacher"]["us_per_row"] = 1e6 * _med(Cf, 3) / Xte.shape[0]
    return out

ALLD = [distil_one(s) for s in SEEDS]

# ==============================================================================
# 12. REPORT
# ==============================================================================
ref = {}
_cf = os.path.join(RESULTS_DIR, "piqgat_results_ablation.json")
if os.path.isfile(_cf):
    for v, rs in json.load(open(_cf)).items():
        ref[v] = {r["seed"]: r["test"] for r in rs}
print("\n" + "=" * 118 + f"\nFINAL REPORT | quantum-block distillation | seeds {SEEDS} | TEST set, log1p(pT) scale; mean +- std over seeds\n" + "=" * 118)
print(f"{'model':<26}{'params':>8}{'R2_log':>18}{'MAE_log':>18}{'MAE_pT (GeV)':>18}{'dR2 vs teacher':>16}{'agree R2':>10}{'us/row':>8}")
def row(lab, npar, ms_list, us, tR2=None, agree=None):
    r2 = [m["r2_log"] for m in ms_list]
    d = f"{np.mean(np.array(r2) - np.array(tR2)):+.4f}" if tR2 is not None else "-"
    ag = f"{np.mean([m['agree_r2_vs_teacher'] for m in ms_list]):.4f}" if agree else "-"
    print(f"{lab:<26}{npar:>8}{ms(r2, '{:.4f}'):>18}{ms([m['mae_log'] for m in ms_list], '{:.4f}'):>18}"
          f"{ms([m['mae_pt'] for m in ms_list], '{:.3f}'):>18}{d:>16}{ag:>10}{us:>8.2f}")
tR2 = [d["teacher"]["test"]["r2_log"] for d in ALLD]
row("TEACHER (quantum)", ALLD[0]["teacher"]["n_par"], [d["teacher"]["test"] for d in ALLD], np.mean([d["teacher"]["us_per_row"] for d in ALLD]), None)
for H in H_LIST:
    nm = f"distilled_H{H}"
    rs = [d["students"][nm] for d in ALLD]
    row(f"  A only (zero-shot) H={H}", rs[0]["n_par"], [r["zero_shot_test"] for r in rs], np.mean([r["us_per_row"] for r in rs]), tR2, True)
    row(f"  A+B distilled   H={H}", rs[0]["n_par"], [r["test"] for r in rs], np.mean([r["us_per_row"] for r in rs]), tR2, True)
if RUN_CONTROL:
    rs = [d["students"][f"control_H{HEADLINE_H}"] for d in ALLD]
    row(f"  CONTROL (labels) H={HEADLINE_H}", rs[0]["n_par"], [r["test"] for r in rs], np.mean([r["us_per_row"] for r in rs]), tR2, True)
for v in ("classical36", "classical16", "noquantum"):
    if v in ref:
        ms_ = [ref[v][s] for s in SEEDS if s in ref[v]]
        if ms_: print(f"{'  Cell 4 ' + v + ' (scratch)':<26}{'':>8}{ms([m['r2_log'] for m in ms_], '{:.4f}'):>18}{ms([m['mae_log'] for m in ms_], '{:.4f}'):>18}"
                      f"{ms([m['mae_pt'] for m in ms_], '{:.3f}'):>18}{'':>16}{'':>10}{'':>8}")

print("\nBLOCK FIDELITY after stage A (validation edges):")
for H in H_LIST:
    fv = [d["students"][f"distilled_H{H}"]["fid_val"] for d in ALLD]
    print(f"   H={H:<3} z-R2 {ms([f['z_r2'] for f in fv], '{:.4f}')}   z-RMSE {ms([f['z_rmse'] for f in fv], '{:.4f}')}   max|dz| {ms([f['z_maxerr'] for f in fv], '{:.3f}')}")

print(f"\nPER pT CLASS, test (seed-mean): teacher vs distilled H={HEADLINE_H}" + (" vs control" if RUN_CONTROL else ""))
print(f"{'pT class (GeV)':>16}{'N':>9}{'MAE_pT teach':>14}{'MAE_pT dist':>13}" + (f"{'MAE_pT ctrl':>13}" if RUN_CONTROL else "") + f"{'MAE_log teach':>15}{'MAE_log dist':>14}")
for i, b0 in enumerate(ALLD[0]["teacher"]["bins"]):
    g = lambda get: np.mean([get(d, i) for d in ALLD])
    line = (f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}{g(lambda d, i: d['teacher']['bins'][i]['mae_pt']):>14.3f}"
            f"{g(lambda d, i: d['students'][f'distilled_H{HEADLINE_H}']['bins'][i]['mae_pt']):>13.3f}")
    if RUN_CONTROL: line += f"{g(lambda d, i: d['students'][f'control_H{HEADLINE_H}']['bins'][i]['mae_pt']):>13.3f}"
    line += (f"{g(lambda d, i: d['teacher']['bins'][i]['mae_log']):>15.4f}"
             f"{g(lambda d, i: d['students'][f'distilled_H{HEADLINE_H}']['bins'][i]['mae_log']):>14.4f}")
    print(line)
hs = [d["students"][f"distilled_H{HEADLINE_H}"] for d in ALLD]
print(f"\nDEPLOYMENT: the H={HEADLINE_H} student is {hs[0]['n_par']} params, pure tanh/linear/softmax; pure-NumPy re-implementation max|diff| vs JAX "
      f"{max(h['numpy_maxdiff'] for h in hs):.1e}. Inference {np.mean([h['us_per_row'] for h in hs]):.2f} us/row vs teacher "
      f"{np.mean([d['teacher']['us_per_row'] for d in ALLD]):.2f} us/row (folded circuit, same device, same batch size).")
print("HOW TO READ: A-only = pure representability of the circuit; A+B vs CONTROL = value of the teacher's knowledge beyond a warm-started trunk;\n"
      "             A+B vs teacher = accuracy cost (or gain) of removing the circuit; H sweep = classical capacity needed to absorb it.")

_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(ALLD, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

Backend: JAX 0.11.1 | running on cpu:0 (all devices: [CpuDevice(id=0)])   |   seeds [42]   |   variants ['distilled']
Train 235871 | Val 235871 | Test 707614 rows   |   epochs 50, batch 64, lr 0.001, wd 3e-05, clip 1.0, SmoothL1 beta 1.0

################################################################################################
# CELL 5: distilling the quantum block into classical parameters
################################################################################################

[teacher seed 42] 144 params | TEST R2_log 0.8395  MAE_log 0.2143  MAE_pT 11.852 GeV
   edge samples for the block fit: 2,830,452 train / 2,830,452 val   (z std per qubit: [0.2930000126361847, 0.7929999828338623, 0.5189999938011169, 0.734000027179718])

--- student distilled_H4: 144 params (block 36) ---
   [A] ep  1/40  block-fit MSE 0.27634 | val z-R2 0.5677  z-RMSE 0.4055  max|dz| 1.720
   [A] ep  5/40  block-fit MSE 0.11628 | val z-R2 0.6987  z-RMSE 0.3386  max|dz| 1.877
   [A] ep 10/40  bloc

##physics prior on vs off

In [ ]:
"""
================================================================================
CELL -- PHYSICS PRIOR ON vs OFF  (PI-QGAT, current Cell 3 model, 144 params)                 [FAST JAX BACKEND]
Needs: ONLY Cell 0 (installs) and Cell 1 (data).

QUESTION: how much does the physics-informed attention prior actually contribute?
  attention logit of edge i<-j:   s_ij = mean_k z_ij,k  +  lambda * P_ij
                                  P_ij = 6 * tanh( physics_prior(std(|d bending|, |d phi|)) / 6 )
  prior_ON  : lambda = 0.1   (the full Cell 3 model)
  prior_OFF : lambda = 0     (everything else IDENTICAL: same 144 params incl. the 3 physics_prior weights, which
                              simply get no signal; same initial weights, same seed, same batch order, same Cell 3
                              training recipe: SmoothL1, Adam 1e-3, wd 3e-5 (none on circuit angles), clip 1.0,
                              batch 64, 50 epochs, best-val checkpoint, gamma escape)

REPORTS
  A) per model: test log1p(pT) R2/MAE/MSE/RMSE, pT MAE/MSE/RMSE (GeV), clamped rows, per-pT-class table,
     convergence (val R2 at epochs 1/2/5/10/25/50, epochs-to-2%, best epoch), training time
  B) PAIRED ON-vs-OFF difference over seeds: mean +- std, wins, paired t-test, Wilcoxon (needs >= 2 seeds)
  C) inference-time check: trained prior_ON model with lambda set to 0 at test time (no retraining)
  D) attention diagnostics of prior_ON: how much of the attention logit the prior drives

SEEDS: the prior's effect is small, so one seed cannot show it reliably. Default = 5 seeds
       (2 models x 5 seeds = 10 trainings, ~4-5 min each on CPU). For a quick look use SEEDS = [SEED].
================================================================================
"""
SEEDS = [42, 1, 60, 100, 120]
VARIANTS = {
    "prior_ON":  dict(lam=0.1, mask=(1, 1), bound=True, shuffle=False, use=True),
    "prior_OFF": dict(lam=0.0, mask=(1, 1), bound=True, shuffle=False, use=True),   # lambda = 0, nothing else changed
}
DIAG_ROWS = 65536
RUN_ATTENTION_DIAGNOSTICS = True
RESULTS_FILE = "piqgat_physics_prior_on_off.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3; the prior settings come from each variant)
# ==============================================================================
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 110)
print(f"PHYSICS PRIOR ON vs OFF (PI-QGAT, 144 params) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM 4-QUBIT CIRCUIT  (identical to Cell 3; verified there against PennyLane)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. PI-QGAT MODEL  (identical to Cell 3; the prior is configured per variant)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(spec, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> "full" starts identical to Cell 3
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
         "pair": _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS), "gate1": _init_lin(ks[4], N_QUBITS, len(RING_PAIRS)),
         "qw": (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi}
    if spec["use"]:
        P["phys"] = _init_lin(ks[2], 2, 1)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward_parts(P, x, phi, ba, spec):
    """returns (y_hat, quantum attention term (B,12), prior attention term lambda*P (B,12), attention weights (B,4,3))."""
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
    tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z = circuit_core(tf, P["qw"][0], P["qw"][1], a1).reshape(B, N_EDGES, N_QUBITS)
    q = z.mean(-1)
    if spec["use"]:
        d = phi[:, SEND] - phi[:, RECV]
        dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
        r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
        Pp = _lin(P["phys"], ((r - RM) / RS) * jnp.asarray(spec["mask"], jnp.float32))[..., 0]
        if spec["bound"]:
            Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
        p = spec["lam"] * Pp
    else:
        p = jnp.zeros_like(q)
    alpha = jax.nn.softmax((q + p).reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), q, p, alpha

def forward(P, x, phi, ba, spec):
    return forward_parts(P, x, phi, ba, spec)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, name, spec=None):
        self.name, self.spec = name, dict(spec or VARIANTS[name])
        spec = self.spec
        P0 = init_params(spec, 0)
        _, self.unravel = ravel_pytree(P0)
        self.n_dead = (2 - sum(spec["mask"])) if spec["use"] else 0
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, spec), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd_vec * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, x, phi, ba):
            return forward(unravel(w), x, phi, ba, spec)

        @jax.jit
        def parts(w, x, phi, ba):
            return forward_parts(unravel(w), x, phi, ba, spec)[1:]

        self.epoch, self.one_step, self.pred, self.parts = epoch, one_step, pred, parts

# ==============================================================================
# 4. DATA + METRICS  (metrics identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, PTR, BTR, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, PVA, BVA = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, PTE, BTE = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

# fixed event-level shuffles used ONLY by the "shuffled" variant (prior inputs come from another event)
_sh = np.random.default_rng(12345)
SHUF = {k: jnp.asarray(_sh.permutation(n)) for k, n in (("tr", Xtr.shape[0]), ("va", Xva.shape[0]), ("te", Xte.shape[0]))}

def prior_inputs(spec, split):
    p, b = {"tr": (PTR, BTR), "va": (PVA, BVA), "te": (PTE, BTE)}[split]
    return (p[SHUF[split]], b[SHUF[split]]) if spec["shuffle"] else (p, b)

def predict_log(C, w, X, P_, B_, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([C.pred(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 5. TRAIN ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    ptr, btr = prior_inputs(C.spec, "tr"); pva, bva = prior_inputs(C.spec, "va")
    w, _ = ravel_pytree(init_params(C.spec, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.name} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}", flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

def evaluate_test(C, w):
    pte, bte = prior_inputs(C.spec, "te")
    pred = predict_log(C, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    return m, bin_stats(pT_test64, pt_pred, pred, y_test)

def attention_diag(C, w):
    pte, bte = prior_inputs(C.spec, "te")
    q, p, alpha = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])
    q, p, a = np.asarray(q, np.float64).ravel(), np.asarray(p, np.float64).ravel(), np.asarray(alpha, np.float64)
    s = q + p
    ent = float((-(a * np.log(np.clip(a, 1e-12, 1))).sum(-1)).mean() / math.log(N_NODES - 1))   # 1 = uniform attention
    return dict(std_quantum=float(q.std()), std_prior=float(p.std()),
                prior_share=float(p.var() / max(s.var(), 1e-12)),
                corr=float(np.corrcoef(q, p)[0, 1]) if p.std() > 0 and q.std() > 0 else float("nan"),
                attn_entropy=ent)

# ==============================================================================
# 6. TRAIN BOTH MODELS FOR EVERY SEED
# ==============================================================================
ALL, MODELS = {}, {}
for name in VARIANTS:
    C = Compiled(name); spec = C.spec
    n_total = int(ravel_pytree(init_params(spec, 0))[0].shape[0])
    print("\n" + "#" * 110 + f"\n# {name}: lambda = {spec['lam']}   |   {n_total} params"
          + ("   (the 3 physics_prior weights receive no gradient)" if spec["lam"] == 0 else "") + "\n" + "#" * 110)
    res = []
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins = evaluate_test(C, tr["w"])
        print(f"   >> {name} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  RMSE {m['rmse_pt']:.3f} | "
              f"clamped {m['n_clamped']:.0f} | train {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        MODELS[(name, seed)] = (C, tr["w"])
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_{name}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"],
                        escapes=tr["escapes"], test=m, bins=bins, hist=tr["hist"]))
    ALL[name] = res

# ==============================================================================
# 7. A) PER-MODEL REPORT
# ==============================================================================
def ep2(r):
    best_v = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * best_v), np.nan)

ORDER = list(VARIANTS)
print("\n" + "=" * 130 + f"\nA) TEST SET  (mean +- std over seeds {SEEDS})\n" + "=" * 130)
print(f"{'model':<11}{'params':>7}{'R2_log':>18}{'MAE_log':>18}{'MSE_log':>18}{'RMSE_log':>18}"
      f"{'MAE_pT GeV':>17}{'RMSE_pT GeV':>19}{'clamped':>9}")
for v in ORDER:
    rs = ALL[v]; g = lambda k: [r["test"][k] for r in rs]
    print(f"{v:<11}{rs[0]['n_par']:>7}{ms(g('r2_log')):>18}{ms(g('mae_log')):>18}{ms(g('mse_log')):>18}{ms(g('rmse_log')):>18}"
          f"{ms(g('mae_pt'), '{:.3f}'):>17}{ms(g('rmse_pt'), '{:.3f}'):>19}{np.mean(g('n_clamped')):>9.0f}")

print("\nPER-SEED TEST R2_log:")
print(f"{'model':<11}" + "".join(f"{'seed ' + str(s):>12}" for s in SEEDS))
for v in ORDER:
    print(f"{v:<11}" + "".join(f"{r['test']['r2_log']:>12.4f}" for r in ALL[v]))

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TRAINING TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'model':<11}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}")
for v in ORDER:
    rs = ALL[v]
    print(f"{v:<11}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}")

print("\nPER pT CLASS (test, seed-mean):")
print(f"{'pT class (GeV)':>16}{'N':>9}" + "".join(f"{v + ' MAE_pT':>18}" for v in ORDER) + f"{'diff':>9}"
      + "".join(f"{v + ' MAE_log':>19}" for v in ORDER))
for b0 in ALL[ORDER[0]][0]["bins"]:
    col = lambda v, k: np.mean([[b for b in r["bins"] if b["lo"] == b0["lo"]][0][k] for r in ALL[v]])
    print(f"{_label(b0['lo'], b0['hi']):>16}{b0['n']:>9}" + "".join(f"{col(v, 'mae_pt'):>18.3f}" for v in ORDER)
          + f"{col('prior_OFF', 'mae_pt') - col('prior_ON', 'mae_pt'):>+9.3f}" + "".join(f"{col(v, 'mae_log'):>19.4f}" for v in ORDER))
print("(diff = OFF minus ON; positive = the prior helps in that class)")

# ==============================================================================
# 8. B) PAIRED ON-vs-OFF DIFFERENCE  (same seed = same init + same batch order -> paired)
# ==============================================================================
on, off = {r["seed"]: r for r in ALL["prior_ON"]}, {r["seed"]: r for r in ALL["prior_OFF"]}
common = [s for s in SEEDS if s in on and s in off]
print("\n" + "=" * 110 + f"\nB) PAIRED DIFFERENCE  ON minus OFF  ({len(common)} seeds)\n" + "=" * 110)
print(f"{'metric':<10}{'mean diff':>11}{'std':>9}{'ON better':>11}{'paired t p':>12}{'Wilcoxon p':>12}{'d_z':>8}")
try:
    from scipy import stats as _st
except Exception:
    _st = None
PAIRED = {}
for lab, key, higher_better in [("R2_log", "r2_log", True), ("MAE_log", "mae_log", False), ("RMSE_log", "rmse_log", False),
                                ("MAE_pT", "mae_pt", False), ("RMSE_pT", "rmse_pt", False)]:
    d = np.array([on[s]["test"][key] - off[s]["test"][key] for s in common])
    wins = int((d > 0).sum()) if higher_better else int((d < 0).sum())
    sd = d.std(ddof=1) if len(d) > 1 else float("nan")
    p_t = p_w = float("nan")
    if _st is not None and len(d) > 1 and np.any(d != 0):
        p_t = float(_st.ttest_rel([on[s]["test"][key] for s in common], [off[s]["test"][key] for s in common]).pvalue)
        try:
            p_w = float(_st.wilcoxon(d).pvalue)
        except Exception:
            pass
    dz = d.mean() / sd if len(d) > 1 and sd > 0 else float("nan")
    PAIRED[key] = dict(mean=float(d.mean()), std=float(sd), wins=wins, n=len(d), p_t=p_t, p_w=p_w, d_z=float(dz))
    print(f"{lab:<10}{d.mean():>+11.5f}{sd:>9.5f}{f'{wins}/{len(d)}':>11}{p_t:>12.4f}{p_w:>12.4f}{dz:>8.2f}")
print("(R2: positive diff = prior ON better; errors: negative diff = prior ON better.  With 5 seeds the smallest possible\n"
      " Wilcoxon p is 0.0625, so use the t-test p, the win count and d_z together.)")

# ==============================================================================
# 9. C) INFERENCE-TIME CHECK: trained prior_ON model, lambda -> 0 at test time
# ==============================================================================
print("\n" + "=" * 110 + "\nC) TRAINED prior_ON MODEL WITH THE PRIOR SWITCHED OFF AT TEST TIME (no retraining)\n" + "=" * 110)
print(f"{'seed':<6}{'R2_log ON':>11}{'R2_log OFF@test':>17}{'dR2_log':>10}{'MAE_pT ON':>11}{'MAE_pT OFF@test':>17}{'dMAE_pT':>10}")
C_test_off = Compiled("prior_ON", dict(VARIANTS["prior_ON"], lam=0.0))
REL = {}
for s in common:
    Cn, wn = MODELS[("prior_ON", s)]
    m_on = on[s]["test"]
    m_off, _ = split_metrics(predict_log(C_test_off, wn, Xte, PTE, BTE), y_test, pT_test64)
    REL[s] = m_off
    print(f"{s:<6}{m_on['r2_log']:>11.4f}{m_off['r2_log']:>17.4f}{m_off['r2_log'] - m_on['r2_log']:>+10.4f}"
          f"{m_on['mae_pt']:>11.3f}{m_off['mae_pt']:>17.3f}{m_off['mae_pt'] - m_on['mae_pt']:>+10.3f}")
print("(big drop = the trained model relies on the prior; ~0 = the prior barely changes its attention)")

# ==============================================================================
# 10. D) ATTENTION DIAGNOSTICS (prior_ON)
# ==============================================================================
DIAG = {}
if RUN_ATTENTION_DIAGNOSTICS:
    print("\n" + "=" * 110 + f"\nD) ATTENTION DIAGNOSTICS, prior_ON  (first {DIAG_ROWS} test events; logit = quantum term + lambda*P)\n" + "=" * 110)
    print(f"{'seed':<6}{'std quantum':>13}{'std prior':>11}{'prior share of var(logit)':>27}{'corr(q, p)':>12}{'attn entropy':>14}")
    for s in common:
        Cn, wn = MODELS[("prior_ON", s)]
        d = attention_diag(Cn, wn); DIAG[s] = d
        print(f"{s:<6}{d['std_quantum']:>13.4f}{d['std_prior']:>11.4f}{100 * d['prior_share']:>26.1f}%{d['corr']:>12.3f}{d['attn_entropy']:>14.3f}")
    print("(attention entropy: 1.0 = uniform over the 3 incoming edges, 0 = all weight on one edge)")

# ==============================================================================
# 11. SAVE
# ==============================================================================
out = dict(models=ALL, paired=PAIRED, test_time_off={str(s): v for s, v in REL.items()},
           diag={str(s): v for s, v in DIAG.items()}, seeds=SEEDS,
           variants={k: {kk: (list(vv) if isinstance(vv, tuple) else vv) for kk, vv in v.items()} for k, v in VARIANTS.items()})
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")
print("HOW TO READ: prior_ON clearly better than prior_OFF in (almost) every seed -> the physics prior helps.\n"
      "             A mean gap smaller than its std, or wins near half the seeds -> no real contribution.")

PHYSICS PRIOR ON vs OFF (PI-QGAT, 144 params) | JAX 0.11.1 on cpu:0 | seeds [42, 1, 60, 100, 120]
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001

##############################################################################################################
# prior_ON: lambda = 0.1   |   144 params
##############################################################################################################
[prior_ON | seed 42] ep  1/50  train 0.1842  val loss 0.1540  R2_log 0.5052  MAE_log 0.3613  MAE_pT 15.023 GeV   32.0s *best
[prior_ON | seed 42] ep  2/50  train 0.1359  val loss 0.1007  R2_log 0.7002  MAE_log 0.3057  MAE_pT 13.826 GeV   11.3s *best
[prior_ON | seed 42] ep  3/50  train 0.0806  val loss 0.0707  R2_log 0.7977  MAE_log 0.2497  MAE_pT 12.681 GeV    7.3s *best
[prior_ON | seed 42] ep  4/50  train 0.0675  val loss 0.0640  R2_log 0.8179  MAE_log 0.2359  MAE_pT 12.231 GeV    9.0s *best
[prior_ON | seed 42] ep  5/50  train 0.0646  val loss 0.0618  R2_

##Entanglement Topology Ablation

In [ ]:
"""
================================================================================
EXPERIMENT: ENTANGLEMENT TOPOLOGY ABLATION  (current PI-QGAT, 144 params)                        [FAST JAX + JIT]
CELL -- T_topology (ring vs linear vs all-to-all vs none)
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

PURPOSE: does PI-QGAT's performance depend on the specific RING connectivity, or simply on HAVING entanglement?

THE CIRCUIT (per edge, same as Cell 3, 4 qubits):
    RY(theta) -> [RZ(gamma) RY(omega)] -> IsingZZ on the topology's pairs (angles a = pi*sigmoid(gate1(theta)),
    data dependent) -> RY(theta) re-upload -> [RZ(gamma) RY(omega)] -> <Z_k>
  (The old Q6 had a 2nd IsingZZ layer right before the measurement. ZZ is diagonal, so it commutes with Z and has NO
   effect on <Z_k>; the current model therefore has one entangling layer. The PennyLane check below includes that 2nd
   layer anyway and confirms the outputs are identical.)

TOPOLOGIES (4 qubits, qubit count fixed):
  T_ring     : (0,1) (1,2) (2,3) (3,0)                  4 pairs  [= Cell 3 PI-QGAT]
  T_linear   : (0,1) (1,2) (2,3)                        3 pairs  (ring minus one edge, nearest neighbour)
  T_all2all  : (0,1) (0,2) (0,3) (1,2) (1,3) (2,3)      6 pairs  (full connectivity)
  T_none     : no IsingZZ at all                        0 pairs  (entanglement floor)

PARAMETER BUDGET:
  entangle_gate1 is Linear(4, n_pairs), so a raw implementation gives linear 139 / ring 144 / all2all 154 / none 124
  params -- that mixes TOPOLOGY with CAPACITY.
  MATCH_PARAMS = True (default): every entangled topology uses gate1 = Linear(4, 4) -- the ring's budget -- and pair p
    reads gate output p % 4 (all-to-all: pairs 5, 6 reuse outputs 1, 2). Every entangled variant has exactly 144 params
    and the SAME initial weights; only the WIRING differs. T_none has no gate at all (124 params): with no entangler
    there is nothing to spend those 20 params on. Stated, not hidden.
  MATCH_PARAMS = False: each topology gets its natural parameter count.
  Both counts are printed either way.

Identical to Cell 3 otherwise: closed-form circuit (verified vs PennyLane), architecture, init key layout, seed,
batch order, SmoothL1 (beta 1) on log1p(pT) with standardized output, Adam 1e-3, wd 3e-5 (none on circuit angles),
clip 1.0, batch 64, 50 epochs, best-val checkpoint, gamma escape, metrics, per-pT-class report.

RUN TIME (CPU): ~4 min per topology per seed -> 4 topologies x 1 seed ~ 16 min; x 5 seeds ~ 80 min.
Session may disconnect? Run one topology at a time (VARIANTS_TO_RUN = ["T_linear"]); results accumulate in
TOPO_RESULTS across re-runs of this cell.
================================================================================
"""
VARIANTS_TO_RUN = ["T_ring", "T_linear", "T_all2all", "T_none"]
SEEDS = [SEED]                  # e.g. [42, 1, 60, 100, 120] -> paired comparison vs ring over seeds
MATCH_PARAMS = True
VERIFY_WITH_PENNYLANE = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_topology_ablation.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

TOPOLOGIES = {
    "T_ring":    [(0, 1), (1, 2), (2, 3), (3, 0)],
    "T_linear":  [(0, 1), (1, 2), (2, 3)],
    "T_all2all": [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)],
    "T_none":    [],
}
GATE_WIDTH = len(TOPOLOGIES["T_ring"])          # 4 -> the matched parameter budget
for v in VARIANTS_TO_RUN:
    assert v in TOPOLOGIES, f"unknown topology {v}"

def gate_width(topo):
    pairs = TOPOLOGIES[topo]
    return 0 if not pairs else (GATE_WIDTH if MATCH_PARAMS else len(pairs))

BASE_PARAMS = 124                               # everything except entangle_gate1
natural = lambda topo: BASE_PARAMS + ((NQ + 1) * len(TOPOLOGIES[topo]) if TOPOLOGIES[topo] else 0)
matched = lambda topo: BASE_PARAMS + ((NQ + 1) * GATE_WIDTH if TOPOLOGIES[topo] else 0)

print("=" * 110)
print(f"ENTANGLEMENT TOPOLOGY ABLATION (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | MATCH_PARAMS = {MATCH_PARAMS}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)
for name, pairs in TOPOLOGIES.items():
    print(f"  {name:<11} {len(pairs)} pairs  {str(pairs) if pairs else '(no entanglement)':<50} natural={natural(name):>4}  matched={matched(name):>4}")
print(f"\nVariants to run: {VARIANTS_TO_RUN}\n")

# ==============================================================================
# 2. CLOSED-FORM CIRCUIT FOR ANY ZZ PAIR SET  (ring case == Cell 3's circuit_core)
# ==============================================================================
def closed_form(theta, W, a, pairs):
    """theta (N,4), W (2,4,2) = (omega, gamma), a (N,P) ZZ angles (or None), pairs list -> <Z_k> (N,4). Exact."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0 = W[0, :, 0], W[0, :, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    if a is not None and len(pairs):
        ca, sa = jnp.cos(a), jnp.sin(a)
        hr = [jnp.ones_like(z[:, 0]) for _ in range(NQ)]
        hi = [jnp.zeros_like(z[:, 0]) for _ in range(NQ)]
        for p, (u, v) in enumerate(pairs):          # qubit q's coherence picks up (cos a - i <Z_other> sin a) per ZZ partner
            for q, other in ((u, v), (v, u)):
                fr, fi = ca[:, p], -z[:, other] * sa[:, p]
                hr[q], hi[q] = hr[q] * fr - hi[q] * fi, hr[q] * fi + hi[q] * fr
        hr, hi = jnp.stack(hr, 1), jnp.stack(hi, 1)
        x, y = x * hr + y * hi, -(x * hi - y * hr)
    x, z = x * ct + z * st, z * ct - x * st         # data re-upload RY(theta)
    o1, g1 = W[1, :, 0], W[1, :, 1]
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. VERIFICATION vs PENNYLANE (float64), every topology, WITH the old 2nd ZZ layer before measurement
# ==============================================================================
if VERIFY_WITH_PENNYLANE:
    try:
        import pennylane as qml
        print("=" * 110 + "\nVERIFYING closed-form topology circuits vs PennyLane default.qubit (float64; reference includes a 2nd ZZ layer)\n" + "=" * 110)
        dev = qml.device("default.qubit", wires=NQ)
        rs = np.random.default_rng(9)
        for tname, pairs in TOPOLOGIES.items():
            th = rs.normal(size=(31, NQ)) * 1.5; W = rs.normal(size=(2, NQ, 2)) * 2
            a1 = rs.uniform(0, math.pi, (31, max(len(pairs), 1))); a2 = rs.uniform(0, math.pi, (31, max(len(pairs), 1)))

            @qml.qnode(dev)
            def ref(th, W, a1, a2, pairs=pairs):
                for k in range(NQ): qml.RY(th[:, k], wires=k)
                for k in range(NQ): qml.RZ(W[0, k, 1], wires=k); qml.RY(W[0, k, 0], wires=k)
                for i, (u, v) in enumerate(pairs): qml.IsingZZ(a1[:, i], wires=[u, v])
                for k in range(NQ): qml.RY(th[:, k], wires=k)
                for k in range(NQ): qml.RZ(W[1, k, 1], wires=k); qml.RY(W[1, k, 0], wires=k)
                for i, (u, v) in enumerate(pairs): qml.IsingZZ(a2[:, i], wires=[u, v])     # old 2nd layer (no effect on <Z>)
                return [qml.expval(qml.PauliZ(k)) for k in range(NQ)]

            o_ref = np.stack([np.asarray(o) for o in ref(th, W, a1, a2)], -1)
            o_cf = np.asarray(closed_form(jnp.asarray(th, jnp.float32), jnp.asarray(W, jnp.float32),
                                          jnp.asarray(a1, jnp.float32) if pairs else None, pairs))
            d = np.abs(o_ref - o_cf).max()
            print(f"  {tname:<11} max|PennyLane - closed form| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
            assert d < 5e-5, f"closed form disagrees with PennyLane for {tname}"
        print("  -> PASS\n")
    except ImportError:
        print("PennyLane not installed -> verification skipped (run Cell 0).")

# ==============================================================================
# 4. PI-QGAT WITH A CONFIGURABLE ENTANGLING TOPOLOGY  (T_ring == Cell 3 model, same init)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(topo, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> identical init where shapes agree
    P = {"node": _init_lin(ks[0], N_FEATS, NQ), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], NQ, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
         "pair": _init_lin(ks[1], 2 * NQ, NQ),
         "qw": (jax.random.uniform(ks[5], (2, NQ, 2), jnp.float32) * 2 - 1) * math.pi}
    if TOPOLOGIES[topo]:
        P["gate1"] = _init_lin(ks[4], NQ, gate_width(topo))
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def zz_angles(P, tf, topo):
    pairs = TOPOLOGIES[topo]
    if not pairs:
        return None
    a = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi                     # (N, gate width)
    idx = np.array([i % gate_width(topo) for i in range(len(pairs))])      # pair p reads gate output p % width
    return a[:, idx]                                                        # (N, n_pairs)

def forward_parts(P, x, phi, ba, topo):
    B = x.shape[0]
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, NQ)
    a = zz_angles(P, tf, topo)
    z = closed_form(tf, P["qw"], a, TOPOLOGIES[topo]).reshape(B, N_EDGES, NQ)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, N_IN, NQ)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z, (a if a is not None else jnp.zeros((tf.shape[0], 1)))

def forward(P, x, phi, ba, topo):
    return forward_parts(P, x, phi, ba, topo)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, topo):
        self.topo = topo
        P0 = init_params(topo, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, topo), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on qw)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, topo))
        self.parts = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, topo)[1:])

# ==============================================================================
# 5. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([C.pred(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE (topology, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(C.topo, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.topo} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    z, a = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])
    z, a = np.asarray(z, np.float64).reshape(-1, NQ), np.asarray(a, np.float64)
    full = lambda: predict_log(C, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    diag = dict(z_std=float(z.std(0).mean()), z_absmean=float(np.abs(z).mean()), us_per_row=us,
                zz_mean=float(a.mean()) if TOPOLOGIES[C.topo] else float("nan"),
                zz_std=float(a.std()) if TOPOLOGIES[C.topo] else float("nan"),
                ent_strength=float(np.abs(np.sin(a)).mean()) if TOPOLOGIES[C.topo] else 0.0)   # |sin a| = 0 -> no entangling
    return m, bins, diag

# ==============================================================================
# 7. RUN  (results accumulate in TOPO_RESULTS across re-runs of this cell)
# ==============================================================================
if "TOPO_RESULTS" not in globals():
    TOPO_RESULTS = {}

for topo in VARIANTS_TO_RUN:
    pairs = TOPOLOGIES[topo]
    C = Compiled(topo)
    n_par = int(ravel_pytree(init_params(topo, 0))[0].shape[0])
    print("\n" + "#" * 110 + f"\n# {topo}: {len(pairs)} pairs {pairs if pairs else '(no entanglement)'}  |  params {n_par} "
          f"(ring = 144, diff {n_par - 144:+d})  |  gate width {gate_width(topo)}\n" + "#" * 110)
    if pairs and len(pairs) > gate_width(topo):
        print(f"  NOTE: {len(pairs)} pairs share {gate_width(topo)} gate outputs "
              f"(mapping {[i % gate_width(topo) for i in range(len(pairs))]}) -- parameter budget held at the ring's")
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        print(f"   >> {topo} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        print(f"\n   [{topo} | seed {seed}] PER pT CLASS (test)")
        print(f"   {'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>10}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
        for b in bins:
            print(f"   {_label(b['lo'], b['hi']):>16}{b['n']:>9}{b['frac']:>6.1f}%{b['mae_log']:>9.4f}{b['mae_pt']:>10.3f}"
                  f"{b['mse_pt']:>13.2f}{b['rmse_pt']:>10.3f}{b['med_rel']:>10.3f}{b['bias']:>8.3f}{b['sse_share']:>6.1f}%")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_topology_{topo}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        TOPO_RESULTS[(topo, seed)] = dict(topo=topo, seed=seed, n_pairs=len(pairs), pairs=[list(p) for p in pairs],
                                          params=n_par, params_natural=natural(topo), params_matched=matched(topo),
                                          best_ep=tr["best_ep"], best_val=tr["best_val"], train_total_s=tr["train_total"],
                                          escapes=tr["escapes"], test=m, bins=bins, hist=tr["hist"], diag=diag)

# ==============================================================================
# 8. FINAL REPORT
# ==============================================================================
ORDER = [t for t in ("T_ring", "T_linear", "T_all2all", "T_none") if any(k[0] == t for k in TOPO_RESULTS)]
R = {t: [TOPO_RESULTS[k] for k in sorted(TOPO_RESULTS) if k[0] == t] for t in ORDER}
seeds_of = lambda t: sorted(r["seed"] for r in R[t])

print("\n" + "=" * 150 + "\nSUMMARY (test set, mean +- std over seeds)\n" + "=" * 150)
print(f"{'variant':<20}{'pairs':>6}{'params':>8}{'seeds':>7}{'best val loss':>15}{'R2_log':>18}{'MAE_log':>18}{'MSE_log':>10}"
      f"{'RMSE_log':>10}{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}{'escapes':>9}")
for t in ORDER:
    rs = R[t]; g = lambda k: [r["test"][k] for r in rs]
    tag = t + (" (Cell 3 ref)" if t == "T_ring" else "")
    print(f"{tag:<20}{rs[0]['n_pairs']:>6}{rs[0]['params']:>8}{len(rs):>7}{np.mean([r['best_val'] for r in rs]):>15.4f}"
          f"{ms(g('r2_log')):>18}{ms(g('mae_log')):>18}{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}"
          f"{ms(g('mae_pt'), '{:.3f}'):>17}{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}"
          f"{np.mean(g('n_clamped')):>9.0f}{np.mean([r['escapes'] for r in rs]):>9.1f}")
print(f"(params: MATCH_PARAMS = {MATCH_PARAMS}; natural counts " + ", ".join(f"{t} {natural(t)}" for t in ORDER) + ")")

# ---- deltas vs ring (paired by seed) ----
DELTAS = {}
if "T_ring" in R:
    ring = {r["seed"]: r for r in R["T_ring"]}
    print("\n" + "=" * 110 + "\nDELTAS vs T_ring  [variant minus ring, paired by seed; R2: + = variant better; errors: - = variant better]\n" + "=" * 110)
    print(f"{'variant':<12}{'seeds':>6}{'d R2_log':>18}{'d MAE_log':>18}{'d RMSE_log':>12}{'d MAE_pT':>16}{'d val loss':>12}{'better than ring':>18}")
    for t in ORDER:
        if t == "T_ring":
            continue
        common = [r["seed"] for r in R[t] if r["seed"] in ring]
        if not common:
            print(f"{t:<12} no seed in common with T_ring"); continue
        v = {r["seed"]: r for r in R[t]}
        d = lambda k: [v[s]["test"][k] - ring[s]["test"][k] for s in common]
        wins = sum(v[s]["test"]["r2_log"] > ring[s]["test"]["r2_log"] for s in common)
        DELTAS[t] = {k: d(k) for k in ("r2_log", "mae_log", "rmse_log", "mae_pt")}
        print(f"{t:<12}{len(common):>6}{ms(d('r2_log'), '{:+.4f}'):>18}{ms(d('mae_log'), '{:+.4f}'):>18}{np.mean(d('rmse_log')):>+12.4f}"
              f"{ms(d('mae_pt'), '{:+.3f}'):>16}{np.mean([v[s]['best_val'] - ring[s]['best_val'] for s in common]):>+12.4f}"
              f"{f'{wins}/{len(common)}':>18}")
else:
    print("\n(T_ring not run -> no deltas. Add 'T_ring' to VARIANTS_TO_RUN; it is the Cell 3 model with the same init.)")

# ---- spread among entangled topologies vs gap to no entanglement ----
ent = [t for t in ORDER if TOPOLOGIES[t]]
if len(ent) > 1:
    r2m = {t: np.mean([r["test"]["r2_log"] for r in R[t]]) for t in ORDER}
    spread = max(r2m[t] for t in ent) - min(r2m[t] for t in ent)
    print(f"\nSpread in R2_log ACROSS entangled topologies ({', '.join(ent)}): {spread:.4f}")
    if "T_none" in R:
        gap = max(r2m[t] for t in ent) - r2m["T_none"]
        print(f"Gap from best entangled topology to T_none:  {gap:+.4f}")
        if gap > 0:
            print(f"  ratio (topology spread / entanglement gap) = {spread / gap:.3f}")
            print("  ratio << 1 -> HAVING entanglement matters; WHICH topology matters little")
            print("  ratio ~ 1  -> the specific connectivity matters as much as entanglement itself")
    if any(len(R[t]) == 1 for t in ORDER):
        print("  (single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01, so gaps smaller than that are not reliable)")

# ---- per pT class ----
for key, title, fmt in (("med_rel", "med|rel| per pT class", "{:>13.3f}"), ("mae_pt", "MAE (GeV) per pT class", "{:>13.3f}"),
                        ("mae_log", "MAE_log per pT class", "{:>13.4f}")):
    print(f"\n{title} (test, seed-mean)")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{t:>13}" for t in ORDER))
    for b0 in R[ORDER[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r["bins"] if b["lo"] == b0["lo"]][0][key] for r in R[t]])) for t in ORDER))

# ---- convergence + timing ----
def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<12}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for t in ORDER:
    rs = R[t]
    print(f"{t:<12}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}")

# ---- circuit diagnostics ----
print(f"\nCIRCUIT DIAGNOSTICS (first {DIAG_ROWS} test events, seed-mean):")
print(f"{'variant':<12}{'std <Z>':>10}{'mean|<Z>|':>11}{'ZZ angle mean':>15}{'ZZ angle std':>14}{'mean|sin a|':>13}")
for t in ORDER:
    g = lambda k: np.nanmean([r["diag"][k] for r in R[t]])
    print(f"{t:<12}{g('z_std'):>10.4f}{g('z_absmean'):>11.4f}{g('zz_mean'):>15.3f}{g('zz_std'):>14.3f}{g('ent_strength'):>13.3f}")
print("(ZZ angles in radians, range 0..pi; mean|sin a| = 0 means the trained gates switched entanglement off)")

# ==============================================================================
# 9. SAVE
# ==============================================================================
out = dict(results={f"{t}_seed{s}": v for (t, s), v in TOPO_RESULTS.items()}, deltas_vs_ring=DELTAS,
           match_params=MATCH_PARAMS, topologies={k: [list(p) for p in v] for k, v in TOPOLOGIES.items()}, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

print("\nReading guide:")
print("  ring/linear/all2all close, all >> T_none -> ENTANGLEMENT matters, topology does not")
print("  ring clearly best                        -> the ring connectivity itself is doing work")
print("  all2all best                             -> more connectivity helps; ring is a compromise")
print("  everything close to T_none               -> entanglement contributes little on this task")

ENTANGLEMENT TOPOLOGY ABLATION (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | MATCH_PARAMS = True
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001
  T_ring      4 pairs  [(0, 1), (1, 2), (2, 3), (3, 0)]                   natural= 144  matched= 144
  T_linear    3 pairs  [(0, 1), (1, 2), (2, 3)]                           natural= 139  matched= 144
  T_all2all   6 pairs  [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]   natural= 154  matched= 144
  T_none      0 pairs  (no entanglement)                                  natural= 124  matched= 124

Variants to run: ['T_ring', 'T_linear', 'T_all2all', 'T_none']

VERIFYING closed-form topology circuits vs PennyLane default.qubit (float64; reference includes a 2nd ZZ layer)
  T_ring      max|PennyLane - closed form| = 2.08e-07   (float32 vs float64; must be < 5e-5)
  T_linear    max|PennyLane - closed form| = 2.12e-07   (float32 vs float64; must be < 5e-5)
  T_all2all   max|PennyLane - closed form| = 1.76e-07   (float3

## Adaptive RZZ to Fixed RZZ	Ablation

In [ ]:
"""
================================================================================
EXPERIMENT: ADAPTIVE RZZ -> FIXED RZZ   (current PI-QGAT, 144 params)                        [FAST JAX + JIT]
CELL -- F_fixed_rzz (PI-QGAT with data-dependent RZZ angles replaced by learned GLOBAL angles)
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

PURPOSE: test directly whether ADAPTIVE (data-dependent) entanglement is responsible for PI-QGAT's performance, or
whether a FIXED learned entangling pattern does just as well.

THE CHANGE (circuit structure otherwise untouched):
  adaptive (Cell 3) : a = pi * sigmoid(entangle_gate1(theta))    -> per edge, per event   (gate1 = Linear(4,4), 20 params)
  fixed             : a = pi * sigmoid(fixed_angle_logits)       -> SAME for every event  (4 params)
  Same sigmoid*pi parameterisation, same [0, pi] range, same ring pairs (0,1)(1,2)(2,3)(3,0), same encoding, re-upload
  and measurement. Fixed logits start at 0 -> angle pi/2 (middle of the range), as in the old F6 cell.
  (The old Q6 had a 2nd RZZ layer right before the measurement; it commutes with Z and has no effect on <Z_k>, so the
   current model has one RZZ layer. The PennyLane check below includes that 2nd layer and confirms identical outputs.)

PARAMETER COUNT:
  adaptive       : 144 params (= Cell 3)
  fixed_plain    : 128 params (gate1 20 -> 4 global angles; clean ablation, 16 fewer)
  fixed_matched  : 144 params (adds a bias-free classical Linear(4,4) on the node features, h <- h + tanh(W h), 16 params,
                   so the count matches exactly; rules out "the gap is just fewer parameters")

Identical to Cell 3 otherwise: closed-form circuit (verified vs PennyLane), architecture, init key layout, seed, batch
order, SmoothL1 (beta 1) on log1p(pT) with standardized output, Adam 1e-3, wd 3e-5 (none on circuit angles incl. the
fixed angle logits), clip 1.0, batch 64, 50 epochs, best-val checkpoint, gamma escape, metrics, per-pT-class report.

EXTRA (no retraining): the trained ADAPTIVE model is evaluated with its angles frozen to their average value
  (data dependence removed at test time) -> shows how much the trained model actually uses the adaptivity.

RUN TIME (CPU): ~4 min per variant per seed -> 3 variants x 1 seed ~ 12 min; x 5 seeds ~ 60 min.
================================================================================
"""
VARIANTS_TO_RUN = ["adaptive", "fixed_plain", "fixed_matched"]
SEEDS = [SEED]                  # e.g. [42, 1, 60, 100, 120] -> paired comparison over seeds
VERIFY_WITH_PENNYLANE = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_adaptive_vs_fixed_rzz.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
QUANTUM_KEYS = ("qw", "fixed")                 # circuit angles: no weight decay
EXPECTED = {"adaptive": 144, "fixed_plain": 128, "fixed_matched": 144}
for v in VARIANTS_TO_RUN:
    assert v in EXPECTED, f"unknown variant {v}"

print("=" * 110)
print(f"ADAPTIVE vs FIXED RZZ (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM RING CIRCUIT  (== Cell 3's circuit_core; a is (N,4) per-event or broadcast global angles)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. VERIFICATION vs PENNYLANE (float64): per-event angles AND global angles, reference incl. the old 2nd RZZ layer
# ==============================================================================
if VERIFY_WITH_PENNYLANE:
    try:
        import pennylane as qml
        print("=" * 110 + "\nVERIFYING closed-form ring circuit vs PennyLane default.qubit (float64; reference includes a 2nd RZZ layer)\n" + "=" * 110)
        dev = qml.device("default.qubit", wires=NQ)

        @qml.qnode(dev)
        def _ref(th, W, a1, a2):
            for k in range(NQ): qml.RY(th[:, k], wires=k)
            for k in range(NQ): qml.RZ(W[0, k, 1], wires=k); qml.RY(W[0, k, 0], wires=k)
            for i, (u, v) in enumerate(RING_PAIRS): qml.IsingZZ(a1[..., i], wires=[u, v])
            for k in range(NQ): qml.RY(th[:, k], wires=k)
            for k in range(NQ): qml.RZ(W[1, k, 1], wires=k); qml.RY(W[1, k, 0], wires=k)
            for i, (u, v) in enumerate(RING_PAIRS): qml.IsingZZ(a2[..., i], wires=[u, v])
            return [qml.expval(qml.PauliZ(k)) for k in range(NQ)]

        rs = np.random.default_rng(9)
        th = rs.normal(size=(31, NQ)) * 1.5; W = rs.normal(size=(2, NQ, 2)) * 2
        for lab, a1, a2 in [("adaptive (per-event angles)", rs.uniform(0, math.pi, (31, 4)), rs.uniform(0, math.pi, (31, 4))),
                            ("fixed (global angles)", rs.uniform(0, math.pi, 4), rs.uniform(0, math.pi, 4))]:
            o_ref = np.stack([np.asarray(o) for o in _ref(th, W, a1, a2)], -1)
            a_b = np.broadcast_to(a1, (31, 4))
            o_cf = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), jnp.asarray(W[0], jnp.float32),
                                           jnp.asarray(W[1], jnp.float32), jnp.asarray(a_b, jnp.float32)))
            d = np.abs(o_ref - o_cf).max()
            print(f"  {lab:<30} max|PennyLane - closed form| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
            assert d < 5e-5, f"closed form disagrees with PennyLane ({lab})"
        print("  -> PASS\n")
    except ImportError:
        print("PennyLane not installed -> verification skipped (run Cell 0).")

# ==============================================================================
# 4. MODEL: PI-QGAT with adaptive or fixed RZZ angles  (adaptive == Cell 3 model, same init)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out, bias=True):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    p = {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b)}
    if bias:
        p["b"] = jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)
    return p

def init_params(variant, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> shared layers start identical
    P = {"node": _init_lin(ks[0], N_FEATS, NQ), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], NQ, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
         "pair": _init_lin(ks[1], 2 * NQ, NQ),
         "qw": (jax.random.uniform(ks[5], (2, NQ, 2), jnp.float32) * 2 - 1) * math.pi}
    if variant == "adaptive":
        P["gate1"] = _init_lin(ks[4], NQ, len(RING_PAIRS))                    # 20 params, data-dependent angles
    else:
        P["fixed"] = jnp.zeros((len(RING_PAIRS),), jnp.float32)              # 4 global angle logits -> pi/2 at start
        if variant == "fixed_matched":
            P["extra"] = _init_lin(ks[7], NQ, NQ, bias=False)                # 16 params, parameter matching only
    return P

def _lin(p, x):
    return x @ p["W"].T + p["b"] if "b" in p else x @ p["W"].T

def rzz_angles(P, tf, variant):
    if variant == "adaptive":
        return jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi                 # (N, 4) per edge, per event
    return jnp.broadcast_to(jax.nn.sigmoid(P["fixed"]) * math.pi, (tf.shape[0], len(RING_PAIRS)))   # same for all

def forward_parts(P, x, phi, ba, variant, a_const=None):
    """a_const: optional (4,) angles that REPLACE the model's angles (used for the frozen-angle test)."""
    B = x.shape[0]
    h = _lin(P["node"], x)
    if "extra" in P:
        h = h + jnp.tanh(_lin(P["extra"], h))
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, NQ)
    a = rzz_angles(P, tf, variant) if a_const is None else jnp.broadcast_to(a_const, (tf.shape[0], len(RING_PAIRS)))
    z = circuit_core(tf, P["qw"][0], P["qw"][1], a).reshape(B, N_EDGES, NQ)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, N_IN, NQ)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z, a

def forward(P, x, phi, ba, variant):
    return forward_parts(P, x, phi, ba, variant)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, variant):
        self.variant = variant
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, variant), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on circuit angles)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, variant))
        self.pred_const = jax.jit(lambda w, x, phi, ba, ac: forward_parts(unravel(w), x, phi, ba, variant, ac)[0])
        self.parts = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, variant)[1:])

def count_rows(variant):
    P = init_params(variant, 0)
    nl = lambda p: int(sum(a.size for a in jax.tree_util.tree_leaves(p)))
    names = {"node": "node_proj 7->4", "phys": "physics_prior 2->1", "head1": "head 4->6", "head2": "head 6->1",
             "pair": "pair_proj 8->4", "gate1": "entangle_gate1 4->4 (adaptive RZZ angles)", "qw": "circuit angles (2x4x2)",
             "fixed": "fixed_angle_logits (4 GLOBAL RZZ angles)", "extra": "extra Linear 4->4, no bias (param matching)"}
    rows = [(names[k], nl(v)) for k, v in P.items()]
    return rows, nl(P)

# ==============================================================================
# 5. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(fn, w, X, P_, B_, *extra, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([fn(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs], *extra) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(C.variant, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C.pred, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.variant} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C.pred, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    z, a = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])
    z, a = np.asarray(z, np.float64).reshape(-1, NQ), np.asarray(a, np.float64)        # a: (edges, 4)
    full = lambda: predict_log(C.pred, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    diag = dict(z_std=float(z.std(0).mean()), us_per_row=us,
                angle_mean=a.mean(0).tolist(),                       # per ring pair
                angle_std_across_events=a.std(0).tolist(),           # 0 for fixed; >0 = adaptive angles really vary
                ent_strength=float(np.abs(np.sin(a)).mean()))        # 0 -> entanglement switched off
    return m, bins, diag

# ==============================================================================
# 7. RUN  (results accumulate in RZZ_RESULTS across re-runs of this cell)
# ==============================================================================
if "RZZ_RESULTS" not in globals():
    RZZ_RESULTS = {}
WEIGHTS = {}

for variant in VARIANTS_TO_RUN:
    C = Compiled(variant)
    rows, total = count_rows(variant)
    assert total == EXPECTED[variant], f"{variant}: {total} params, expected {EXPECTED[variant]}"
    print("\n" + "#" * 110 + f"\n# VARIANT: {variant}  |  {total} params  (adaptive = 144, diff {total - 144:+d})\n" + "#" * 110)
    for nm, n in rows:
        print(f"   {nm:<46}{n:>6}")
    print(f"   {'TOTAL':<46}{total:>6}")
    if variant != "adaptive":
        print(f"   initial global RZZ angles (rad): {np.round(np.full(4, math.pi / 2), 4)}  (sigmoid(0) * pi)")
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        WEIGHTS[(variant, seed)] = (C, tr["w"])
        print(f"   >> {variant} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        print(f"\n   [{variant} | seed {seed}] PER pT CLASS (test)")
        print(f"   {'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>10}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
        for b in bins:
            print(f"   {_label(b['lo'], b['hi']):>16}{b['n']:>9}{b['frac']:>6.1f}%{b['mae_log']:>9.4f}{b['mae_pt']:>10.3f}"
                  f"{b['mse_pt']:>13.2f}{b['rmse_pt']:>10.3f}{b['med_rel']:>10.3f}{b['bias']:>8.3f}{b['sse_share']:>6.1f}%")
        if variant != "adaptive":
            fin = np.asarray(jax.nn.sigmoid(C.unravel(tr["w"])["fixed"]) * math.pi)
            print("\n   Learned GLOBAL RZZ angles (rad, range [0, pi]): "
                  + "  ".join(f"{RING_PAIRS[i]}={fin[i]:.4f}" for i in range(4))
                  + f"\n   (near 0 or pi -> that RZZ is effectively OFF / a pure phase; near pi/2 = {math.pi / 2:.4f} -> still at init)")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_rzz_{variant}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        RZZ_RESULTS[(variant, seed)] = dict(variant=variant, seed=seed, params=total, best_ep=tr["best_ep"],
                                            best_val=tr["best_val"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                                            test=m, bins=bins, hist=tr["hist"], diag=diag)

# ==============================================================================
# 8. FINAL REPORT
# ==============================================================================
ORDER = [v for v in ("adaptive", "fixed_plain", "fixed_matched") if any(k[0] == v for k in RZZ_RESULTS)]
R = {v: [RZZ_RESULTS[k] for k in sorted(RZZ_RESULTS) if k[0] == v] for v in ORDER}

print("\n" + "=" * 150 + "\nSUMMARY (test set, mean +- std over seeds)\n" + "=" * 150)
print(f"{'variant':<24}{'params':>7}{'seeds':>7}{'best val loss':>15}{'R2_log':>18}{'MAE_log':>18}{'MSE_log':>10}{'RMSE_log':>10}"
      f"{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}{'escapes':>9}")
for v in ORDER:
    rs = R[v]; g = lambda k: [r["test"][k] for r in rs]
    tag = v + (" (Cell 3 model)" if v == "adaptive" else "")
    print(f"{tag:<24}{rs[0]['params']:>7}{len(rs):>7}{np.mean([r['best_val'] for r in rs]):>15.4f}{ms(g('r2_log')):>18}"
          f"{ms(g('mae_log')):>18}{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}{ms(g('mae_pt'), '{:.3f}'):>17}"
          f"{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}{np.mean([r['escapes'] for r in rs]):>9.1f}")

# ---- head-to-head: fixed minus adaptive (paired by seed) ----
H2H = {}
if "adaptive" in R:
    ad = {r["seed"]: r for r in R["adaptive"]}
    for v in ORDER:
        if v == "adaptive":
            continue
        common = [r["seed"] for r in R[v] if r["seed"] in ad]
        if not common:
            continue
        fx = {r["seed"]: r for r in R[v]}
        print("\n" + "=" * 90 + f"\nHEAD-TO-HEAD: {v} minus adaptive  (paired by seed, {len(common)} seed(s))\n" + "=" * 90)
        H2H[v] = {}
        for lab, key, sign in [("val loss", None, "positive"), ("R2_log", "r2_log", "negative"), ("MAE_log", "mae_log", "positive"),
                               ("RMSE_log", "rmse_log", "positive"), ("MAE (GeV)", "mae_pt", "positive"), ("RMSE (GeV)", "rmse_pt", "positive")]:
            d = [fx[s]["best_val"] - ad[s]["best_val"] for s in common] if key is None else \
                [fx[s]["test"][key] - ad[s]["test"][key] for s in common]
            helps = sum((x > 0) if sign == "positive" else (x < 0) for x in d)
            H2H[v][lab] = d
            print(f"  d {lab:<11}: {ms(d, '{:+.4f}'):>20}   ({sign} = adaptive RZZ helps; adaptive better in {helps}/{len(d)} seeds)")
        print(f"\n{'pT class (GeV)':>16}{'MAE adapt':>11}{'MAE fixed':>11}{'d MAE':>10}{'med|rel| ad':>13}{'med|rel| fx':>13}{'d med|rel|':>12}")
        for b0 in ad[common[0]]["bins"]:
            pick = lambda rs_, k: np.mean([[b for b in rs_[s]["bins"] if b["lo"] == b0["lo"]][0][k] for s in common])
            qa, qf, ra, rf = pick(ad, "mae_pt"), pick(fx, "mae_pt"), pick(ad, "med_rel"), pick(fx, "med_rel")
            print(f"{_label(b0['lo'], b0['hi']):>16}{qa:>11.3f}{qf:>11.3f}{qf - qa:>+10.3f}{ra:>13.3f}{rf:>13.3f}{rf - ra:>+12.3f}")
else:
    print("\n(adaptive not run -> no head-to-head. Add 'adaptive' to VARIANTS_TO_RUN; it is the Cell 3 model with the same init.)")

# ---- convergence + time ----
def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<15}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for v in ORDER:
    rs = R[v]
    print(f"{v:<15}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}")

# ---- RZZ angle diagnostics ----
print(f"\nRZZ ANGLE DIAGNOSTICS (first {DIAG_ROWS} test events, seed-mean; radians, range 0..pi):")
print(f"{'variant':<15}" + "".join(f"{'mean ' + str(p):>14}" for p in RING_PAIRS) + "".join(f"{'std ' + str(p):>13}" for p in RING_PAIRS)
      + f"{'mean|sin a|':>13}{'std <Z>':>9}")
for v in ORDER:
    rs = R[v]
    mean_a = np.mean([r["diag"]["angle_mean"] for r in rs], 0); std_a = np.mean([r["diag"]["angle_std_across_events"] for r in rs], 0)
    print(f"{v:<15}" + "".join(f"{x:>14.3f}" for x in mean_a) + "".join(f"{x:>13.3f}" for x in std_a)
          + f"{np.mean([r['diag']['ent_strength'] for r in rs]):>13.3f}{np.mean([r['diag']['z_std'] for r in rs]):>9.4f}")
print("(std across events = how much the angles really change from event to event: 0 for fixed; small for adaptive ->\n"
      " the adaptive gate behaves almost like a fixed pattern)")

# ---- test-time check: trained ADAPTIVE model with its angles frozen to their average ----
FROZEN = {}
if "adaptive" in R:
    print("\n" + "=" * 110 + "\nTRAINED ADAPTIVE MODEL WITH ANGLES FROZEN TO THEIR TRAIN-SET AVERAGE (no retraining; test set)\n" + "=" * 110)
    print(f"{'seed':<6}{'R2_log adaptive':>17}{'R2_log frozen':>15}{'dR2_log':>10}{'MAE_pT adaptive':>17}{'MAE_pT frozen':>15}{'dMAE_pT':>10}")
    for r in R["adaptive"]:
        s = r["seed"]
        if ("adaptive", s) not in WEIGHTS:
            continue                                                     # weights from an earlier run of the cell
        C, w = WEIGHTS[("adaptive", s)]
        _, a_tr = C.parts(w, Xtr[:DIAG_ROWS], ptr[:DIAG_ROWS], btr[:DIAG_ROWS])
        a_mean = jnp.asarray(np.asarray(a_tr).mean(0), jnp.float32)
        mf, _ = split_metrics(predict_log(C.pred_const, w, Xte, pte, bte, a_mean), y_test, pT_test64)
        FROZEN[s] = dict(angles=np.asarray(a_mean).tolist(), test=mf)
        print(f"{s:<6}{r['test']['r2_log']:>17.4f}{mf['r2_log']:>15.4f}{mf['r2_log'] - r['test']['r2_log']:>+10.4f}"
              f"{r['test']['mae_pt']:>17.3f}{mf['mae_pt']:>15.3f}{mf['mae_pt'] - r['test']['mae_pt']:>+10.3f}")
    print("(big drop -> the trained model really uses event-dependent angles; ~0 -> a fixed angle pattern is enough)")

# ==============================================================================
# 9. SAVE
# ==============================================================================
out = dict(results={f"{v}_seed{s}": x for (v, s), x in RZZ_RESULTS.items()}, head_to_head=H2H,
           frozen_adaptive={str(s): x for s, x in FROZEN.items()}, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

print("\nReading guide:")
print("  fixed clearly worse (both plain and matched) -> data-dependent entanglement IS responsible for the gain")
print("  fixed about equal                           -> adaptive RZZ adds little; a fixed entangling pattern suffices")
print("  fixed_plain worse but fixed_matched equal   -> the gap was the 16 missing parameters, not adaptivity")
print("  single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01, so smaller gaps need SEEDS = [42, 1, 60, 100, 120]")

ADAPTIVE vs FIXED RZZ (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['adaptive', 'fixed_plain', 'fixed_matched']
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001
VERIFYING closed-form ring circuit vs PennyLane default.qubit (float64; reference includes a 2nd RZZ layer)
  adaptive (per-event angles)    max|PennyLane - closed form| = 2.08e-07   (float32 vs float64; must be < 5e-5)
  fixed (global angles)          max|PennyLane - closed form| = 1.67e-07   (float32 vs float64; must be < 5e-5)
  -> PASS


##############################################################################################################
# VARIANT: adaptive  |  144 params  (adaptive = 144, diff +0)
##############################################################################################################
   node_proj 7->4                                    32
   physics_prior 2->1                                 3
   head 4->6                                         30
   head 6

##Quantum Attention vs Quantum Message

In [ ]:
"""
================================================================================
EXPERIMENT: QUANTUM ATTENTION vs QUANTUM MESSAGE   (current PI-QGAT, 144 params)               [FAST JAX + JIT]
CELL -- QA_vs_QM (where is the quantum component useful?)
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

In PI-QGAT the circuit output z_ij (4 values per edge, <Z_k> in [-1, 1]) is used twice:
    attention : s_ij = mean_k z_ij,k + lambda * P_ij        (which neighbour to listen to)
    message   : m_i  = sum_j alpha_ij (h_j + z_ij)          (what is passed along)

  A_attn_only : circuit output used ONLY for the attention score; the message uses a classical pass-through
  B_msg_only  : circuit output used ONLY for the message;          the attention uses a classical pass-through
  C_both      : circuit output used for both  (= the Cell 3 PI-QGAT model, same init)
  D_neither   : classical pass-through in both places (no quantum at all; floor)  [optional]

Classical pass-through = tanh(u), u = pair_proj([h_i, h_j]) -- the same pair feature that feeds the circuit
(theta = pi * tanh(u)). Same shape (B, 12, 4), same range [-1, 1], ZERO extra parameters -> every variant has
exactly 144 params (in D_neither the 36 circuit/gate params receive no gradient).
CLASSICAL_BYPASS = "zero" instead removes the path completely (z replaced by 0).

Identical to Cell 3 otherwise: closed-form circuit, architecture, init key layout (same init + same batch order per
variant), SmoothL1 (beta 1) on log1p(pT) with standardized output, Adam 1e-3, wd 3e-5 (none on circuit angles),
clip 1.0, batch 64, 50 epochs, best-val checkpoint, gamma escape, metrics, per-pT-class report.

RUN TIME (CPU): ~4 min per variant per seed -> 3 variants x 1 seed ~ 12 min; x 5 seeds ~ 60 min.
Session may disconnect? Run one variant at a time; results accumulate in QA_QM_RESULTS across re-runs.
================================================================================
"""
VARIANTS_TO_RUN = ["A_attn_only", "B_msg_only", "C_both"]   # add "D_neither" for the no-quantum floor
SEEDS = [SEED]                  # e.g. [42, 1, 60, 100, 120] -> paired comparison over seeds
CLASSICAL_BYPASS = "pair_tanh"  # "pair_tanh" (default) or "zero" (path simply removed)
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_quantum_attention_vs_message.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
QUANTUM_KEYS = ("qw",)                         # circuit angles: no weight decay (as Cell 3)
VALID_MODES = ("A_attn_only", "B_msg_only", "C_both", "D_neither")
for v in VARIANTS_TO_RUN:
    assert v in VALID_MODES, f"unknown variant {v}"
assert CLASSICAL_BYPASS in ("pair_tanh", "zero")

print("=" * 110)
print(f"QUANTUM ATTENTION vs QUANTUM MESSAGE (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | variants {VARIANTS_TO_RUN} | bypass {CLASSICAL_BYPASS}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM RING CIRCUIT  (identical to Cell 3's circuit_core, verified there against PennyLane)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL: PI-QGAT with a switch for where the circuit output is routed  (C_both == Cell 3 model, same init)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))
ROUTE = {"A_attn_only": ("q", "c"), "B_msg_only": ("c", "q"), "C_both": ("q", "q"), "D_neither": ("c", "c")}   # (attention, message)

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> every variant starts identical
    return {"node": _init_lin(ks[0], N_FEATS, NQ), "phys": _init_lin(ks[2], 2, 1),
            "head1": _init_lin(ks[3], NQ, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
            "pair": _init_lin(ks[1], 2 * NQ, NQ), "gate1": _init_lin(ks[4], NQ, len(RING_PAIRS)),
            "qw": (jax.random.uniform(ks[5], (2, NQ, 2), jnp.float32) * 2 - 1) * math.pi}

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward_parts(P, x, phi, ba, mode):
    B = x.shape[0]
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))                    # (B, 12, 4)
    tf = (math.pi * jnp.tanh(u)).reshape(-1, NQ)
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z_q = circuit_core(tf, P["qw"][0], P["qw"][1], a1).reshape(B, N_EDGES, NQ)   # quantum path
    z_c = jnp.tanh(u) if CLASSICAL_BYPASS == "pair_tanh" else jnp.zeros_like(z_q)  # classical pass-through, 0 params
    route_att, route_msg = ROUTE[mode]
    z_att = z_q if route_att == "q" else z_c
    z_msg = z_q if route_msg == "q" else z_c
    s = z_att.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + z_msg).reshape(B, N_NODES, N_IN, NQ)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z_q, z_c, alpha

def forward(P, x, phi, ba, mode):
    return forward_parts(P, x, phi, ba, mode)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, mode):
        self.mode = mode
        P0 = init_params(0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, mode), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on qw)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, mode))
        self.parts = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, mode)[1:])

# ==============================================================================
# 5. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(fn, w, X, P_, B_, *extra, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([fn(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs], *extra) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C.pred, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.mode} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C.pred, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    zq, zc, alpha = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])
    zq, zc, a = np.asarray(zq, np.float64).reshape(-1, NQ), np.asarray(zc, np.float64).reshape(-1, NQ), np.asarray(alpha, np.float64)
    ent = float((-(a * np.log(np.clip(a, 1e-12, 1))).sum(-1)).mean() / math.log(N_IN))     # 1 = uniform attention
    corr = float(np.mean([np.corrcoef(zq[:, k], zc[:, k])[0, 1] for k in range(NQ)])) if zc.std() > 0 else float("nan")
    full = lambda: predict_log(C.pred, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    return m, bins, dict(zq_std=float(zq.std(0).mean()), zc_std=float(zc.std(0).mean()), corr_q_c=corr,
                         attn_entropy=ent, us_per_row=us)

# ==============================================================================
# 7. RUN  (results accumulate in QA_QM_RESULTS across re-runs of this cell)
# ==============================================================================
if "QA_QM_RESULTS" not in globals():
    QA_QM_RESULTS = {}

N_PAR = int(ravel_pytree(init_params(0))[0].shape[0])
assert N_PAR == 144, f"model has {N_PAR} params, expected 144 (Cell 3)"
for mode in VARIANTS_TO_RUN:
    C = Compiled(mode)
    ra, rm = ROUTE[mode]
    print("\n" + "#" * 110 + f"\n# VARIANT: {mode}  |  attention <- {'QUANTUM' if ra == 'q' else 'classical'},  "
          f"message <- {'QUANTUM' if rm == 'q' else 'classical'}  |  {N_PAR} params"
          + ("  (36 circuit/gate params unused)" if mode == "D_neither" else "") + "\n" + "#" * 110)
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        print(f"   >> {mode} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        print(f"\n   [{mode} | seed {seed}] PER pT CLASS (test)")
        print(f"   {'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>10}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
        for b in bins:
            print(f"   {_label(b['lo'], b['hi']):>16}{b['n']:>9}{b['frac']:>6.1f}%{b['mae_log']:>9.4f}{b['mae_pt']:>10.3f}"
                  f"{b['mse_pt']:>13.2f}{b['rmse_pt']:>10.3f}{b['med_rel']:>10.3f}{b['bias']:>8.3f}{b['sse_share']:>6.1f}%")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_route_{mode}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        QA_QM_RESULTS[(mode, seed)] = dict(mode=mode, seed=seed, params=N_PAR, best_ep=tr["best_ep"], best_val=tr["best_val"],
                                           train_total_s=tr["train_total"], escapes=tr["escapes"], test=m, bins=bins,
                                           hist=tr["hist"], diag=diag, bypass=CLASSICAL_BYPASS)

# ==============================================================================
# 8. FINAL REPORT
# ==============================================================================
ORDER = [v for v in ("C_both", "A_attn_only", "B_msg_only", "D_neither") if any(k[0] == v for k in QA_QM_RESULTS)]
R = {v: [QA_QM_RESULTS[k] for k in sorted(QA_QM_RESULTS) if k[0] == v] for v in ORDER}

print("\n" + "=" * 150 + "\nSUMMARY (test set, mean +- std over seeds)\n" + "=" * 150)
print(f"{'variant':<24}{'params':>7}{'seeds':>7}{'best val loss':>15}{'R2_log':>18}{'MAE_log':>18}{'MSE_log':>10}{'RMSE_log':>10}"
      f"{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}{'escapes':>9}")
for v in ORDER:
    rs = R[v]; g = lambda k: [r["test"][k] for r in rs]
    tag = v + (" (Cell 3 model)" if v == "C_both" else "")
    print(f"{tag:<24}{rs[0]['params']:>7}{len(rs):>7}{np.mean([r['best_val'] for r in rs]):>15.4f}{ms(g('r2_log')):>18}"
          f"{ms(g('mae_log')):>18}{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}{ms(g('mae_pt'), '{:.3f}'):>17}"
          f"{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}{np.mean([r['escapes'] for r in rs]):>9.1f}")

# ---- deltas vs C_both (paired by seed): positive error delta = removing quantum from that path hurts ----
DELTAS = {}
if "C_both" in R:
    ref = {r["seed"]: r for r in R["C_both"]}
    print("\n" + "=" * 112 + "\nDELTAS vs C_both  [variant minus C_both, paired by seed; R2: - = quantum in the removed path helped; errors: +]\n" + "=" * 112)
    print(f"{'variant':<14}{'seeds':>6}{'d R2_log':>18}{'d MAE_log':>18}{'d RMSE_log':>12}{'d MAE_pT':>16}{'d val loss':>12}{'C_both better':>15}")
    for v in ORDER:
        if v == "C_both":
            continue
        common = [r["seed"] for r in R[v] if r["seed"] in ref]
        if not common:
            print(f"{v:<14} no seed in common with C_both"); continue
        x = {r["seed"]: r for r in R[v]}
        d = lambda k: [x[s]["test"][k] - ref[s]["test"][k] for s in common]
        wins = sum(ref[s]["test"]["r2_log"] > x[s]["test"]["r2_log"] for s in common)
        DELTAS[v] = {k: d(k) for k in ("r2_log", "mae_log", "rmse_log", "mae_pt")}
        print(f"{v:<14}{len(common):>6}{ms(d('r2_log'), '{:+.4f}'):>18}{ms(d('mae_log'), '{:+.4f}'):>18}{np.mean(d('rmse_log')):>+12.4f}"
              f"{ms(d('mae_pt'), '{:+.3f}'):>16}{np.mean([x[s]['best_val'] - ref[s]['best_val'] for s in common]):>+12.4f}"
              f"{f'{wins}/{len(common)}':>15}")
else:
    print("\n(C_both not run -> no deltas. Add 'C_both' to VARIANTS_TO_RUN; it is the Cell 3 model with the same init.)")

# ---- per pT class ----
for key, title, fmt in (("med_rel", "med|rel| per pT class", "{:>15.3f}"), ("mae_pt", "MAE (GeV) per pT class", "{:>15.3f}"),
                        ("mae_log", "MAE_log per pT class", "{:>15.4f}")):
    print(f"\n{title} (test, seed-mean)")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{v:>15}" for v in ORDER))
    for b0 in R[ORDER[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r["bins"] if b["lo"] == b0["lo"]][0][key] for r in R[v]])) for v in ORDER))

# ---- convergence + time ----
def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<14}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for v in ORDER:
    rs = R[v]
    print(f"{v:<14}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}")

# ---- diagnostics ----
print(f"\nDIAGNOSTICS (first {DIAG_ROWS} test events, seed-mean):")
print(f"{'variant':<14}{'std quantum z':>15}{'std classical z':>17}{'corr(q, c)':>12}{'attn entropy':>14}")
for v in ORDER:
    g = lambda k: np.nanmean([r["diag"][k] for r in R[v]])
    print(f"{v:<14}{g('zq_std'):>15.4f}{g('zc_std'):>17.4f}{g('corr_q_c'):>12.3f}{g('attn_entropy'):>14.3f}")
print("(std quantum z ~ 0 -> the circuit output is nearly constant; attention entropy 1.0 = uniform over the 3 edges)")

# ==============================================================================
# 9. SAVE
# ==============================================================================
out = dict(results={f"{v}_seed{s}": x for (v, s), x in QA_QM_RESULTS.items()}, deltas_vs_C=DELTAS, seeds=SEEDS, bypass=CLASSICAL_BYPASS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

print("\nReading guide:")
print("  A close to C, B clearly worse  -> quantum helps mainly in ATTENTION (edge selection)")
print("  B close to C, A clearly worse  -> quantum helps mainly in MESSAGE (feature transform)")
print("  both A and B worse than C      -> quantum helps in BOTH, and the two uses add up")
print("  A and B both about equal to C  -> where the quantum output goes matters little (compare with D_neither)")
print("  single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01, so smaller gaps need SEEDS = [42, 1, 60, 100, 120]")

QUANTUM ATTENTION vs QUANTUM MESSAGE (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['A_attn_only', 'B_msg_only', 'C_both'] | bypass pair_tanh
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001

##############################################################################################################
# VARIANT: A_attn_only  |  attention <- QUANTUM,  message <- classical  |  144 params
##############################################################################################################
[A_attn_only | seed 42] ep  1/50  train 0.2369  val loss 0.1805  R2_log 0.4074  MAE_log 0.4061  MAE_pT 15.398 GeV   16.6s *best   (epoch 1 includes XLA compile)
[A_attn_only | seed 42] ep  2/50  train 0.1741  val loss 0.1707  R2_log 0.4674  MAE_log 0.4099  MAE_pT 15.403 GeV    7.5s *best
[A_attn_only | seed 42] ep  3/50  train 0.1685  val loss 0.1671  R2_log 0.4700  MAE_log 0.3925  MAE_pT 15.254 GeV    6.8s *best
[A_attn_only | seed 42] ep  4/50  train 0.1662  va

##Station Ablation

In [ ]:
"""
================================================================================
EXPERIMENT: STATION ABLATION -- which muon station matters most?   (current PI-QGAT, 144 params)   [FAST JAX + JIT]
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

PI-QGAT sees one event as a graph of 4 station nodes (Cell 1: STATIONS = [0, 2, 3, 4]), each with 7 features
(Phi, Theta, BendingAngle, TimeInfo, RingNumber, Front, Mask), fully connected (12 directed edges).

A) RETRAINING ABLATION (main result): drop one station completely and retrain from scratch
     all        : 4 stations, 12 edges  (= the Cell 3 model)
     drop_stK   : station K removed -> 3-node fully connected graph, 6 edges, softmax over 2 incoming edges.
                  The station's 7 features AND its edges (|d bending|, |d phi| to the others) are gone.
   Every weight in PI-QGAT is shared across nodes/edges, so EVERY variant has exactly the same 144 params,
   the same init and the same batch order -> only the input changes.
   optional RUN_STATION_PAIRS = True: keep only 2 stations (6 pairs, 2-node graphs) -> which PAIR carries the most.
B) PERMUTATION IMPORTANCE (trained 'all' model, no retraining): one station's features + phi + bending are
   shuffled across test events (3 repeats) -> drop in R2 / MAE, overall and per pT class.
C) ATTENTION MAP (trained 'all' model): average attention each receiving station gives to each sending station.

Recipe identical to Cell 3: closed-form circuit, SmoothL1 (beta 1) on log1p(pT) with standardized output, Adam 1e-3,
wd 3e-5 (none on circuit angles), clip 1.0, batch 64, 50 epochs, best-val checkpoint, gamma escape, same metrics.

RUN TIME (CPU): 3-node graphs are faster (6 edges). all + 4 drops ~ 15-18 min for one seed; pairs add ~6 x 2 min.
Results accumulate in STATION_RESULTS across re-runs (run variants one at a time if the session may disconnect).
================================================================================
"""
SEEDS = [SEED]                  # e.g. [42, 1, 60, 100, 120] -> paired comparison over seeds
RUN_STATION_PAIRS = False       # True -> also train the 6 two-station models
RUN_PERMUTATION = True
RUN_ATTENTION_MAP = True
PERM_REPEATS = 3
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_station_ablation.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
QUANTUM_KEYS = ("qw",)                         # circuit angles: no weight decay (as Cell 3)
STATION_IDS = list(globals().get("STATIONS", [0, 2, 3, 4]))     # Cell 1 order of the 4 nodes
assert len(STATION_IDS) == N_NODES

print("=" * 110)
print(f"STATION ABLATION (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | stations {STATION_IDS}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM RING CIRCUIT  (identical to Cell 3's circuit_core, verified there against PennyLane)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. MODEL: PI-QGAT on a SUBSET of stations  (subset = all 4 -> exactly the Cell 3 model)
# ==============================================================================
FULL_RECV, FULL_SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def subgraph(nodes):
    """keep Cell 1's receiver-major edge order, restricted to `nodes`; returns (node idx, recv, send) in local indices."""
    nodes = tuple(sorted(nodes)); loc = {n: i for i, n in enumerate(nodes)}
    e = [(loc[r], loc[s]) for r, s in zip(FULL_RECV, FULL_SEND) if r in loc and s in loc]
    recv, send = np.array([a for a, _ in e]), np.array([b for _, b in e])
    n = len(nodes)
    assert len(e) == n * (n - 1) and np.all(recv.reshape(n, n - 1) == recv.reshape(n, n - 1)[:, :1]), "receiver-major order lost"
    return np.array(nodes), recv, send

def variant_nodes():
    V = {"all": tuple(range(N_NODES))}
    for k, st in enumerate(STATION_IDS):
        V[f"drop_st{st}"] = tuple(i for i in range(N_NODES) if i != k)
    if RUN_STATION_PAIRS:
        for a in range(N_NODES):
            for b in range(a + 1, N_NODES):
                V[f"only_st{STATION_IDS[a]}+st{STATION_IDS[b]}"] = (a, b)
    return V
VARIANTS = variant_nodes()

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> every variant starts identical
    return {"node": _init_lin(ks[0], N_FEATS, NQ), "phys": _init_lin(ks[2], 2, 1),
            "head1": _init_lin(ks[3], NQ, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
            "pair": _init_lin(ks[1], 2 * NQ, NQ), "gate1": _init_lin(ks[4], NQ, len(RING_PAIRS)),
            "qw": (jax.random.uniform(ks[5], (2, NQ, 2), jnp.float32) * 2 - 1) * math.pi}

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward_parts(P, x, phi, ba, nodes):
    idx, RECV, SEND = subgraph(nodes)
    n = len(idx); n_in = n - 1; n_e = n * n_in
    x, phi, ba = x[:, idx], phi[:, idx], ba[:, idx]
    B = x.shape[0]
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, NQ)
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z = circuit_core(tf, P["qw"][0], P["qw"][1], a1).reshape(B, n_e, NQ)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, n, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, n, n_in, NQ)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), alpha

def forward(P, x, phi, ba, nodes):
    return forward_parts(P, x, phi, ba, nodes)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, name):
        self.name, self.nodes = name, VARIANTS[name]
        nodes = self.nodes
        P0 = init_params(0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, nodes), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on qw)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, nodes))
        self.attn = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, nodes)[1])

# ==============================================================================
# 5. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(fn, w, X, P_, B_, *extra, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([fn(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs], *extra) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C.pred, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.name} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C.pred, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    full = lambda: predict_log(C.pred, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    return m, bins, us

def vlabel(name):
    nodes = VARIANTS[name]
    return f"{len(nodes)} stations {[STATION_IDS[i] for i in nodes]}, {len(nodes) * (len(nodes) - 1)} edges"

# ==============================================================================
# 7. A) RETRAINING ABLATION  (results accumulate in STATION_RESULTS across re-runs of this cell)
# ==============================================================================
if "STATION_RESULTS" not in globals():
    STATION_RESULTS = {}
WEIGHTS = {}
N_PAR = int(ravel_pytree(init_params(0))[0].shape[0])
assert N_PAR == 144, f"model has {N_PAR} params, expected 144 (Cell 3)"
VARIANTS_TO_RUN = list(VARIANTS)       # all, drop_st0, drop_st2, drop_st3, drop_st4 (+ pairs if enabled)

for name in VARIANTS_TO_RUN:
    C = Compiled(name)
    print("\n" + "#" * 110 + f"\n# VARIANT: {name}  ({vlabel(name)})  |  {N_PAR} params" + "\n" + "#" * 110)
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, us = evaluate(C, tr["w"])
        WEIGHTS[(name, seed)] = (C, tr["w"])
        print(f"   >> {name} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_station_{name}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        STATION_RESULTS[(name, seed)] = dict(name=name, seed=seed, stations=[STATION_IDS[i] for i in VARIANTS[name]],
                                             params=N_PAR, best_ep=tr["best_ep"], best_val=tr["best_val"],
                                             train_total_s=tr["train_total"], escapes=tr["escapes"], test=m, bins=bins,
                                             hist=tr["hist"], us_per_row=us)

ORDER = [v for v in VARIANTS if any(k[0] == v for k in STATION_RESULTS)]
R = {v: [STATION_RESULTS[k] for k in sorted(STATION_RESULTS) if k[0] == v] for v in ORDER}

print("\n" + "=" * 156 + "\nA) RETRAINING ABLATION -- SUMMARY (test set, mean +- std over seeds)\n" + "=" * 156)
print(f"{'variant':<20}{'stations kept':<17}{'edges':>6}{'seeds':>6}{'best val loss':>14}{'R2_log':>18}{'MAE_log':>18}{'MSE_log':>10}"
      f"{'RMSE_log':>10}{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}")
for v in ORDER:
    rs = R[v]; g = lambda k: [r["test"][k] for r in rs]; nn_ = len(VARIANTS[v])
    print(f"{v + (' (Cell 3)' if v == 'all' else ''):<20}{str(rs[0]['stations']):<17}{nn_ * (nn_ - 1):>6}{len(rs):>6}"
          f"{np.mean([r['best_val'] for r in rs]):>14.4f}{ms(g('r2_log')):>18}{ms(g('mae_log')):>18}{np.mean(g('mse_log')):>10.4f}"
          f"{np.mean(g('rmse_log')):>10.4f}{ms(g('mae_pt'), '{:.3f}'):>17}{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}"
          f"{np.mean(g('n_clamped')):>9.0f}")

RETRAIN_DROP = {}
if "all" in R:
    ref = {r["seed"]: r for r in R["all"]}
    print("\n" + "=" * 110 + "\nCOST OF REMOVING EACH STATION  [variant minus all, paired by seed; bigger loss = more important station]\n" + "=" * 110)
    print(f"{'variant':<20}{'seeds':>6}{'d R2_log':>18}{'d MAE_log':>18}{'d MAE_pT GeV':>18}{'d RMSE_pT':>12}{'all better':>12}")
    for v in ORDER:
        if v == "all":
            continue
        common = [r["seed"] for r in R[v] if r["seed"] in ref]
        if not common:
            continue
        x = {r["seed"]: r for r in R[v]}
        d = lambda k: [x[s]["test"][k] - ref[s]["test"][k] for s in common]
        wins = sum(ref[s]["test"]["r2_log"] > x[s]["test"]["r2_log"] for s in common)
        RETRAIN_DROP[v] = {k: d(k) for k in ("r2_log", "mae_log", "mae_pt", "rmse_pt")}
        print(f"{v:<20}{len(common):>6}{ms(d('r2_log'), '{:+.4f}'):>18}{ms(d('mae_log'), '{:+.4f}'):>18}{ms(d('mae_pt'), '{:+.3f}'):>18}"
              f"{np.mean(d('rmse_pt')):>+12.3f}{f'{wins}/{len(common)}':>12}")
else:
    print("\n('all' not run -> no deltas.)")

for key, title, fmt in (("mae_pt", "MAE (GeV) per pT class", "{:>14.3f}"), ("mae_log", "MAE_log per pT class", "{:>14.4f}"),
                        ("med_rel", "med|rel| per pT class", "{:>14.3f}")):
    print(f"\n{title} (test, seed-mean)")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{v[:13]:>14}" for v in ORDER))
    for b0 in R[ORDER[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r["bins"] if b["lo"] == b0["lo"]][0][key] for r in R[v]])) for v in ORDER))

def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<20}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for v in ORDER:
    rs = R[v]
    print(f"{v:<20}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['us_per_row'] for r in rs]):>9.2f}")

# ==============================================================================
# 8. B) PERMUTATION IMPORTANCE  (trained 'all' model; one station shuffled across test events)
# ==============================================================================
PERM = {}
fulls = [(s, WEIGHTS[("all", s)]) for s in SEEDS if ("all", s) in WEIGHTS]
if RUN_PERMUTATION and fulls:
    print("\n" + "=" * 110 + f"\nB) PERMUTATION IMPORTANCE (trained 'all' model, {PERM_REPEATS} shuffles, no retraining; test set)\n" + "=" * 110)
    bin_edges = PT_BIN_EDGES
    for s, (C, w) in fulls:
        base = predict_log(C.pred, w, Xte, pte, bte)
        mb, ptb = split_metrics(base, y_test, pT_test64)
        binb = bin_stats(pT_test64, ptb, base, y_test)
        PERM[s] = {}
        for k, st in enumerate(STATION_IDS):
            res = []
            for rep in range(PERM_REPEATS):
                pr = jnp.asarray(np.random.default_rng(1000 * rep + k).permutation(Xte.shape[0]))
                Xs = Xte.at[:, k].set(Xte[pr, k]); ps = pte.at[:, k].set(pte[pr, k]); bs_ = bte.at[:, k].set(bte[pr, k])
                pl = predict_log(C.pred, w, Xs, ps, bs_)
                mp, ptp = split_metrics(pl, y_test, pT_test64)
                res.append((mp, bin_stats(pT_test64, ptp, pl, y_test)))
            PERM[s][st] = dict(d_r2_log=float(np.mean([r[0]["r2_log"] - mb["r2_log"] for r in res])),
                               d_mae_log=float(np.mean([r[0]["mae_log"] - mb["mae_log"] for r in res])),
                               d_mae_pt=float(np.mean([r[0]["mae_pt"] - mb["mae_pt"] for r in res])),
                               d_bin_mae_pt=[float(np.mean([r[1][i]["mae_pt"] - binb[i]["mae_pt"] for r in res])) for i in range(len(binb))])
        PERM[s]["_bins"] = [(b["lo"], b["hi"]) for b in binb]
    print(f"{'station':<10}{'d R2_log':>18}{'d MAE_log':>18}{'d MAE_pT GeV':>18}")
    for st in STATION_IDS:
        g = lambda k: [PERM[s][st][k] for s, _ in fulls]
        print(f"{'st ' + str(st):<10}{ms(g('d_r2_log'), '{:+.4f}'):>18}{ms(g('d_mae_log'), '{:+.4f}'):>18}{ms(g('d_mae_pt'), '{:+.3f}'):>18}")
    print("\nPermutation: increase in MAE_pT (GeV) per pT class when the station is shuffled (seed-mean)")
    first = fulls[0][0]
    print(f"{'pT class (GeV)':>16}" + "".join(f"{'st ' + str(st):>12}" for st in STATION_IDS))
    for i, (lo, hi) in enumerate(PERM[first]["_bins"]):
        print(f"{_label(lo, hi):>16}" + "".join(f"{np.mean([PERM[s][st]['d_bin_mae_pt'][i] for s, _ in fulls]):>+12.3f}" for st in STATION_IDS))

# ==============================================================================
# 9. C) ATTENTION MAP  (trained 'all' model): receiver station -> sender station
# ==============================================================================
ATTN = {}
if RUN_ATTENTION_MAP and fulls:
    print("\n" + "=" * 110 + f"\nC) ATTENTION MAP (trained 'all' model, first {DIAG_ROWS} test events, seed-mean)\n" + "=" * 110)
    mats = []
    for s, (C, w) in fulls:
        a = np.asarray(C.attn(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS]), np.float64).mean(0)   # (4, 3)
        M = np.full((N_NODES, N_NODES), np.nan)
        for e, (rr, ss) in enumerate(zip(FULL_RECV, FULL_SEND)):
            M[rr, ss] = a[rr, e % (N_NODES - 1)]
        mats.append(M)
    M = np.nanmean(np.stack(mats), 0) if len(mats) > 1 else mats[0]
    ATTN = {"matrix": M.tolist(), "stations": STATION_IDS}
    print("rows = receiving station, columns = sending station; each row sums to 1 (uniform = 0.333)")
    print(f"{'':>12}" + "".join(f"{'from st ' + str(st):>13}" for st in STATION_IDS))
    for i, st in enumerate(STATION_IDS):
        print(f"{'to st ' + str(st):>12}" + "".join(f"{'-':>13}" if np.isnan(M[i, j]) else f"{M[i, j]:>13.3f}" for j in range(N_NODES)))
    recv_att = np.nanmean(M, 0)
    print(f"{'mean recv.':>12}" + "".join(f"{x:>13.3f}" for x in recv_att) + "   <- average attention a station RECEIVES as a sender")

# ==============================================================================
# 10. RANKING + SAVE
# ==============================================================================
print("\n" + "=" * 110 + "\nSTATION RANKING (1 = most important)\n" + "=" * 110)
rank_rows = []
for k, st in enumerate(STATION_IDS):
    v = f"drop_st{st}"
    rr = -np.mean(RETRAIN_DROP[v]["r2_log"]) if v in RETRAIN_DROP else float("nan")
    pp = -np.mean([PERM[s][st]["d_r2_log"] for s, _ in fulls]) if PERM else float("nan")
    at = float(np.nanmean(np.asarray(ATTN["matrix"]), 0)[k]) if ATTN else float("nan")
    rank_rows.append((st, rr, pp, at))
def _rank(vals):
    order = np.argsort([-x if np.isfinite(x) else np.inf for x in vals])
    rk = np.empty(len(vals), int); rk[order] = np.arange(1, len(vals) + 1); return rk
rk_r, rk_p = _rank([r[1] for r in rank_rows]), _rank([r[2] for r in rank_rows])
print(f"{'station':<10}{'R2 lost when removed (retrain)':>32}{'rank':>6}{'R2 lost when shuffled':>24}{'rank':>6}{'attention received':>20}")
for i, (st, rr, pp, at) in enumerate(rank_rows):
    print(f"{'st ' + str(st):<10}{rr:>32.4f}{rk_r[i]:>6}{pp:>24.4f}{rk_p[i]:>6}{at:>20.3f}")
print("(retrain = how much the model can NOT recover without the station -> the main answer;\n"
      " shuffled = how much the trained model RELIES on it; the two can differ when stations carry overlapping information)")

out = dict(results={f"{v}_seed{s}": x for (v, s), x in STATION_RESULTS.items()}, retrain_drop=RETRAIN_DROP,
           permutation={str(s): {str(k): v for k, v in d.items()} for s, d in PERM.items()}, attention=ATTN,
           stations=STATION_IDS, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")
print("Single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01; station gaps smaller than that need SEEDS = [42, 1, 60, 100, 120].")

STATION ABLATION (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | stations [0, 2, 3, 4]
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001

##############################################################################################################
# VARIANT: all  (4 stations [0, 2, 3, 4], 12 edges)  |  144 params
##############################################################################################################
[all | seed 42] ep  1/50  train 0.1842  val loss 0.1540  R2_log 0.5052  MAE_log 0.3613  MAE_pT 15.023 GeV    8.6s *best   (epoch 1 includes XLA compile)
[all | seed 42] ep  2/50  train 0.1359  val loss 0.1007  R2_log 0.7002  MAE_log 0.3057  MAE_pT 13.826 GeV    6.6s *best
[all | seed 42] ep  3/50  train 0.0806  val loss 0.0707  R2_log 0.7977  MAE_log 0.2497  MAE_pT 12.681 GeV    7.5s *best
[all | seed 42] ep  4/50  train 0.0675  val loss 0.0640  R2_log 0.8179  MAE_log 0.2359  MAE_pT 12.231 GeV    6.2s *best
[all | seed 42] ep  5/50  train 0.0646 

##Re-uploading ON vs OFF Ablation

In [ ]:
"""
================================================================================
EXPERIMENT: DATA RE-UPLOADING ON vs OFF   (current PI-QGAT, 144 params)                       [FAST JAX + JIT]
CELL -- N_no_reupload (PI-QGAT with the second data encoding removed)
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

PURPOSE: does data re-uploading genuinely add representational capacity, or is the gain just a deeper circuit?

THE CHANGE (everything else in the circuit untouched):
  reupload_ON  (Cell 3) : RY(theta) -> V1 -> RZZ(a) -> RY(theta) RE-ENCODE -> V2 -> <Z>
  reupload_OFF          : RY(theta) -> V1 -> RZZ(a) ->      [see mode]      -> V2 -> <Z>
      V1, V2 = [RZ(gamma) RY(omega)] trainable layers; a = pi*sigmoid(gate1(theta)) data-dependent ring RZZ angles.
  OFF modes (both are trained by default):
    reupload_OFF_drop           : second encoding simply removed (one RY layer shallower) -- the plain ablation
    reupload_OFF_depth_matched  : second encoding replaced by a FIXED, data-independent RY(DEPTH_MATCH_ANGLE) layer;
                                  gate count and depth equal Cell 3 exactly, only the DATA is missing
                                  -> rules out "the gap is just lost depth"
  Encoding layers carry NO parameters -> every variant has exactly 144 params, same init, same batch order.
  Note: without re-uploading the data still enters the circuit through the first RY(theta) and through the
  data-dependent RZZ angles a(theta). This test removes only the second injection of theta itself.
  (The old Q6 had a 2nd RZZ layer right before the measurement; it commutes with Z and has no effect on <Z_k>, so the
   current model has one RZZ layer. The PennyLane check below includes it and confirms identical outputs.)

Identical to Cell 3 otherwise: closed-form circuit (verified vs PennyLane), architecture, init key layout, SmoothL1
(beta 1) on log1p(pT) with standardized output, Adam 1e-3, wd 3e-5 (none on circuit angles), clip 1.0, batch 64,
50 epochs, best-val checkpoint, gamma escape, metrics, per-pT-class report.

RUN TIME (CPU): ~4 min per variant per seed -> 3 variants x 1 seed ~ 12 min; x 5 seeds ~ 60 min.
Results accumulate in REUP_RESULTS across re-runs (run one variant at a time if the session may disconnect).
================================================================================
"""
VARIANTS_TO_RUN = ["reupload_ON", "reupload_OFF_drop", "reupload_OFF_depth_matched"]
SEEDS = [SEED]                       # e.g. [42, 1, 60, 100, 120] -> paired comparison over seeds
DEPTH_MATCH_ANGLE = 0.7853981633974483   # = pi/4, fixed RY angle used by reupload_OFF_depth_matched
VERIFY_WITH_PENNYLANE = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_reupload_on_off.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
QUANTUM_KEYS = ("qw",)                         # circuit angles: no weight decay (as Cell 3)
VALID_MODES = ("reupload_ON", "reupload_OFF_drop", "reupload_OFF_depth_matched")
for v in VARIANTS_TO_RUN:
    assert v in VALID_MODES, f"unknown variant {v}"

print("=" * 110)
print(f"DATA RE-UPLOADING ON vs OFF (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM CIRCUIT WITH A RE-UPLOAD SWITCH   (reupload ON == Cell 3's circuit_core exactly)
# ==============================================================================
def circuit_core(theta, W0, W1, a1, mid):
    """mid = "data" (re-upload RY(theta), Cell 3) | "fixed" (RY(DEPTH_MATCH_ANGLE)) | "none" (layer dropped)."""
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    # ---- THE ONLY STRUCTURAL DIFFERENCE ----
    if mid == "data":
        x, z = x * ct + z * st, z * ct - x * st                     # data re-upload RY(theta)
    elif mid == "fixed":
        cf, sf = math.cos(DEPTH_MATCH_ANGLE), math.sin(DEPTH_MATCH_ANGLE)
        x, z = x * cf + z * sf, z * cf - x * sf                     # same depth, no data
    # "none" -> nothing here
    # ----------------------------------------
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

MID = {"reupload_ON": "data", "reupload_OFF_drop": "none", "reupload_OFF_depth_matched": "fixed"}

# ==============================================================================
# 3. VERIFICATION vs PENNYLANE (float64), every variant, reference incl. the old 2nd RZZ layer
# ==============================================================================
if VERIFY_WITH_PENNYLANE:
    try:
        import pennylane as qml
        print("=" * 110 + "\nVERIFYING closed-form circuits vs PennyLane default.qubit (float64; reference includes a 2nd RZZ layer)\n" + "=" * 110)
        dev = qml.device("default.qubit", wires=NQ)
        rs = np.random.default_rng(9)
        th = rs.normal(size=(31, NQ)) * 1.5; W = rs.normal(size=(2, NQ, 2)) * 2
        a1 = rs.uniform(0, math.pi, (31, 4)); a2 = rs.uniform(0, math.pi, (31, 4))
        outs = {}
        for v, mid in MID.items():
            @qml.qnode(dev)
            def _ref(th, W, a1, a2, mid=mid):
                for k in range(NQ): qml.RY(th[:, k], wires=k)
                for k in range(NQ): qml.RZ(W[0, k, 1], wires=k); qml.RY(W[0, k, 0], wires=k)
                for i, (u, w_) in enumerate(RING_PAIRS): qml.IsingZZ(a1[:, i], wires=[u, w_])
                if mid == "data":
                    for k in range(NQ): qml.RY(th[:, k], wires=k)
                elif mid == "fixed":
                    for k in range(NQ): qml.RY(DEPTH_MATCH_ANGLE, wires=k)
                for k in range(NQ): qml.RZ(W[1, k, 1], wires=k); qml.RY(W[1, k, 0], wires=k)
                for i, (u, w_) in enumerate(RING_PAIRS): qml.IsingZZ(a2[:, i], wires=[u, w_])
                return [qml.expval(qml.PauliZ(k)) for k in range(NQ)]
            o_ref = np.stack([np.asarray(o) for o in _ref(th, W, a1, a2)], -1)
            o_cf = np.asarray(circuit_core(jnp.asarray(th, jnp.float32), jnp.asarray(W[0], jnp.float32),
                                           jnp.asarray(W[1], jnp.float32), jnp.asarray(a1, jnp.float32), mid))
            outs[v] = o_cf
            d = np.abs(o_ref - o_cf).max()
            print(f"  {v:<28} max|PennyLane - closed form| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
            assert d < 5e-5, f"closed form disagrees with PennyLane for {v}"
        d_on_off = np.abs(outs["reupload_ON"] - outs["reupload_OFF_drop"]).max()
        print(f"  ON vs OFF outputs actually differ: max|d| = {d_on_off:.3e} (must be >> 0)")
        assert d_on_off > 1e-3
        print("  -> PASS\n")
    except ImportError:
        print("PennyLane not installed -> verification skipped (run Cell 0).")

# ==============================================================================
# 4. MODEL: PI-QGAT with a re-upload switch  (reupload_ON == Cell 3 model, same init)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> every variant starts identical
    return {"node": _init_lin(ks[0], N_FEATS, NQ), "phys": _init_lin(ks[2], 2, 1),
            "head1": _init_lin(ks[3], NQ, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
            "pair": _init_lin(ks[1], 2 * NQ, NQ), "gate1": _init_lin(ks[4], NQ, len(RING_PAIRS)),
            "qw": (jax.random.uniform(ks[5], (2, NQ, 2), jnp.float32) * 2 - 1) * math.pi}

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward_parts(P, x, phi, ba, mode):
    B = x.shape[0]
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, NQ)
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z = circuit_core(tf, P["qw"][0], P["qw"][1], a1, MID[mode]).reshape(B, N_EDGES, NQ)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, N_IN, NQ)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z, tf

def forward(P, x, phi, ba, mode):
    return forward_parts(P, x, phi, ba, mode)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, mode):
        self.mode = mode
        P0 = init_params(0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, mode), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on qw)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, mode))
        self.parts = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, mode)[1:])

# ==============================================================================
# 5. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(fn, w, X, P_, B_, *extra, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([fn(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs], *extra) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C.pred, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.mode} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C.pred, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    z, tf = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])
    z, tf = np.asarray(z, np.float64).reshape(-1, NQ), np.asarray(tf, np.float64).reshape(-1, NQ)
    # how much of each <Z_k> is explained by a LINEAR function of the 4 input angles (theta, sin, cos)? -> nonlinearity
    F_ = np.concatenate([tf, np.sin(tf), np.cos(tf), np.ones((len(tf), 1))], 1)
    coef, *_ = np.linalg.lstsq(F_, z, rcond=None)
    r2_simple = float(1 - ((z - F_ @ coef) ** 2).sum() / max(((z - z.mean(0)) ** 2).sum(), 1e-12))
    full = lambda: predict_log(C.pred, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    return m, bins, dict(z_std=float(z.std(0).mean()), z_absmean=float(np.abs(z).mean()), r2_first_harmonic=r2_simple, us_per_row=us)

# ==============================================================================
# 7. RUN  (results accumulate in REUP_RESULTS across re-runs of this cell)
# ==============================================================================
if "REUP_RESULTS" not in globals():
    REUP_RESULTS = {}

N_PAR = int(ravel_pytree(init_params(0))[0].shape[0])
assert N_PAR == 144, f"model has {N_PAR} params, expected 144 (Cell 3)"
DESC = {"reupload_ON": "RY(theta) -> V1 -> RZZ(a) -> RY(theta) -> V2   (Cell 3)",
        "reupload_OFF_drop": "RY(theta) -> V1 -> RZZ(a) ->            V2   (re-encoding dropped)",
        "reupload_OFF_depth_matched": f"RY(theta) -> V1 -> RZZ(a) -> RY({DEPTH_MATCH_ANGLE:.4f}) -> V2   (fixed, no data)"}
for mode in VARIANTS_TO_RUN:
    C = Compiled(mode)
    print("\n" + "#" * 110 + f"\n# VARIANT: {mode}  |  {DESC[mode]}  |  {N_PAR} params (encoding has no params)\n" + "#" * 110)
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        print(f"   >> {mode} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        print(f"\n   [{mode} | seed {seed}] PER pT CLASS (test)")
        print(f"   {'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>10}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
        for b in bins:
            print(f"   {_label(b['lo'], b['hi']):>16}{b['n']:>9}{b['frac']:>6.1f}%{b['mae_log']:>9.4f}{b['mae_pt']:>10.3f}"
                  f"{b['mse_pt']:>13.2f}{b['rmse_pt']:>10.3f}{b['med_rel']:>10.3f}{b['bias']:>8.3f}{b['sse_share']:>6.1f}%")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_{mode}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        REUP_RESULTS[(mode, seed)] = dict(mode=mode, seed=seed, params=N_PAR, best_ep=tr["best_ep"], best_val=tr["best_val"],
                                          train_total_s=tr["train_total"], escapes=tr["escapes"], test=m, bins=bins,
                                          hist=tr["hist"], diag=diag)

# ==============================================================================
# 8. FINAL REPORT
# ==============================================================================
ORDER = [v for v in VALID_MODES if any(k[0] == v for k in REUP_RESULTS)]
R = {v: [REUP_RESULTS[k] for k in sorted(REUP_RESULTS) if k[0] == v] for v in ORDER}

print("\n" + "=" * 156 + "\nSUMMARY (test set, mean +- std over seeds)\n" + "=" * 156)
print(f"{'variant':<36}{'params':>7}{'seeds':>6}{'best val loss':>14}{'R2_log':>18}{'MAE_log':>18}{'MSE_log':>10}{'RMSE_log':>10}"
      f"{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}")
for v in ORDER:
    rs = R[v]; g = lambda k: [r["test"][k] for r in rs]
    print(f"{v + (' (Cell 3)' if v == 'reupload_ON' else ''):<36}{rs[0]['params']:>7}{len(rs):>6}{np.mean([r['best_val'] for r in rs]):>14.4f}"
          f"{ms(g('r2_log')):>18}{ms(g('mae_log')):>18}{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}"
          f"{ms(g('mae_pt'), '{:.3f}'):>17}{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}")

H2H = {}
if "reupload_ON" in R:
    on = {r["seed"]: r for r in R["reupload_ON"]}
    for v in ORDER:
        if v == "reupload_ON":
            continue
        common = [r["seed"] for r in R[v] if r["seed"] in on]
        if not common:
            continue
        off = {r["seed"]: r for r in R[v]}
        print("\n" + "=" * 90 + f"\nHEAD-TO-HEAD: {v} minus reupload_ON  (paired by seed, {len(common)} seed(s))\n" + "=" * 90)
        H2H[v] = {}
        for lab, key, sign in [("val loss", None, "positive"), ("R2_log", "r2_log", "negative"), ("MAE_log", "mae_log", "positive"),
                               ("RMSE_log", "rmse_log", "positive"), ("MAE (GeV)", "mae_pt", "positive"), ("RMSE (GeV)", "rmse_pt", "positive")]:
            d = [off[s]["best_val"] - on[s]["best_val"] for s in common] if key is None else \
                [off[s]["test"][key] - on[s]["test"][key] for s in common]
            helps = sum((x > 0) if sign == "positive" else (x < 0) for x in d)
            H2H[v][lab] = d
            print(f"  d {lab:<11}: {ms(d, '{:+.4f}'):>20}   ({sign} = re-uploading helps; ON better in {helps}/{len(d)} seeds)")
        print(f"\n{'pT class (GeV)':>16}{'MAE ON':>10}{'MAE OFF':>10}{'d MAE':>10}{'med|rel| ON':>13}{'med|rel| OFF':>14}{'d med|rel|':>12}")
        for b0 in on[common[0]]["bins"]:
            pick = lambda rs_, k: np.mean([[b for b in rs_[s]["bins"] if b["lo"] == b0["lo"]][0][k] for s in common])
            qa, qf, ra, rf = pick(on, "mae_pt"), pick(off, "mae_pt"), pick(on, "med_rel"), pick(off, "med_rel")
            print(f"{_label(b0['lo'], b0['hi']):>16}{qa:>10.3f}{qf:>10.3f}{qf - qa:>+10.3f}{ra:>13.3f}{rf:>14.3f}{rf - ra:>+12.3f}")
else:
    print("\n(reupload_ON not run -> no head-to-head. Add it to VARIANTS_TO_RUN; it is the Cell 3 model with the same init.)")

def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<28}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for v in ORDER:
    rs = R[v]
    print(f"{v:<28}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}")

print(f"\nCIRCUIT DIAGNOSTICS (first {DIAG_ROWS} test events, seed-mean):")
print(f"{'variant':<28}{'std <Z>':>10}{'mean|<Z>|':>11}{'R2 of <Z> from (theta, sin, cos)':>34}")
for v in ORDER:
    g = lambda k: np.mean([r["diag"][k] for r in R[v]])
    print(f"{v:<28}{g('z_std'):>10.4f}{g('z_absmean'):>11.4f}{g('r2_first_harmonic'):>34.3f}")
print("(last column near 1 -> the circuit output is a simple function of the encoded angles; LOWER with re-uploading\n"
      " means the circuit builds richer (higher-frequency) functions of the data -- the capacity re-uploading is supposed to add)")

out = dict(results={f"{v}_seed{s}": x for (v, s), x in REUP_RESULTS.items()}, head_to_head=H2H,
           depth_match_angle=DEPTH_MATCH_ANGLE, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

print("\nReading guide:")
print("  OFF (both modes) clearly worse, same 144 params -> re-uploading adds REPRESENTATIONAL capacity, not just depth")
print("  drop worse but depth_matched equal             -> the gap was the missing RY layer (depth), not the data")
print("  OFF about equal                                -> re-uploading adds little here")
print("  single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01, so smaller gaps need SEEDS = [42, 1, 60, 100, 120]")

DATA RE-UPLOADING ON vs OFF (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['reupload_ON', 'reupload_OFF_drop', 'reupload_OFF_depth_matched']
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001
VERIFYING closed-form circuits vs PennyLane default.qubit (float64; reference includes a 2nd RZZ layer)
  reupload_ON                  max|PennyLane - closed form| = 2.08e-07   (float32 vs float64; must be < 5e-5)
  reupload_OFF_drop            max|PennyLane - closed form| = 2.17e-07   (float32 vs float64; must be < 5e-5)
  reupload_OFF_depth_matched   max|PennyLane - closed form| = 1.91e-07   (float32 vs float64; must be < 5e-5)
  ON vs OFF outputs actually differ: max|d| = 1.792e+00 (must be >> 0)
  -> PASS


##############################################################################################################
# VARIANT: reupload_ON  |  RY(theta) -> V1 -> RZZ(a) -> RY(theta) -> V2   (Cell 3)  |  144 params (encoding has no params)
##########################

##Quantum Circuit capacity ablation

In [ ]:
"""
================================================================================
EXPERIMENT: QUANTUM CIRCUIT CAPACITY ABLATION   (current PI-QGAT, 144 params)                [FAST JAX + JIT]
CELL -- C_nqubit (2-qubit vs 3-qubit vs 4-qubit circuit)
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

PURPOSE: does performance scale with quantum dimensionality / circuit capacity, or saturate early?

VARIANTS (classical structure held fixed; only the circuit width changes):
  C2_2qubit : 2 qubits, state dim  4, ring pairs [(0,1)]
  C3_3qubit : 3 qubits, state dim  8, ring pairs [(0,1),(1,2),(2,0)]
  C4_4qubit : 4 qubits, state dim 16, ring pairs [(0,1),(1,2),(2,3),(3,0)]   == Cell 3 PI-QGAT (same init)

THE CIRCUIT (same family as Cell 3, any n):  RY(theta) -> [RZ RY] -> ring RZZ(a(theta)) -> RY(theta) -> [RZ RY] -> <Z_k>
  (the old Q6's 2nd RZZ layer before the measurement has no effect on <Z>, so the current model has one RZZ layer;
   the PennyLane check below includes it anyway.)

DECOUPLING QUBITS FROM NODE WIDTH (as in the old cell):
  PI-QGAT adds the circuit output to h_j, so normally n_qubits == node width == 4. Here the node width stays 4 in
  EVERY variant -- message passing, attention, physics prior and the 4->6->1 head are untouched -- while pair_proj
  maps 8 -> n_qubits and the circuit output is lifted back to 4 with ZERO parameters:
    LIFT_MODE = "tile" (default): [z0, z1] -> [z0, z1, z0, z1]   (no dead channels)
    LIFT_MODE = "pad"           : [z0, z1] -> [z0, z1, 0, 0]
  At 4 qubits the lift is the identity, so C4 IS the Cell 3 model.
  (This differs from experiment 22 in the circuit-experiments cell, where the node width followed the qubit count.)

PARAMETER BUDGET:
  natural counts are 101 / 123 / 144 (pair_proj, circuit angles and entangle_gate1 all shrink with n), which would
  confound quantum dimension with total capacity.
  PARAM_MATCH = True (default): a purely CLASSICAL filler on the node features, h <- h + tanh(W2 tanh(W1 h + b1) + b2)
    with hidden width H chosen so the total lands closest to 144 -> 141 / 145 / 144 (H = 4 / 2 / 0).
    Any remaining gap is then attributable to quantum dimension, not parameter count.
  PARAM_MATCH = False: natural counts, no filler. Both counts are printed either way.

Identical to Cell 3 otherwise: exact closed-form circuit (verified vs PennyLane per qubit count), init key layout,
SmoothL1 (beta 1) on log1p(pT) with standardized output, Adam 1e-3, wd 3e-5 (none on circuit angles), clip 1.0,
batch 64, 50 epochs, best-val checkpoint, gamma escape, metrics, per-pT-class report.

RUN TIME (CPU): ~3-4 min per variant per seed -> 3 variants x 1 seed ~ 11 min; x 5 seeds ~ 55 min.
Results accumulate in CAPACITY_RESULTS across re-runs (run one variant at a time if the session may disconnect).
================================================================================
"""
VARIANTS_TO_RUN = ["C2_2qubit", "C3_3qubit", "C4_4qubit"]
SEEDS = [SEED]                  # e.g. [42, 1, 60, 100, 120] -> paired comparison over seeds
PARAM_MATCH = True              # True -> classical filler brings every variant near 144
LIFT_MODE = "tile"              # "tile" or "pad" (both parameter-free)
VERIFY_WITH_PENNYLANE = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_circuit_capacity.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
QUANTUM_KEYS = ("qw",)                         # circuit angles: no weight decay (as Cell 3)
VARIANT_NQ = {"C2_2qubit": 2, "C3_3qubit": 3, "C4_4qubit": 4}
D_NODE = NQ                                     # node width stays 4 in EVERY variant (as Cell 3)
TARGET_PARAMS = 144                             # Cell 3 PI-QGAT -> the matching target
for v in VARIANTS_TO_RUN:
    assert v in VARIANT_NQ, f"unknown variant {v}"
assert LIFT_MODE in ("tile", "pad")

print("=" * 110)
print(f"QUANTUM CIRCUIT CAPACITY (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | variants {VARIANTS_TO_RUN} | "
      f"PARAM_MATCH {PARAM_MATCH}, LIFT_MODE {LIFT_MODE}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. PLANNING + EXACT CLOSED-FORM n-QUBIT CIRCUIT  (4 qubits == Cell 3's circuit_core)
# ==============================================================================
def ring_pairs_for(nq):
    if nq == 2:
        return [(0, 1)]                                   # a 2-qubit "ring" is one edge
    return [(i, (i + 1) % nq) for i in range(nq)]

def base_param_count(nq):
    np_ = len(ring_pairs_for(nq))
    return (N_FEATS * D_NODE + D_NODE) + 3 + (D_NODE * HEAD_HIDDEN + HEAD_HIDDEN) + (HEAD_HIDDEN + 1) \
        + (2 * D_NODE * nq + nq) + 2 * nq * 2 + (nq * np_ + np_)
        # node_proj + physics_prior + head 4->6->1 + pair_proj 8->nq + circuit angles + entangle_gate1 nq->pairs

def filler_param_count(H):
    return 0 if H <= 0 else (D_NODE * H + H) + (H * D_NODE + D_NODE)

def choose_filler(nq):
    if not PARAM_MATCH:
        return 0
    base = base_param_count(nq)
    if base >= TARGET_PARAMS:
        return 0
    return min(range(0, 65), key=lambda H: (abs(base + filler_param_count(H) - TARGET_PARAMS), H))

assert base_param_count(4) == 144, "4-qubit model must be the 144-param Cell 3 model"
print("=" * 100)
print(f"CAPACITY PLAN  (node width fixed at {D_NODE}, LIFT_MODE = {LIFT_MODE}, PARAM_MATCH = {PARAM_MATCH})")
print("=" * 100)
print(f"{'variant':<12}{'qubits':>7}{'state dim':>11}{'pairs':>7}{'natural params':>16}{'filler H':>10}{'total':>8}{'vs 144':>8}   ring pairs")
for v, nq in VARIANT_NQ.items():
    H = choose_filler(nq); tot = base_param_count(nq) + filler_param_count(H)
    print(f"{v:<12}{nq:>7}{2 ** nq:>11}{len(ring_pairs_for(nq)):>7}{base_param_count(nq):>16}{H:>10}{tot:>8}{tot - TARGET_PARAMS:>+8}   {ring_pairs_for(nq)}")
print()

def closed_form(theta, W, a, pairs):
    """theta (N,n), W (2,n,2) = (omega, gamma), a (N,P) ZZ angles -> <Z_k> (N,n). Exact for one diagonal ZZ layer."""
    n = theta.shape[1]
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0 = W[0, :, 0], W[0, :, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a), jnp.sin(a)
    hr = [jnp.ones_like(z[:, 0]) for _ in range(n)]
    hi = [jnp.zeros_like(z[:, 0]) for _ in range(n)]
    for p, (u, v) in enumerate(pairs):                  # qubit q's coherence picks up (cos a - i <Z_other> sin a) per ZZ partner
        for q, other in ((u, v), (v, u)):
            fr, fi = ca[:, p], -z[:, other] * sa[:, p]
            hr[q], hi[q] = hr[q] * fr - hi[q] * fi, hr[q] * fi + hi[q] * fr
    hr, hi = jnp.stack(hr, 1), jnp.stack(hi, 1)
    x, y = x * hr + y * hi, -(x * hi - y * hr)
    x, z = x * ct + z * st, z * ct - x * st             # data re-upload RY(theta)
    o1, g1 = W[1, :, 0], W[1, :, 1]
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. VERIFICATION vs PENNYLANE (float64), every qubit count, reference incl. the old 2nd RZZ layer
# ==============================================================================
if VERIFY_WITH_PENNYLANE:
    try:
        import pennylane as qml
        print("=" * 110 + "\nVERIFYING n-qubit closed-form circuits vs PennyLane default.qubit (float64; reference includes a 2nd RZZ layer)\n" + "=" * 110)
        rs = np.random.default_rng(9)
        for v, nq in VARIANT_NQ.items():
            pairs = ring_pairs_for(nq); dev = qml.device("default.qubit", wires=nq)
            th = rs.normal(size=(31, nq)) * 1.5; W = rs.normal(size=(2, nq, 2)) * 2
            a1 = rs.uniform(0, math.pi, (31, len(pairs))); a2 = rs.uniform(0, math.pi, (31, len(pairs)))

            @qml.qnode(dev)
            def _ref(th, W, a1, a2, nq=nq, pairs=pairs):
                for k in range(nq): qml.RY(th[:, k], wires=k)
                for k in range(nq): qml.RZ(W[0, k, 1], wires=k); qml.RY(W[0, k, 0], wires=k)
                for i, (u, w_) in enumerate(pairs): qml.IsingZZ(a1[:, i], wires=[u, w_])
                for k in range(nq): qml.RY(th[:, k], wires=k)
                for k in range(nq): qml.RZ(W[1, k, 1], wires=k); qml.RY(W[1, k, 0], wires=k)
                for i, (u, w_) in enumerate(pairs): qml.IsingZZ(a2[:, i], wires=[u, w_])
                return [qml.expval(qml.PauliZ(k)) for k in range(nq)]

            o_ref = np.stack([np.asarray(o) for o in _ref(th, W, a1, a2)], -1)
            o_cf = np.asarray(closed_form(jnp.asarray(th, jnp.float32), jnp.asarray(W, jnp.float32), jnp.asarray(a1, jnp.float32), pairs))
            d = np.abs(o_ref - o_cf).max()
            print(f"  {nq} qubits ({v:<10}) max|PennyLane - closed form| = {d:.2e}   (float32 vs float64; must be < 5e-5)")
            assert d < 5e-5, f"closed form disagrees with PennyLane for {nq} qubits"
        print("  -> PASS\n")
    except ImportError:
        print("PennyLane not installed -> verification skipped (run Cell 0).")

# ==============================================================================
# 4. MODEL: PI-QGAT with a configurable qubit count  (4 qubits == Cell 3 model, same init)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(nq, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> identical init where shapes agree
    P = {"node": _init_lin(ks[0], N_FEATS, D_NODE), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], D_NODE, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
         "pair": _init_lin(ks[1], 2 * D_NODE, nq), "gate1": _init_lin(ks[4], nq, len(ring_pairs_for(nq))),
         "qw": (jax.random.uniform(ks[5], (2, nq, 2), jnp.float32) * 2 - 1) * math.pi}
    H = choose_filler(nq)
    if H > 0:                                                # purely CLASSICAL parameter-matching filler
        P["fill1"] = _init_lin(ks[6], D_NODE, H); P["fill2"] = _init_lin(ks[7], H, D_NODE)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def lift(z, nq):
    """(..., nq) -> (..., D_NODE), no parameters; identity at nq == D_NODE."""
    if nq == D_NODE:
        return z
    if LIFT_MODE == "tile":
        return z[..., np.array([i % nq for i in range(D_NODE)])]
    return jnp.concatenate([z, jnp.zeros(z.shape[:-1] + (D_NODE - nq,), z.dtype)], -1)

def forward_parts(P, x, phi, ba, nq):
    B = x.shape[0]
    h = _lin(P["node"], x)
    if "fill1" in P:
        h = h + jnp.tanh(_lin(P["fill2"], jnp.tanh(_lin(P["fill1"], h))))   # classical filler
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, nq)   # nq angles per edge
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z_q = closed_form(tf, P["qw"], a1, ring_pairs_for(nq)).reshape(B, N_EDGES, nq)
    s = z_q.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp                                # mean over qubits (any nq)
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + lift(z_q, nq)).reshape(B, N_NODES, N_IN, D_NODE)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z_q

def forward(P, x, phi, ba, nq):
    return forward_parts(P, x, phi, ba, nq)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

def count_parts(nq):
    P = init_params(nq, 0)
    nl = lambda p: int(sum(a.size for a in jax.tree_util.tree_leaves(p)))
    q = nl(P["qw"]) + nl(P["gate1"])
    fill = nl(P.get("fill1", {})) + nl(P.get("fill2", {}))
    return nl(P), q, fill

class Compiled:
    def __init__(self, name):
        self.name, self.nq = name, VARIANT_NQ[name]
        nq = self.nq
        self.init = lambda seed: init_params(nq, seed)
        P0 = init_params(nq, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, nq), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on qw)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, nq))
        self.zout = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, nq)[1])

# ==============================================================================
# 5. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(fn, w, X, P_, B_, *extra, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([fn(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs], *extra) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(C.init(seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C.pred, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.name} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C.pred, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    z = np.asarray(C.zout(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS]), np.float64).reshape(-1, C.nq)
    cz = np.corrcoef(z.T) if C.nq > 1 else np.ones((1, 1))
    redund = float(np.abs(cz[np.triu_indices(C.nq, 1)]).mean()) if C.nq > 1 else float("nan")
    full = lambda: predict_log(C.pred, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    return m, bins, dict(z_std=float(z.std(0).mean()), qubit_corr=redund, us_per_row=us)

# ==============================================================================
# 7. RUN  (results accumulate in CAPACITY_RESULTS across re-runs of this cell)
# ==============================================================================
if "CAPACITY_RESULTS" not in globals():
    CAPACITY_RESULTS = {}

for name in VARIANTS_TO_RUN:
    C = Compiled(name); nq = C.nq
    n_par, n_q, n_fill = count_parts(nq)
    print("\n" + "#" * 110 + f"\n# {name}: {nq} qubits, state dim {2 ** nq}, ring pairs {ring_pairs_for(nq)}  |  params {n_par} "
          f"(Cell 3 = 144, diff {n_par - 144:+d})\n#   circuit angles + entangle gate: {n_q}   classical filler (H={choose_filler(nq)}): {n_fill}"
          f"   lift: {LIFT_MODE if nq < D_NODE else 'identity'} (0 params)\n" + "#" * 110)
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        print(f"   >> {name} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        print(f"\n   [{name} | seed {seed}] PER pT CLASS (test)")
        print(f"   {'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>10}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
        for b in bins:
            print(f"   {_label(b['lo'], b['hi']):>16}{b['n']:>9}{b['frac']:>6.1f}%{b['mae_log']:>9.4f}{b['mae_pt']:>10.3f}"
                  f"{b['mse_pt']:>13.2f}{b['rmse_pt']:>10.3f}{b['med_rel']:>10.3f}{b['bias']:>8.3f}{b['sse_share']:>6.1f}%")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_capacity_{name}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        CAPACITY_RESULTS[(name, seed)] = dict(name=name, seed=seed, nq=nq, qdim=2 ** nq, n_pairs=len(ring_pairs_for(nq)),
                                              params=n_par, params_natural=base_param_count(nq), q_params=n_q,
                                              H=choose_filler(nq), best_ep=tr["best_ep"], best_val=tr["best_val"],
                                              train_total_s=tr["train_total"], escapes=tr["escapes"], test=m, bins=bins,
                                              hist=tr["hist"], diag=diag)

# ==============================================================================
# 8. FINAL REPORT
# ==============================================================================
ORDER = [v for v in VARIANT_NQ if any(k[0] == v for k in CAPACITY_RESULTS)]
R = {v: [CAPACITY_RESULTS[k] for k in sorted(CAPACITY_RESULTS) if k[0] == v] for v in ORDER}

print("\n" + "=" * 160 + "\nSUMMARY (test set, mean +- std over seeds)\n" + "=" * 160)
print(f"{'variant':<20}{'qubits':>7}{'dim':>5}{'params':>8}{'q-par':>7}{'H':>4}{'seeds':>6}{'best val loss':>14}{'R2_log':>18}{'MAE_log':>18}"
      f"{'RMSE_log':>10}{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}")
for v in ORDER:
    rs = R[v]; g = lambda k: [r["test"][k] for r in rs]; r0 = rs[0]
    print(f"{v + (' (Cell 3)' if v == 'C4_4qubit' else ''):<20}{r0['nq']:>7}{r0['qdim']:>5}{r0['params']:>8}{r0['q_params']:>7}{r0['H']:>4}"
          f"{len(rs):>6}{np.mean([r['best_val'] for r in rs]):>14.4f}{ms(g('r2_log')):>18}{ms(g('mae_log')):>18}{np.mean(g('rmse_log')):>10.4f}"
          f"{ms(g('mae_pt'), '{:.3f}'):>17}{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}")
print(f"(natural, unmatched counts: " + ", ".join(f"{v} {base_param_count(VARIANT_NQ[v])}" for v in ORDER) + ")")

DELTAS = {}
if "C4_4qubit" in R:
    ref = {r["seed"]: r for r in R["C4_4qubit"]}
    print("\n" + "=" * 100 + "\nDELTAS vs C4_4qubit  [variant minus 4-qubit, paired by seed; R2: - = fewer qubits worse; errors: +]\n" + "=" * 100)
    print(f"{'variant':<14}{'seeds':>6}{'d R2_log':>18}{'d MAE_log':>18}{'d RMSE_log':>12}{'d MAE_pT':>16}{'d val loss':>12}{'4q better':>11}")
    for v in ORDER:
        if v == "C4_4qubit":
            continue
        common = [r["seed"] for r in R[v] if r["seed"] in ref]
        if not common:
            continue
        x = {r["seed"]: r for r in R[v]}
        d = lambda k: [x[s]["test"][k] - ref[s]["test"][k] for s in common]
        wins = sum(ref[s]["test"]["r2_log"] > x[s]["test"]["r2_log"] for s in common)
        DELTAS[v] = {k: d(k) for k in ("r2_log", "mae_log", "rmse_log", "mae_pt")}
        print(f"{v:<14}{len(common):>6}{ms(d('r2_log'), '{:+.4f}'):>18}{ms(d('mae_log'), '{:+.4f}'):>18}{np.mean(d('rmse_log')):>+12.4f}"
              f"{ms(d('mae_pt'), '{:+.3f}'):>16}{np.mean([x[s]['best_val'] - ref[s]['best_val'] for s in common]):>+12.4f}{f'{wins}/{len(common)}':>11}")

# ---- scaling: does each added qubit still buy accuracy? ----
if len(ORDER) > 1:
    print("\n" + "=" * 80 + "\nSCALING -- marginal gain per added qubit (test R2_log, seed-mean)\n" + "=" * 80)
    chain = sorted(ORDER, key=lambda v: VARIANT_NQ[v])
    r2m = {v: np.mean([r["test"]["r2_log"] for r in R[v]]) for v in chain}
    gains, prev = [], None
    for v in chain:
        nq = VARIANT_NQ[v]
        if prev is None:
            print(f"  {nq} qubits (dim {2 ** nq:>2}): R2_log = {r2m[v]:.4f}   (baseline)")
        else:
            gsc = r2m[v] - r2m[prev]; gains.append((VARIANT_NQ[prev], nq, gsc))
            print(f"  {nq} qubits (dim {2 ** nq:>2}): R2_log = {r2m[v]:.4f}   gain over {VARIANT_NQ[prev]}q = {gsc:+.4f}")
        prev = v
    if len(gains) >= 2:
        first, last = gains[0][2], gains[-1][2]
        print(f"\n  first step {gains[0][0]}->{gains[0][1]}q: {first:+.4f}   last step {gains[-1][0]}->{gains[-1][1]}q: {last:+.4f}")
        if abs(first) > 1e-9:
            print(f"  ratio last/first = {last / first:+.3f}")
            print("    ratio << 1 -> gains SATURATE; extra quantum dimension buys little")
            print("    ratio ~ 1  -> gains still scaling; more qubits likely to help further")

for key, title, fmt in (("med_rel", "med|rel| per pT class", "{:>14.3f}"), ("mae_pt", "MAE (GeV) per pT class", "{:>14.3f}"),
                        ("mae_log", "MAE_log per pT class", "{:>14.4f}")):
    print(f"\n{title} (test, seed-mean)")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{v:>14}" for v in ORDER))
    for b0 in R[ORDER[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r["bins"] if b["lo"] == b0["lo"]][0][key] for r in R[v]])) for v in ORDER))

def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<14}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for v in ORDER:
    rs = R[v]
    print(f"{v:<14}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}")

print(f"\nCIRCUIT DIAGNOSTICS (first {DIAG_ROWS} test events, seed-mean):")
print(f"{'variant':<14}{'std <Z>':>10}{'mean |corr| between qubit outputs':>36}")
for v in ORDER:
    print(f"{v:<14}{np.mean([r['diag']['z_std'] for r in R[v]]):>10.4f}{np.nanmean([r['diag']['qubit_corr'] for r in R[v]]):>36.3f}")
print("(high correlation between qubits -> the extra qubits carry redundant information, i.e. little new capacity)")

out = dict(results={f"{v}_seed{s}": x for (v, s), x in CAPACITY_RESULTS.items()}, deltas_vs_4q=DELTAS,
           param_match=PARAM_MATCH, lift_mode=LIFT_MODE, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

print("\nReading guide:")
print("  monotone improvement 2q < 3q < 4q, gains shrinking -> capacity helps but SATURATES")
print("  gains still large at 4q                            -> worth testing 5-6 qubits next")
print("  2q ~= 4q                                           -> quantum dimension is not the driver; the benefit lies elsewhere")
print("  Note: the qubit count also changes the ring pairs (1/3/4) -- dimension and connectivity move together.")
print("  single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01, so smaller gaps need SEEDS = [42, 1, 60, 100, 120]")

QUANTUM CIRCUIT CAPACITY (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['C2_2qubit', 'C3_3qubit', 'C4_4qubit'] | PARAM_MATCH True, LIFT_MODE tile
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001
CAPACITY PLAN  (node width fixed at 4, LIFT_MODE = tile, PARAM_MATCH = True)
variant      qubits  state dim  pairs  natural params  filler H   total  vs 144   ring pairs
C2_2qubit         2          4      1             101         4     141      -3   [(0, 1)]
C3_3qubit         3          8      3             123         2     145      +1   [(0, 1), (1, 2), (2, 0)]
C4_4qubit         4         16      4             144         0     144      +0   [(0, 1), (1, 2), (2, 3), (3, 0)]

VERIFYING n-qubit closed-form circuits vs PennyLane default.qubit (float64; reference includes a 2nd RZZ layer)
  2 qubits (C2_2qubit ) max|PennyLane - closed form| = 1.76e-07   (float32 vs float64; must be < 5e-5)
  3 qubits (C3_3qubit ) max|PennyLane - closed form| = 1.94e-07   (float3

##QVC Architecture Ablation

In [ ]:
"""
================================================================================
EXPERIMENT: QVC ARCHITECTURE ABLATION (ENTANGLER TYPE)   (current PI-QGAT, 144 params)         [FAST JAX + JIT]
CELL -- E_entangler (which entangling gate, holding topology / depth / params fixed)
Run AFTER Cell 0 and Cell 1. Does not need Cell 3.

PURPOSE: is the result specific to ADAPTIVE RZZ, or a generic variational-circuit effect any entangler would give?

ALL VARIANTS (same circuit skeleton as Cell 3, ring pairs (0,1)(1,2)(2,3)(3,0), same depth):
    RY(theta) -> [RZ(gamma) RY(omega)] -> ENTANGLER on the ring -> RY(theta) re-upload -> [RZ RY] -> <Z_k>
  A_rzz_adaptive : RZZ,  angles = pi*sigmoid(gate1(theta)), per edge per event       [= Cell 3, same init]
  B_cnot         : CNOT (control u -> target v), no angles (discrete, non-parameterised entangler)
  C_crx_adaptive : CRX,  angles = pi*sigmoid(gate1(theta))   (adaptivity kept, GENERATOR swapped ZZ -> controlled-X)
  D_rzz_fixed    : RZZ,  angles = pi*sigmoid(4 learned GLOBAL logits)   (no data dependence; logits start at 0 -> pi/2)

WHY THESE FOUR (as in the old cell):
  B -> does a plain discrete entangler suffice?
  C -> the sharp test: adaptivity preserved, only the generator changes. C ~= A -> the effect is ADAPTIVITY, not RZZ.
  D -> fixed-angle RZZ control, so all four sit in one table.

ONE ENTANGLING LAYER: the old Q6 had a 2nd entangling layer right before the measurement. For RZZ that layer commutes
  with Z and has no effect, so the current model (Cell 3) has one. For CNOT / CRX a layer there WOULD act, so to keep
  the skeleton and depth identical to Cell 3 every variant has exactly one entangling layer, at the same position.

PARAMETER BUDGET:
  natural counts: A 144, B 124 (no gate), C 144, D 128 (4 global angles).
  MATCH_PARAMS = True (default): a purely CLASSICAL filler on the node features, h <- h + tanh(W2 tanh(W1 h + b1) + b2),
    hidden width H chosen so the total lands closest to 144 -> 144 / 146 / 144 / 141 (H = 0 / 2 / 0 / 1).
  MATCH_PARAMS = False: natural counts. Both are printed.
HONEST CAVEAT (as before): CNOT carries no angle, so B is structurally less expressive than A whatever the filler.
  Read B as "discrete vs adaptive entangler"; C is the clean generator-only comparison.

SIMULATION: RZZ variants use Cell 3's exact closed form; CNOT/CRX are not diagonal, so they use an exact 16-amplitude
  state-vector simulation in JAX. Both are verified against PennyLane below (and against each other for RZZ).

Identical to Cell 3 otherwise: init key layout, SmoothL1 (beta 1) on log1p(pT) with standardized output, Adam 1e-3,
wd 3e-5 (none on circuit angles), clip 1.0, batch 64, 50 epochs, best-val checkpoint, gamma escape, metrics.

RUN TIME (CPU): RZZ variants ~4 min, state-vector variants ~6-10 min per seed -> all four ~ 25-30 min for one seed.
Results accumulate in ENTANGLER_RESULTS across re-runs (run one variant at a time if the session may disconnect).
================================================================================
"""
VARIANTS_TO_RUN = ["A_rzz_adaptive", "B_cnot", "C_crx_adaptive", "D_rzz_fixed"]
SEEDS = [SEED]                  # e.g. [42, 1, 60, 100, 120] -> paired comparison over seeds
MATCH_PARAMS = True             # True -> classical filler brings every variant near 144
VERIFY_WITH_PENNYLANE = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_entangler_ablation.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
NQ = 4
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
QUANTUM_KEYS = ("qw", "fixed")                # circuit angles (incl. global entangling-angle logits): no weight decay
TARGET_PARAMS = 144                           # Cell 3 PI-QGAT -> the matching target
D_NODE = NQ
N_PAIRS = len(RING_PAIRS)
ENTANGLER_SPECS = {   # gate: two-qubit entangler | adaptive: "data" (per edge from theta), "global" (learned scalars), "none"
    "A_rzz_adaptive": dict(gate="rzz",  adaptive="data"),      # = Cell 3
    "B_cnot":         dict(gate="cnot", adaptive="none"),
    "C_crx_adaptive": dict(gate="crx",  adaptive="data"),
    "D_rzz_fixed":    dict(gate="rzz",  adaptive="global"),
}
for v in VARIANTS_TO_RUN:
    assert v in ENTANGLER_SPECS, f"unknown variant {v}"

print("=" * 110)
print(f"QVC ARCHITECTURE / ENTANGLER ABLATION (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | "
      f"variants {VARIANTS_TO_RUN} | MATCH_PARAMS {MATCH_PARAMS}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. PARAMETER PLAN
# ==============================================================================
def base_param_count(spec):
    base = (N_FEATS * D_NODE + D_NODE) + 3 + (D_NODE * HEAD_HIDDEN + HEAD_HIDDEN) + (HEAD_HIDDEN + 1) \
         + (2 * D_NODE * NQ + NQ) + 2 * NQ * 2          # node_proj, physics_prior, head 4->6->1, pair_proj, circuit angles
    ent = {"data": NQ * N_PAIRS + N_PAIRS, "global": N_PAIRS, "none": 0}[spec["adaptive"]]
    return base + ent

def filler_param_count(H):
    return 0 if H <= 0 else (D_NODE * H + H) + (H * D_NODE + D_NODE)

def choose_filler(spec):
    if not MATCH_PARAMS:
        return 0
    base = base_param_count(spec)
    if base >= TARGET_PARAMS:
        return 0
    return min(range(0, 65), key=lambda H: (abs(base + filler_param_count(H) - TARGET_PARAMS), H))

assert base_param_count(ENTANGLER_SPECS["A_rzz_adaptive"]) == 144
print("=" * 96)
print(f"ENTANGLER PLAN  (ring topology fixed: {RING_PAIRS}, MATCH_PARAMS = {MATCH_PARAMS})")
print("=" * 96)
print(f"{'variant':<18}{'gate':>7}{'angles':>10}{'natural':>9}{'filler H':>10}{'total':>8}{'vs 144':>8}")
for v, spec in ENTANGLER_SPECS.items():
    H = choose_filler(spec); tot = base_param_count(spec) + filler_param_count(H)
    print(f"{v:<18}{spec['gate'].upper():>7}{ {'data': 'per-edge', 'global': 'global', 'none': 'none'}[spec['adaptive']]:>10}"
          f"{base_param_count(spec):>9}{H:>10}{tot:>8}{tot - TARGET_PARAMS:>+8}")
print()

# ==============================================================================
# 3. CIRCUITS: exact closed form (RZZ, == Cell 3) and exact 16-amplitude state vector (any entangler)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

_IDX = np.arange(2 ** NQ)
_bit = lambda q: (_IDX >> (NQ - 1 - q)) & 1
ZS = jnp.asarray(np.stack([1 - 2 * _bit(q) for q in range(NQ)], 1).astype(np.float32))                       # (16, 4)
ZZ = jnp.asarray(np.stack([(1 - 2 * _bit(u)) * (1 - 2 * _bit(v)) for u, v in RING_PAIRS]).astype(np.float32))  # (4, 16)
CNOT_PERM = [_IDX ^ (_bit(c) << (NQ - 1 - t)) for c, t in RING_PAIRS]
CTRL_T0 = [_IDX[(_bit(c) == 1) & (_bit(t) == 0)] for c, t in RING_PAIRS]
CTRL_T1 = [t0 | (1 << (NQ - 1 - t)) for t0, (c, t) in zip(CTRL_T0, RING_PAIRS)]

def _apply_1q(psi, U, k):
    N = psi.shape[0]
    r = psi.reshape(N, 2 ** k, 2, 2 ** (NQ - k - 1))
    out = jnp.einsum("nij,najb->naib" if U.ndim == 3 else "ij,najb->naib", U, r)
    return out.reshape(N, 2 ** NQ)

def _ry(t):
    c, s = jnp.cos(t / 2), jnp.sin(t / 2)
    return jnp.stack([jnp.stack([c, -s], -1), jnp.stack([s, c], -1)], -2)

def _var_layer(psi, W):
    for k in range(NQ):
        ga = W[k, 1] / 2
        rz = jnp.diag(jnp.stack([jnp.exp(-1j * ga), jnp.exp(1j * ga)]))
        psi = _apply_1q(psi, _ry(W[k, 0]).astype(jnp.complex64) @ rz.astype(jnp.complex64), k)   # RZ first, then RY
    return psi

def statevector(theta, W, a, gate):
    """exact simulation: RY(theta) -> V1 -> entangler -> RY(theta) -> V2 -> <Z_k>.  a: (N, 4) angles or None (CNOT)."""
    N = theta.shape[0]
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, NQ):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(N, -1)
    psi = _var_layer(psi.astype(jnp.complex64), W[0])
    if gate == "rzz":
        psi = psi * jnp.exp(-0.5j * (a @ ZZ)).astype(jnp.complex64)
    elif gate == "cnot":
        for pm in CNOT_PERM:
            psi = psi[:, pm]
    elif gate == "crx":
        for p in range(N_PAIRS):
            cc = jnp.cos(a[:, p:p + 1] / 2).astype(jnp.complex64); ss = jnp.sin(a[:, p:p + 1] / 2).astype(jnp.complex64)
            a0, a1_ = psi[:, CTRL_T0[p]], psi[:, CTRL_T1[p]]
            psi = psi.at[:, CTRL_T0[p]].set(cc * a0 - 1j * ss * a1_).at[:, CTRL_T1[p]].set(-1j * ss * a0 + cc * a1_)
    ry_data = _ry(theta).astype(jnp.complex64)                                        # (N, 4, 2, 2)
    for k in range(NQ):
        psi = _apply_1q(psi, ry_data[:, k], k)
    psi = _var_layer(psi, W[1])
    return (psi.real ** 2 + psi.imag ** 2) @ ZS

def run_circuit(theta, W, a, gate):
    if gate == "rzz":
        return circuit_core(theta, W[0], W[1], a)          # exact closed form, fast (== Cell 3)
    return statevector(theta, W, a, gate)

# ==============================================================================
# 4. VERIFICATION vs PENNYLANE (float64) + closed form vs state vector for RZZ
# ==============================================================================
if VERIFY_WITH_PENNYLANE:
    try:
        import pennylane as qml
        print("=" * 110 + "\nVERIFYING entangler circuits vs PennyLane default.qubit (float64)\n" + "=" * 110)
        dev = qml.device("default.qubit", wires=NQ)
        rs = np.random.default_rng(9)
        th = rs.normal(size=(31, NQ)) * 1.5; W = rs.normal(size=(2, NQ, 2)) * 2
        for v, spec in ENTANGLER_SPECS.items():
            gate, ad = spec["gate"], spec["adaptive"]
            a = rs.uniform(0, math.pi, (31, N_PAIRS)) if ad == "data" else np.broadcast_to(rs.uniform(0, math.pi, N_PAIRS), (31, N_PAIRS))

            @qml.qnode(dev)
            def _ref(th, W, a, gate=gate):
                for k in range(NQ): qml.RY(th[:, k], wires=k)
                for k in range(NQ): qml.RZ(W[0, k, 1], wires=k); qml.RY(W[0, k, 0], wires=k)
                for i, (u, t) in enumerate(RING_PAIRS):
                    if gate == "cnot": qml.CNOT(wires=[u, t])
                    elif gate == "rzz": qml.IsingZZ(a[:, i], wires=[u, t])
                    else: qml.CRX(a[:, i], wires=[u, t])
                for k in range(NQ): qml.RY(th[:, k], wires=k)
                for k in range(NQ): qml.RZ(W[1, k, 1], wires=k); qml.RY(W[1, k, 0], wires=k)
                return [qml.expval(qml.PauliZ(k)) for k in range(NQ)]

            o_ref = np.stack([np.asarray(o) for o in _ref(th, W, np.ascontiguousarray(a))], -1)
            thj, Wj, aj = jnp.asarray(th, jnp.float32), jnp.asarray(W, jnp.float32), jnp.asarray(a, jnp.float32)
            o_run = np.asarray(run_circuit(thj, Wj, aj, gate))
            msg = f"max|PL - {'closed form' if gate == 'rzz' else 'state vector'}| = {np.abs(o_ref - o_run).max():.2e}"
            if gate == "rzz":
                o_sv = np.asarray(statevector(thj, Wj, aj, gate)); msg += f"   max|PL - state vector| = {np.abs(o_ref - o_sv).max():.2e}"
                assert np.abs(o_ref - o_sv).max() < 5e-5
            print(f"  {v:<16} gate={gate.upper():<5} {msg}")
            assert np.abs(o_ref - o_run).max() < 5e-5, f"{v} disagrees with PennyLane"
        print("  -> PASS (float32 vs float64; must be < 5e-5)\n")
    except ImportError:
        print("PennyLane not installed -> verification skipped (run Cell 0).")

# ==============================================================================
# 5. MODEL: PI-QGAT with a configurable entangler  (A_rzz_adaptive == Cell 3 model, same init)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(variant, seed):
    spec = ENTANGLER_SPECS[variant]
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> identical init where shapes agree
    P = {"node": _init_lin(ks[0], N_FEATS, D_NODE), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], D_NODE, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
         "pair": _init_lin(ks[1], 2 * D_NODE, NQ),
         "qw": (jax.random.uniform(ks[5], (2, NQ, 2), jnp.float32) * 2 - 1) * math.pi}
    if spec["adaptive"] == "data":
        P["gate1"] = _init_lin(ks[4], NQ, N_PAIRS)
    elif spec["adaptive"] == "global":
        P["fixed"] = jnp.zeros((N_PAIRS,), jnp.float32)                 # -> pi/2 at start
    H = choose_filler(spec)
    if H > 0:
        P["fill1"] = _init_lin(ks[6], D_NODE, H); P["fill2"] = _init_lin(ks[7], H, D_NODE)
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def ent_angles(P, tf, variant):
    ad = ENTANGLER_SPECS[variant]["adaptive"]
    if ad == "data":
        return jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    if ad == "global":
        return jnp.broadcast_to(jax.nn.sigmoid(P["fixed"]) * math.pi, (tf.shape[0], N_PAIRS))
    return None

def forward_parts(P, x, phi, ba, variant):
    B = x.shape[0]
    h = _lin(P["node"], x)
    if "fill1" in P:
        h = h + jnp.tanh(_lin(P["fill2"], jnp.tanh(_lin(P["fill1"], h))))   # classical filler
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, NQ)
    a = ent_angles(P, tf, variant)
    z = run_circuit(tf, P["qw"], a, ENTANGLER_SPECS[variant]["gate"]).reshape(B, N_EDGES, NQ)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, N_IN, NQ)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z, (a if a is not None else jnp.zeros((tf.shape[0], N_PAIRS)))

def forward(P, x, phi, ba, variant):
    return forward_parts(P, x, phi, ba, variant)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

def count_parts(variant):
    P = init_params(variant, 0)
    nl = lambda p: int(sum(a.size for a in jax.tree_util.tree_leaves(p)))
    return nl(P), nl(P.get("gate1", {})) + nl(P.get("fixed", {})), nl(P.get("fill1", {})) + nl(P.get("fill2", {}))

class Compiled:
    def __init__(self, name):
        self.name = name
        variant = name
        P0 = init_params(variant, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, variant), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)     # clip_grad_norm_
            g = g + wd_vec * w                                                              # torch-Adam L2 (0 on circuit angles)
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, variant))
        self.parts = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, variant)[1:])

# ==============================================================================
# 6. DATA + METRICS  (identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(fn, w, X, P_, B_, *extra, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([fn(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs], *extra) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6b. TRAIN / EVALUATE ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(C.name, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C.pred, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.name} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}"
              + ("   (epoch 1 includes XLA compile)" if ep == 1 else ""), flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, best_val=best, n_par=n_par,
                train_total=time.time() - t_start, escapes=escapes)

def evaluate(C, w):
    pred = predict_log(C.pred, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    z, a = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])
    z, a = np.asarray(z, np.float64).reshape(-1, NQ), np.asarray(a, np.float64)
    full = lambda: predict_log(C.pred, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    has_angles = ENTANGLER_SPECS[C.name]["adaptive"] != "none"
    return m, bins, dict(z_std=float(z.std(0).mean()), us_per_row=us,
                         angle_mean=a.mean(0).tolist() if has_angles else None,
                         angle_std_across_events=a.std(0).tolist() if has_angles else None,
                         ent_strength=float(np.abs(np.sin(a)).mean()) if has_angles else float("nan"))

# ==============================================================================
# 7. RUN  (results accumulate in ENTANGLER_RESULTS across re-runs of this cell)
# ==============================================================================
if "ENTANGLER_RESULTS" not in globals():
    ENTANGLER_RESULTS = {}

for name in VARIANTS_TO_RUN:
    spec = ENTANGLER_SPECS[name]
    C = Compiled(name)
    n_par, n_ent, n_fill = count_parts(name)
    print("\n" + "#" * 110 + f"\n# {name}: entangler = {spec['gate'].upper()}, angles = {spec['adaptive']}  |  params {n_par} "
          f"(Cell 3 = 144, diff {n_par - 144:+d})\n#   circuit angles: 16   entangler params: {n_ent}   classical filler "
          f"(H={choose_filler(spec)}): {n_fill}   simulator: {'closed form' if spec['gate'] == 'rzz' else 'state vector'}\n" + "#" * 110)
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        print(f"   >> {name} | seed {seed} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  "
              f"MSE {m['mse_log']:.4f}  RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  MSE {m['mse_pt']:.1f}  "
              f"RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | {tr['train_total']:.0f}s | gamma escapes {tr['escapes']}")
        if spec["adaptive"] == "global":
            fin = np.asarray(jax.nn.sigmoid(C.unravel(tr["w"])["fixed"]) * math.pi)
            print("   Learned GLOBAL angles (rad, [0, pi]): " + "  ".join(f"{RING_PAIRS[i]}={fin[i]:.4f}" for i in range(N_PAIRS)))
        print(f"\n   [{name} | seed {seed}] PER pT CLASS (test)")
        print(f"   {'pT class (GeV)':>16}{'N':>9}{'% set':>7}{'MAE_log':>9}{'MAE_pT':>10}{'MSE_pT':>13}{'RMSE_pT':>10}{'med|rel|':>10}{'bias':>8}{'%SSE':>7}")
        for b in bins:
            print(f"   {_label(b['lo'], b['hi']):>16}{b['n']:>9}{b['frac']:>6.1f}%{b['mae_log']:>9.4f}{b['mae_pt']:>10.3f}"
                  f"{b['mse_pt']:>13.2f}{b['rmse_pt']:>10.3f}{b['med_rel']:>10.3f}{b['bias']:>8.3f}{b['sse_share']:>6.1f}%")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_entangler_{name}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        ENTANGLER_RESULTS[(name, seed)] = dict(name=name, seed=seed, gate=spec["gate"], adaptive=spec["adaptive"], params=n_par,
                                               params_natural=base_param_count(spec), ent_params=n_ent, H=choose_filler(spec),
                                               best_ep=tr["best_ep"], best_val=tr["best_val"], train_total_s=tr["train_total"],
                                               escapes=tr["escapes"], test=m, bins=bins, hist=tr["hist"], diag=diag)

# ==============================================================================
# 8. FINAL REPORT
# ==============================================================================
ORDER = [v for v in ENTANGLER_SPECS if any(k[0] == v for k in ENTANGLER_RESULTS)]
R = {v: [ENTANGLER_RESULTS[k] for k in sorted(ENTANGLER_RESULTS) if k[0] == v] for v in ORDER}
ANG = {"data": "per-edge", "global": "global", "none": "none"}

print("\n" + "=" * 160 + "\nSUMMARY (test set, mean +- std over seeds)\n" + "=" * 160)
print(f"{'variant':<22}{'gate':>6}{'angles':>10}{'params':>8}{'H':>4}{'seeds':>6}{'best val loss':>14}{'R2_log':>18}{'MAE_log':>18}"
      f"{'RMSE_log':>10}{'MAE_pT GeV':>17}{'MSE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}")
for v in ORDER:
    rs = R[v]; g = lambda k: [r["test"][k] for r in rs]; r0 = rs[0]
    print(f"{v + (' (Cell 3)' if v == 'A_rzz_adaptive' else ''):<22}{r0['gate'].upper():>6}{ANG[r0['adaptive']]:>10}{r0['params']:>8}{r0['H']:>4}"
          f"{len(rs):>6}{np.mean([r['best_val'] for r in rs]):>14.4f}{ms(g('r2_log')):>18}{ms(g('mae_log')):>18}{np.mean(g('rmse_log')):>10.4f}"
          f"{ms(g('mae_pt'), '{:.3f}'):>17}{np.mean(g('mse_pt')):>10.1f}{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}")
print("(natural, unmatched counts: " + ", ".join(f"{v} {base_param_count(ENTANGLER_SPECS[v])}" for v in ORDER) + ")")

DELTAS = {}
if "A_rzz_adaptive" in R:
    ref = {r["seed"]: r for r in R["A_rzz_adaptive"]}
    print("\n" + "=" * 104 + "\nDELTAS vs A_rzz_adaptive  [variant minus adaptive RZZ, paired by seed; R2: - = adaptive RZZ better; errors: +]\n" + "=" * 104)
    print(f"{'variant':<18}{'seeds':>6}{'d R2_log':>18}{'d MAE_log':>18}{'d RMSE_log':>12}{'d MAE_pT':>16}{'d val loss':>12}{'A better':>10}")
    for v in ORDER:
        if v == "A_rzz_adaptive":
            continue
        common = [r["seed"] for r in R[v] if r["seed"] in ref]
        if not common:
            continue
        x = {r["seed"]: r for r in R[v]}
        d = lambda k: [x[s]["test"][k] - ref[s]["test"][k] for s in common]
        wins = sum(ref[s]["test"]["r2_log"] > x[s]["test"]["r2_log"] for s in common)
        DELTAS[v] = {k: d(k) for k in ("r2_log", "mae_log", "rmse_log", "mae_pt")}
        print(f"{v:<18}{len(common):>6}{ms(d('r2_log'), '{:+.4f}'):>18}{ms(d('mae_log'), '{:+.4f}'):>18}{np.mean(d('rmse_log')):>+12.4f}"
              f"{ms(d('mae_pt'), '{:+.3f}'):>16}{np.mean([x[s]['best_val'] - ref[s]['best_val'] for s in common]):>+12.4f}{f'{wins}/{len(common)}':>10}")

    # ---- the decomposition this experiment is really after ----
    if "C_crx_adaptive" in DELTAS or "D_rzz_fixed" in DELTAS:
        print("\n" + "=" * 80 + "\nATTRIBUTION -- is it the GENERATOR (RZZ) or the ADAPTIVITY?\n" + "=" * 80)
        d_gen = np.mean(DELTAS["C_crx_adaptive"]["r2_log"]) if "C_crx_adaptive" in DELTAS else None
        d_ad = np.mean(DELTAS["D_rzz_fixed"]["r2_log"]) if "D_rzz_fixed" in DELTAS else None
        if d_gen is not None:
            print(f"  generator effect  (CRX adaptive - RZZ adaptive) : {d_gen:+.4f} R2_log")
        if d_ad is not None:
            print(f"  adaptivity effect (RZZ fixed    - RZZ adaptive) : {d_ad:+.4f} R2_log")
        if d_gen is not None and d_ad is not None and abs(d_gen) > 1e-9:
            print(f"  ratio |adaptivity| / |generator| = {abs(d_ad) / abs(d_gen):.3f}")
            print("    ratio >> 1 -> ADAPTIVITY drives the result; RZZ is not special")
            print("    ratio ~ 1  -> generator choice matters as much as adaptivity")
            print("    ratio << 1 -> the RZZ generator itself is doing the work")
else:
    print("\n(A_rzz_adaptive not run -> no deltas. Add it to VARIANTS_TO_RUN; it is the Cell 3 model with the same init.)")

for key, title, fmt in (("med_rel", "med|rel| per pT class", "{:>16.3f}"), ("mae_pt", "MAE (GeV) per pT class", "{:>16.3f}"),
                        ("mae_log", "MAE_log per pT class", "{:>16.4f}")):
    print(f"\n{title} (test, seed-mean)")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{v[:15]:>16}" for v in ORDER))
    for b0 in R[ORDER[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r["bins"] if b["lo"] == b0["lo"]][0][key] for r in R[v]])) for v in ORDER))

def ep2(r):
    bv = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * bv), np.nan)

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<18}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'ep->2%':>9}{'best ep':>9}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}")
for v in ORDER:
    rs = R[v]
    print(f"{v:<18}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.nanmean([ep2(r) for r in rs]):>9.1f}{np.mean([r['best_ep'] for r in rs]):>9.1f}"
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
          + f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}")

print(f"\nCIRCUIT DIAGNOSTICS (first {DIAG_ROWS} test events, seed-mean; angles in radians, 0..pi):")
print(f"{'variant':<18}{'std <Z>':>10}{'angle mean':>12}{'angle std across events':>25}{'mean|sin a|':>13}")
for v in ORDER:
    rs = R[v]
    if rs[0]["diag"]["angle_mean"] is None:
        print(f"{v:<18}{np.mean([r['diag']['z_std'] for r in rs]):>10.4f}{'(no angles: CNOT)':>36}")
        continue
    print(f"{v:<18}{np.mean([r['diag']['z_std'] for r in rs]):>10.4f}{np.mean([np.mean(r['diag']['angle_mean']) for r in rs]):>12.3f}"
          f"{np.mean([np.mean(r['diag']['angle_std_across_events']) for r in rs]):>25.3f}{np.mean([r['diag']['ent_strength'] for r in rs]):>13.3f}")
print("(angle std across events = how data-dependent the angles really are: 0 for fixed; small -> adaptive behaves like fixed)")

out = dict(results={f"{v}_seed{s}": x for (v, s), x in ENTANGLER_RESULTS.items()}, deltas_vs_A=DELTAS,
           match_params=MATCH_PARAMS, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")

print("\nReading guide:")
print("  A ~= C, both >> B, D   -> ADAPTIVITY is the driver; RZZ is not special (report this)")
print("  A >> C                 -> the RZZ generator itself matters")
print("  all four close         -> a generic variational-circuit effect, not architecture-specific")
print("  B close to A           -> even a discrete entangler suffices; adaptivity buys little")
print("  Caveat: CNOT has no angle, so B is inherently less expressive than A.")
print("  single seed: PI-QGAT's seed-to-seed R2 spread is ~0.006-0.01, so smaller gaps need SEEDS = [42, 1, 60, 100, 120]")

QVC ARCHITECTURE / ENTANGLER ABLATION (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['A_rzz_adaptive', 'B_cnot', 'C_crx_adaptive', 'D_rzz_fixed'] | MATCH_PARAMS True
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001
ENTANGLER PLAN  (ring topology fixed: [(0, 1), (1, 2), (2, 3), (3, 0)], MATCH_PARAMS = True)
variant              gate    angles  natural  filler H   total  vs 144
A_rzz_adaptive        RZZ  per-edge      144         0     144      +0
B_cnot               CNOT      none      124         2     146      +2
C_crx_adaptive        CRX  per-edge      144         0     144      +0
D_rzz_fixed           RZZ    global      128         1     141      -3

VERIFYING entangler circuits vs PennyLane default.qubit (float64)
  A_rzz_adaptive   gate=RZZ   max|PL - closed form| = 2.08e-07   max|PL - state vector| = 5.87e-07
  B_cnot           gate=CNOT  max|PL - state vector| = 3.04e-07
  C_crx_adaptive   gate=CRX   max|PL - state vector| = 4.11e-07
  D_rzz_fi

##Feature Ablation

In [ ]:
"""
================================================================================
CELL 10 -- FEATURE ABLATION for PI-QGAT (leave-one-feature-out retraining + permutation importance)   [FAST JAX BACKEND]
Needs: ONLY Cell 0 (installs) and Cell 1 (data). Independent of every other cell.

WHAT THIS CELL DOES
  A) LEAVE-ONE-FEATURE-OUT (LOFO) RETRAINING  -- the main ablation
     For each of the 7 station features (Phi, Theta, BendingAngle, TimeInfo, RingNumber, Front, Mask) the feature is
     REMOVED from all 4 stations (node_proj becomes 6->4) and PI-QGAT is retrained from scratch with exactly the Cell 3
     recipe (same model, same circuit, same seed, same batch order, 50 epochs, best-val checkpoint).
     "none" = the full 7-feature model (reference; identical to Cell 3 for the same seed).
     Phi and BendingAngle also feed the physics prior / edge features (|d phi|, |d bending|). With
     ABLATE_EDGE_FEATS_TOO = True that edge feature is removed as well, so the information is really gone.
     Reported per ablation: test log1p(pT) R2/MAE/MSE/RMSE, pT MAE/MSE/RMSE (GeV), clamped rows, per-pT-class MAE_pT,
     convergence (val R2 at epochs 1/2/5/10/25/50, epochs-to-2%, best epoch), training time, and the change vs full.
  B) PERMUTATION IMPORTANCE  (fast cross-check, no retraining)
     The trained FULL model is evaluated on the test set with one feature shuffled across events (all 4 stations of that
     feature shuffled together; for Phi / BendingAngle the edge-feature inputs are shuffled consistently).
     Drop in R2_log / rise in MAE = how much the trained model relies on that feature.
     LOFO says "can the model do without it?", permutation says "does the trained model use it?" -- report both.

RUN TIME: 8 trainings x ~4 min (CPU) ~= 30-35 min for one seed. Shorten with ABLATIONS = ["none", "Phi", ...].
================================================================================
"""
SEEDS = [SEED]                                   # same seed as Cell 3; e.g. [1, 2, 3] for mean +- std
ABLATIONS = ["none", "Phi", "Theta", "BendingAngle", "TimeInfo", "RingNumber", "Front", "Mask"]
ABLATE_EDGE_FEATS_TOO = True                     # removing Phi / BendingAngle also removes |d phi| / |d bending| from the prior
RUN_PERMUTATION = True
PERM_REPEATS = 3
RESULTS_FILE = "piqgat_feature_ablation.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "STATION_FEATS", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val",
           "ba_test", "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12
FEATS = list(STATION_FEATS)
for a in ABLATIONS:
    assert a == "none" or a in FEATS, f"unknown feature '{a}' (choose from {FEATS})"

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 110)
print(f"FEATURE ABLATION (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | ablations {ABLATIONS}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE} | "
      f"edge features removed with Phi/BendingAngle: {ABLATE_EDGE_FEATS_TOO}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM 4-QUBIT CIRCUIT  (identical to Cell 3; verified there against PennyLane)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. PI-QGAT MODEL  (identical to Cell 3 except the node input dimension n_in and an edge-feature mask)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(n_in, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> "none" starts identical to Cell 3
    return {"node": _init_lin(ks[0], n_in, N_QUBITS), "phys": _init_lin(ks[2], 2, 1),
            "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
            "pair": _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS), "gate1": _init_lin(ks[4], N_QUBITS, len(RING_PAIRS)),
            "qw": (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi}

_lin = lambda p, x: x @ p["W"].T + p["b"]

def forward(P, x, phi, ba, edge_mask):
    B = x.shape[0]; n_in = N_NODES - 1
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], ((r - RM) / RS) * edge_mask)[..., 0]               # edge_mask zeroes a removed edge feature
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    u = _lin(P["pair"], jnp.concatenate([h_i, h_j], -1))
    tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z = circuit_core(tf, P["qw"][0], P["qw"][1], a1).reshape(B, N_EDGES, N_QUBITS)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, n_in), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, n_in, N_QUBITS)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

def ablation_spec(ab):
    """-> (kept feature indices, edge mask [bending, phi], effective params removed from the prior)."""
    keep = [i for i, f in enumerate(FEATS) if f != ab]
    em = [1.0, 1.0]
    if ABLATE_EDGE_FEATS_TOO and ab == "BendingAngle": em[0] = 0.0
    if ABLATE_EDGE_FEATS_TOO and ab == "Phi":          em[1] = 0.0
    return keep, jnp.asarray(em, jnp.float32), int(2 - sum(em))

class Compiled:
    def __init__(self, ab):
        self.ab = ab
        self.keep, self.edge_mask, self.n_dead = ablation_spec(ab)
        self.n_in = len(self.keep)
        P0 = init_params(self.n_in, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel, em = self.unravel, self.edge_mask

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, em), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd_vec * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, x, phi, ba):
            return forward(unravel(w), x, phi, ba, em)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred

# ==============================================================================
# 4. DATA + METRICS  (metrics identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
XTR, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
XVA, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
XTE, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = XTR.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([C.pred(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 5. TRAIN ONE (ablation, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    keep = jnp.asarray(C.keep)
    Xtr, Xva = XTR[:, :, keep], XVA[:, :, keep]
    w, _ = ravel_pytree(init_params(C.n_in, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[-{C.ab} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}", flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

def evaluate_test(C, w):
    pred = predict_log(C, w, XTE[:, :, jnp.asarray(C.keep)], pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    return m, bin_stats(pT_test64, pt_pred, pred, y_test)

# ==============================================================================
# 6. A) LEAVE-ONE-FEATURE-OUT RUNS
# ==============================================================================
ALL_AB, FULL_W = {}, {}
for ab in ABLATIONS:
    C = Compiled(ab)
    n_total = int(ravel_pytree(init_params(C.n_in, 0))[0].shape[0])
    what = "FULL MODEL (all 7 features)" if ab == "none" else (
        f"REMOVED: {ab}" + (f" (+ its edge feature in the physics prior)" if C.n_dead else ""))
    print("\n" + "#" * 110 + f"\n# ABLATION: {what}   |   node_proj {C.n_in}->4   |   {n_total} params"
          + (f" ({n_total - C.n_dead} effective)" if C.n_dead else "") + "\n" + "#" * 110)
    res = []
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins = evaluate_test(C, tr["w"])
        h = tr["hist"]
        print(f"   >> -{ab} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  MSE {m['mse_log']:.4f}  "
              f"RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | "
              f"train {tr['train_total']:.0f}s")
        if ab == "none":
            FULL_W[seed] = (C, tr["w"])
            np.savez(os.path.join(RESULTS_DIR, f"piqgat_ablation_full_seed{seed}.npz"), w=np.asarray(tr["w"]))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], n_dead=C.n_dead, train_total_s=tr["train_total"],
                        escapes=tr["escapes"], test=m, bins=bins, hist=h))
    ALL_AB[ab] = res

# ==============================================================================
# 7. A) REPORT
# ==============================================================================
REF = ALL_AB.get("none")
if REF is None:                                    # no full run here -> fall back to Cell 3's saved results
    _qp = os.path.join(RESULTS_DIR, "piqgat_results_quantum.json")
    if os.path.exists(_qp):
        REF = json.load(open(_qp))["quantum"]; print(f"\n(full-model reference loaded from {_qp})")
ref_by_seed = {r["seed"]: r for r in REF} if REF else {}

def delta(rs, key):
    d = [r["test"][key] - ref_by_seed[r["seed"]]["test"][key] for r in rs if r["seed"] in ref_by_seed]
    return d if d else [float("nan")]

def ep2(r):
    best_v = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * best_v), np.nan)

print("\n" + "=" * 150 + "\nA) LEAVE-ONE-FEATURE-OUT ABLATION  (test set; delta = ablated minus full; for R2 negative = feature matters, "
      "for errors positive = feature matters)\n" + "=" * 150)
print(f"{'removed feature':<16}{'params':>8}{'R2_log':>18}{'dR2_log':>11}{'MAE_log':>18}{'dMAE_log':>11}{'MSE_log':>10}{'RMSE_log':>10}"
      f"{'MAE_pT':>17}{'dMAE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}{'ep->2%':>8}{'best ep':>8}")
order = ([("none")] if "none" in ALL_AB else []) + sorted([a for a in ALL_AB if a != "none"], key=lambda a: np.nanmean(delta(ALL_AB[a], "r2_log")))
for ab in order:
    rs = ALL_AB[ab]; g = lambda k: [r["test"][k] for r in rs]
    dr2 = "-" if ab == "none" else f"{np.nanmean(delta(rs, 'r2_log')):+.4f}"
    dma = "-" if ab == "none" else f"{np.nanmean(delta(rs, 'mae_log')):+.4f}"
    dpt = "-" if ab == "none" else f"{np.nanmean(delta(rs, 'mae_pt')):+.3f}"
    print(f"{('(none = full)' if ab == 'none' else ab):<16}{rs[0]['n_par']:>8}{ms(g('r2_log')):>18}{dr2:>11}{ms(g('mae_log')):>18}{dma:>11}"
          f"{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}{ms(g('mae_pt'), '{:.3f}'):>17}{dpt:>10}"
          f"{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}{np.nanmean([ep2(r) for r in rs]):>8.1f}{np.mean([r['best_ep'] for r in rs]):>8.1f}")
print("(rows sorted by dR2_log: most important feature first)")

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TRAINING TIME:")
eps = [e for e in (1, 2, 5, 10, 25, EPOCHS)]
print(f"{'removed feature':<16}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'sec/epoch':>11}{'total s':>9}")
for ab in order:
    rs = ALL_AB[ab]
    print(f"{('(none = full)' if ab == 'none' else ab):<16}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}")

print("\nPER pT CLASS MAE_pT (GeV), test, seed-mean:")
print(f"{'pT class (GeV)':>16}" + "".join(f"{('full' if ab == 'none' else '-' + ab):>15}" for ab in order))
for i, b0 in enumerate(ALL_AB[order[0]][0]["bins"]):
    print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
        f"{np.mean([[b for b in r['bins'] if b['lo'] == b0['lo']][0]['mae_pt'] for r in ALL_AB[ab]]):>15.3f}" for ab in order))

print("\nPER pT CLASS MAE_log, test, seed-mean:")
print(f"{'pT class (GeV)':>16}" + "".join(f"{('full' if ab == 'none' else '-' + ab):>15}" for ab in order))
for i, b0 in enumerate(ALL_AB[order[0]][0]["bins"]):
    print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
        f"{np.mean([[b for b in r['bins'] if b['lo'] == b0['lo']][0]['mae_log'] for r in ALL_AB[ab]]):>15.4f}" for ab in order))

# ==============================================================================
# 8. B) PERMUTATION IMPORTANCE  (trained full model; test set)
# ==============================================================================
PERM = {}
if RUN_PERMUTATION:
    src = None
    if FULL_W:
        src = "this cell's full model"
    else:                                          # fall back to Cell 3's saved weights
        for seed in SEEDS:
            f = os.path.join(RESULTS_DIR, f"piqgat_quantum_seed{seed}.npz")
            if os.path.isfile(f):
                Cf = Compiled("none"); FULL_W[seed] = (Cf, jnp.asarray(np.load(f)["w"])); src = "Cell 3 weights"
    if not FULL_W:
        print("\nB) PERMUTATION IMPORTANCE skipped: no full model (add 'none' to ABLATIONS or run Cell 3).")
    else:
        print("\n" + "=" * 110 + f"\nB) PERMUTATION IMPORTANCE  ({src}; test set; {PERM_REPEATS} shuffles per feature; "
              "drop = permuted minus intact)\n" + "=" * 110)
        PHI_I, BA_I = FEATS.index("Phi"), FEATS.index("BendingAngle")
        for seed, (Cf, wf) in FULL_W.items():
            base, _ = split_metrics(predict_log(Cf, wf, XTE, pte, bte), y_test, pT_test64)
            rng = np.random.default_rng(1000 + seed); PERM[seed] = {}
            for fi, fname in enumerate(FEATS):
                d_r2, d_mae, d_pt = [], [], []
                for _ in range(PERM_REPEATS):
                    perm = jnp.asarray(rng.permutation(XTE.shape[0]))
                    Xp = XTE.at[:, :, fi].set(XTE[perm][:, :, fi])
                    pp = pte[perm] if (fname == "Phi" and ABLATE_EDGE_FEATS_TOO) else pte
                    bp = bte[perm] if (fname == "BendingAngle" and ABLATE_EDGE_FEATS_TOO) else bte
                    mp, _ = split_metrics(predict_log(Cf, wf, Xp, pp, bp), y_test, pT_test64)
                    d_r2.append(mp["r2_log"] - base["r2_log"]); d_mae.append(mp["mae_log"] - base["mae_log"]); d_pt.append(mp["mae_pt"] - base["mae_pt"])
                PERM[seed][fname] = dict(d_r2=d_r2, d_mae_log=d_mae, d_mae_pt=d_pt)
            print(f"seed {seed}: intact test R2_log {base['r2_log']:.4f}  MAE_log {base['mae_log']:.4f}  MAE_pT {base['mae_pt']:.3f} GeV")
        print(f"{'feature':<16}{'dR2_log (perm)':>22}{'dMAE_log (perm)':>22}{'dMAE_pT GeV (perm)':>22}{'dR2_log (LOFO)':>18}")
        rank = sorted(FEATS, key=lambda f: np.mean([np.mean(PERM[s][f]["d_r2"]) for s in PERM]))
        for f in rank:
            allv = lambda k: [x for s in PERM for x in PERM[s][f][k]]
            lofo = f"{np.nanmean(delta(ALL_AB[f], 'r2_log')):+.4f}" if f in ALL_AB and ref_by_seed else "-"
            print(f"{f:<16}{ms(allv('d_r2'), '{:+.4f}'):>22}{ms(allv('d_mae_log'), '{:+.4f}'):>22}{ms(allv('d_mae_pt'), '{:+.3f}'):>22}{lofo:>18}")
        print("(sorted by permutation dR2_log: most relied-on feature first; +- here is over shuffles x seeds)")

# ==============================================================================
# 9. SAVE
# ==============================================================================
out = dict(ablations=ALL_AB, permutation={str(s): v for s, v in PERM.items()}, features=FEATS,
           ablate_edge_feats_too=ABLATE_EDGE_FEATS_TOO, seeds=SEEDS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")
print("HOW TO READ: LOFO dR2 << 0 -> the model cannot recover the information from the other features (truly needed).\n"
      "             Permutation dR2 << 0 but LOFO ~ 0 -> the trained model uses it, but it is redundant with other features.\n"
      "             Both ~ 0 -> the feature can be dropped.  Single-seed gaps below ~0.005 R2 are within noise.")

FEATURE ABLATION (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | ablations ['none', 'Phi', 'Theta', 'BendingAngle', 'TimeInfo', 'RingNumber', 'Front', 'Mask']
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001 | edge features removed with Phi/BendingAngle: True

##############################################################################################################
# ABLATION: FULL MODEL (all 7 features)   |   node_proj 7->4   |   144 params
##############################################################################################################
[-none | seed 42] ep  1/50  train 0.1842  val loss 0.1540  R2_log 0.5052  MAE_log 0.3613  MAE_pT 15.023 GeV   16.3s *best
[-none | seed 42] ep  2/50  train 0.1359  val loss 0.1007  R2_log 0.7002  MAE_log 0.3057  MAE_pT 13.826 GeV    4.6s *best
[-none | seed 42] ep  3/50  train 0.0806  val loss 0.0707  R2_log 0.7977  MAE_log 0.2497  MAE_pT 12.681 GeV    4.0s *best
[-none | seed 42] ep  4/50  train 0.0675  val loss 

three features carry almost all the information. Phi matters most by far; Theta and BendingAngle matter a lot; the other four barely matter.

#Graph Attention Ablation

In [ ]:
"""
================================================================================
CELL 12 -- GRAPH-ATTENTION ABLATION for PI-QGAT                                                     [FAST JAX BACKEND]
Needs: ONLY Cell 0 (installs) and Cell 1 (data). Independent of every other cell.

PI-QGAT message passing (Cell 3), for receiver station i and its incoming edges j:
    z_ij   = quantum block( pi*tanh(pair_proj[h_i, h_j]) )                     4 numbers per edge
    s_ij   = mean_k z_ij,k  +  lambda * P_ij                                     attention logit (quantum term + physics prior)
    a_ij   = softmax_j(s_ij)                                                     over the 3 incoming edges
    m_i    = sum_j a_ij (h_j + z_ij)                                             message = neighbour state + quantum term
    g      = mean_i (h_i + m_i)  ->  head  ->  log1p(pT)
The quantum output z is used TWICE: inside the attention logit AND inside the message.

A) RETRAINING ABLATION  (Cell 3 recipe, same seed / batch order / 50 epochs, best-val checkpoint)
   full              : Cell 3 model (reference)
   uniform_attn      : a_ij = 1/3 (mean aggregation, no attention at all; prior unused -> 141 params)
   z_msg_only        : quantum term removed from the ATTENTION (s = lambda*P only), kept in the MESSAGE
   z_attn_only       : quantum term kept in the ATTENTION, removed from the MESSAGE (m = sum a h_j)
   no_quantum        : quantum term in neither (attention = prior only, message = h_j)  -> 72 params (= Cell 4 noquantum)
   no_message_passing: no graph at all, g = mean_i h_i  (the stations never exchange information) -> 69 params
   chain_graph       : only ADJACENT stations are connected (0-1, 1-2, 2-3 in station order) instead of all pairs
   Reported per variant: test log1p(pT) R2/MAE/MSE/RMSE, pT MAE/MSE/RMSE (GeV), clamped rows, per-pT-class MAE_pT and
   MAE_log, convergence (val R2 at epochs 1/2/5/10/25/50, epochs-to-2%, best epoch), training time, delta vs full.
B) INFERENCE-TIME RELIANCE  (trained FULL model, no retraining): attention replaced by uniform weights, and attention
   weights taken from a random OTHER event (messages unchanged).  Large drop = the trained model uses event-specific attention.
C) ATTENTION MAPS  (models with learned attention): mean attention matrix receiver x sender (which station listens to which),
   normalised attention entropy (1 = uniform, 0 = one edge), share of attention on adjacent stations.

RUN TIME: 7 trainings x ~4 min (CPU) ~= 25-30 min for one seed. Trim VARIANTS_TO_RUN for a quick look.
================================================================================
"""
SEEDS = [SEED]
VARIANTS = {   # attn: "learned"|"uniform", z_attn / z_msg: quantum term in logit / message, prior: physics prior, mp: message passing, graph
    "full":               dict(attn="learned", z_attn=True,  z_msg=True,  prior=True,  mp=True,  graph="full"),
    "uniform_attn":       dict(attn="uniform", z_attn=False, z_msg=True,  prior=False, mp=True,  graph="full"),
    "z_msg_only":         dict(attn="learned", z_attn=False, z_msg=True,  prior=True,  mp=True,  graph="full"),
    "z_attn_only":        dict(attn="learned", z_attn=True,  z_msg=False, prior=True,  mp=True,  graph="full"),
    "no_quantum":         dict(attn="learned", z_attn=False, z_msg=False, prior=True,  mp=True,  graph="full"),
    "no_message_passing": dict(attn="uniform", z_attn=False, z_msg=False, prior=False, mp=False, graph="full"),
    "chain_graph":        dict(attn="learned", z_attn=True,  z_msg=True,  prior=True,  mp=True,  graph="chain"),
}
VARIANTS_TO_RUN = list(VARIANTS)
RUN_INFERENCE_RELIANCE = True
RUN_ATTENTION_MAPS = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_graph_attention_ablation.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "STATIONS", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12
for v in VARIANTS_TO_RUN:
    assert v in VARIANTS, f"unknown variant {v}"

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 110)
print(f"GRAPH-ATTENTION ABLATION (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE} | "
      f"nodes = stations {list(STATIONS)}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM 4-QUBIT CIRCUIT  (identical to Cell 3; verified there against PennyLane)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. PI-QGAT MODEL WITH SWITCHABLE ATTENTION / MESSAGE / GRAPH  ("full" == Cell 3 exactly)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))
EDGE_OK = {"full": np.ones(N_EDGES, bool), "chain": (np.abs(RECV - SEND) == 1)}     # node index = position in STATIONS
for gname, ok in EDGE_OK.items():
    assert ok.reshape(N_NODES, N_IN).any(1).all(), f"graph '{gname}' leaves a node without incoming edges"

def needs_quantum(spec):
    return spec["mp"] and (spec["z_attn"] or spec["z_msg"])

def uses_prior(spec):
    return spec["mp"] and spec["attn"] == "learned" and spec["prior"]

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(spec, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> "full" starts identical to Cell 3
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if uses_prior(spec):
        P["phys"] = _init_lin(ks[2], 2, 1)
    if needs_quantum(spec):
        P["pair"] = _init_lin(ks[1], 2 * N_QUBITS, N_QUBITS)
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def edge_terms(P, x, phi, ba, spec):
    """node states h, quantum term z (B,12,4) and attention logit s (B,12) for one batch of events."""
    B = x.shape[0]
    h = _lin(P["node"], x)
    if needs_quantum(spec):
        u = _lin(P["pair"], jnp.concatenate([h[:, RECV], h[:, SEND]], -1))
        tf = (math.pi * jnp.tanh(u)).reshape(-1, N_QUBITS)
        a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
        z = circuit_core(tf, P["qw"][0], P["qw"][1], a1).reshape(B, N_EDGES, N_QUBITS)
    else:
        z = jnp.zeros((B, N_EDGES, N_QUBITS), h.dtype)
    s = jnp.zeros((B, N_EDGES), h.dtype)
    if spec["attn"] == "learned":
        if spec["z_attn"]:
            s = s + z.mean(-1)
        if uses_prior(spec):
            d = phi[:, SEND] - phi[:, RECV]
            dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
            r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
            Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
            if BOUND_PHYSICS_PRIOR:
                Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
            s = s + LAMBDA_PHYSICS_ATTN * Pp
    return h, z, s

def attention(s, spec):
    B = s.shape[0]
    if spec.get("force_uniform"):                                     # test-time override (section B), parameters unchanged
        s = jnp.zeros_like(s)
    ok = jnp.asarray(EDGE_OK[spec["graph"]]).reshape(N_NODES, N_IN)
    logit = jnp.where(ok, s.reshape(B, N_NODES, N_IN), -1e9)         # uniform attn: s = 0 -> equal weights on allowed edges
    return jax.nn.softmax(logit, -1) * ok

def readout(P, h, z, alpha, spec):
    B = h.shape[0]
    if spec["mp"]:
        msg = h[:, SEND] + (z if spec["z_msg"] else 0.0)
        m = (alpha[..., None] * msg.reshape(B, N_NODES, N_IN, N_QUBITS)).sum(2)
        g = (h + m).mean(1)
    else:
        g = h.mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y

def forward(P, x, phi, ba, spec):
    h, z, s = edge_terms(P, x, phi, ba, spec)
    return readout(P, h, z, attention(s, spec), spec)

def forward_foreign_attention(P, x, phi, ba, x2, phi2, ba2, spec):
    """messages from event (x, phi, ba) but attention weights computed from another event (x2, phi2, ba2)."""
    h, z, _ = edge_terms(P, x, phi, ba, spec)
    _, _, s2 = edge_terms(P, x2, phi2, ba2, spec)
    return readout(P, h, z, attention(s2, spec), spec)

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, name, spec=None):
        self.name, self.spec = name, dict(spec or VARIANTS[name])
        spec = self.spec
        P0 = init_params(spec, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, spec), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd_vec * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, x, phi, ba):
            return forward(unravel(w), x, phi, ba, spec)

        @jax.jit
        def pred_foreign(w, x, phi, ba, x2, phi2, ba2):
            return forward_foreign_attention(unravel(w), x, phi, ba, x2, phi2, ba2, spec)

        @jax.jit
        def attn(w, x, phi, ba):
            _, _, s = edge_terms(unravel(w), x, phi, ba, spec)
            return attention(s, spec)

        self.epoch, self.one_step, self.pred, self.pred_foreign, self.attn = epoch, one_step, pred, pred_foreign, attn

# ==============================================================================
# 4. DATA + METRICS  (metrics identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([C.pred(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 5. TRAIN ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(C.spec, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    has_q = needs_quantum(C.spec)
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and has_q:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.name} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}", flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

def evaluate_test(C, w):
    pred = predict_log(C, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    return m, bin_stats(pT_test64, pt_pred, pred, y_test)

def attention_map(C, w):
    a = np.asarray(C.attn(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS]), np.float64)      # (B, 4, 3)
    ok = EDGE_OK[C.spec["graph"]].reshape(N_NODES, N_IN)
    M = np.full((N_NODES, N_NODES), np.nan)
    for e in range(N_EDGES):
        i, k = divmod(e, N_IN)
        if ok[i, k]:
            M[RECV[e], SEND[e]] = a[:, i, k].mean()
    n_allowed = ok.sum(1)                                                           # entropy normalised by log(#allowed edges)
    ent = -(a * np.log(np.clip(a, 1e-12, 1))).sum(-1)
    norm = np.where(n_allowed > 1, np.log(np.maximum(n_allowed, 2)), 1.0)
    ent_n = float((ent / norm).mean())
    adj = (np.abs(RECV - SEND) == 1).reshape(N_NODES, N_IN)
    adj_share = float((a * adj).sum(-1).mean())
    return dict(matrix=M.tolist(), entropy=ent_n, adjacent_share=adj_share,
                max_weight_mean=float(a.max(-1).mean()))

# ==============================================================================
# 6. A) RETRAINING ABLATION
# ==============================================================================
ALL_GA, MODELS = {}, {}
for name in VARIANTS_TO_RUN:
    C = Compiled(name); spec = C.spec
    n_total = int(ravel_pytree(init_params(spec, 0))[0].shape[0])
    if not spec["mp"]:
        desc = "no message passing: g = mean of node states (stations never exchange information)"
    else:
        att = "uniform 1/3" if spec["attn"] == "uniform" else (
            " + ".join([t for t, on in (("quantum", spec["z_attn"]), ("physics prior", uses_prior(spec))) if on]) or "constant")
        msg = "h_j + z_ij" if spec["z_msg"] else "h_j"
        desc = f"attention logit = {att} | message = {msg} | graph = {spec['graph']}"
    print("\n" + "#" * 110 + f"\n# VARIANT: {name}   |   {n_total} params\n#   {desc}\n" + "#" * 110)
    res = []
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins = evaluate_test(C, tr["w"])
        amap = attention_map(C, tr["w"]) if (RUN_ATTENTION_MAPS and spec["mp"] and spec["attn"] == "learned") else None
        print(f"   >> {name} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  MSE {m['mse_log']:.4f}  "
              f"RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | "
              f"train {tr['train_total']:.0f}s" + (f" | attention entropy {amap['entropy']:.3f}" if amap else ""))
        MODELS[(name, seed)] = (C, tr["w"])
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_graphattn_{name}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        spec=spec, test=m, bins=bins, hist=tr["hist"], attention=amap))
    ALL_GA[name] = res

# ==============================================================================
# 7. A) REPORT
# ==============================================================================
REF = ALL_GA.get("full")
if REF is None:
    _qp = os.path.join(RESULTS_DIR, "piqgat_results_quantum.json")
    if os.path.exists(_qp):
        REF = json.load(open(_qp))["quantum"]; print(f"\n(full-model reference loaded from {_qp})")
ref_by_seed = {r["seed"]: r for r in REF} if REF else {}

def delta(rs, key):
    d = [r["test"][key] - ref_by_seed[r["seed"]]["test"][key] for r in rs if r["seed"] in ref_by_seed]
    return d if d else [float("nan")]

def ep2(r):
    best_v = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * best_v), np.nan)

order = (["full"] if "full" in ALL_GA else []) + [v for v in VARIANTS_TO_RUN if v != "full"]
W = 20
print("\n" + "=" * 162 + "\nA) GRAPH-ATTENTION RETRAINING ABLATION  (test set; delta = variant minus full; for R2 negative = the removed part "
      "mattered, for errors positive = it mattered)\n" + "=" * 162)
print(f"{'variant':<{W}}{'params':>7}{'R2_log':>18}{'dR2_log':>10}{'MAE_log':>18}{'dMAE_log':>10}{'MSE_log':>10}{'RMSE_log':>10}"
      f"{'MAE_pT':>17}{'dMAE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}{'ep->2%':>8}{'best ep':>8}")
for v in order:
    rs = ALL_GA[v]; g = lambda k: [r["test"][k] for r in rs]
    dd = lambda k, f: "-" if v == "full" else f.format(np.nanmean(delta(rs, k)))
    print(f"{v:<{W}}{rs[0]['n_par']:>7}{ms(g('r2_log')):>18}{dd('r2_log', '{:+.4f}'):>10}{ms(g('mae_log')):>18}{dd('mae_log', '{:+.4f}'):>10}"
          f"{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}{ms(g('mae_pt'), '{:.3f}'):>17}{dd('mae_pt', '{:+.3f}'):>10}"
          f"{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}{np.nanmean([ep2(r) for r in rs]):>8.1f}{np.mean([r['best_ep'] for r in rs]):>8.1f}")

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TRAINING TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<{W}}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'sec/epoch':>11}{'total s':>9}")
for v in order:
    rs = ALL_GA[v]
    print(f"{v:<{W}}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}")

short = {"full": "full", "uniform_attn": "uniform", "z_msg_only": "z_msg_only", "z_attn_only": "z_attn_only",
         "no_quantum": "no_quantum", "no_message_passing": "no_MP", "chain_graph": "chain"}
for title, key, fmt in [("PER pT CLASS MAE_pT (GeV)", "mae_pt", "{:>13.3f}"), ("PER pT CLASS MAE_log", "mae_log", "{:>13.4f}")]:
    print(f"\n{title}, test, seed-mean:")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{short.get(v, v):>13}" for v in order))
    for b0 in ALL_GA[order[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r['bins'] if b['lo'] == b0['lo']][0][key] for r in ALL_GA[v]])) for v in order))

# ==============================================================================
# 8. B) INFERENCE-TIME RELIANCE OF THE TRAINED FULL MODEL
# ==============================================================================
REL = {}
if RUN_INFERENCE_RELIANCE:
    fulls = {s: MODELS[("full", s)] for s in SEEDS if ("full", s) in MODELS}
    if not fulls:
        for s in SEEDS:
            f = os.path.join(RESULTS_DIR, f"piqgat_quantum_seed{s}.npz")
            if os.path.isfile(f):
                fulls[s] = (Compiled("full"), jnp.asarray(np.load(f)["w"]))
    if not fulls:
        print("\nB) skipped: no trained full model (add 'full' to VARIANTS_TO_RUN or run Cell 3).")
    else:
        print("\n" + "=" * 110 + "\nB) INFERENCE-TIME RELIANCE ON ATTENTION  (trained FULL model, no retraining; test set)\n" + "=" * 110)
        print(f"{'test-time change':<46}{'R2_log':>10}{'dR2_log':>10}{'MAE_log':>10}{'dMAE_log':>10}{'MAE_pT':>10}{'dMAE_pT':>10}")
        shuf = jnp.asarray(np.random.default_rng(777).permutation(Xte.shape[0]))
        for s, (Cf, wf) in fulls.items():
            C_uni = Compiled("full", dict(Cf.spec, force_uniform=True))     # same parameters, attention forced to 1/3
            outs = {"intact (reference)": predict_log(Cf, wf, Xte, pte, bte),
                    "attention forced uniform (1/3)": predict_log(C_uni, wf, Xte, pte, bte)}
            bs = EVAL_BATCH_SIZE
            Xs, ps, bs_ = Xte[shuf], pte[shuf], bte[shuf]
            fo = jnp.concatenate([Cf.pred_foreign(wf, Xte[i:i + bs], pte[i:i + bs], bte[i:i + bs], Xs[i:i + bs], ps[i:i + bs], bs_[i:i + bs])
                                  for i in range(0, Xte.shape[0], bs)])
            outs["attention weights from another event"] = np.asarray(fo)
            base = None; REL[s] = {}
            for lab, pred in outs.items():
                m, _ = split_metrics(pred, y_test, pT_test64)
                base = base or m; REL[s][lab] = m
                print(f"{lab:<46}{m['r2_log']:>10.4f}{m['r2_log'] - base['r2_log']:>+10.4f}{m['mae_log']:>10.4f}"
                      f"{m['mae_log'] - base['mae_log']:>+10.4f}{m['mae_pt']:>10.3f}{m['mae_pt'] - base['mae_pt']:>+10.3f}")

# ==============================================================================
# 9. C) ATTENTION MAPS
# ==============================================================================
if RUN_ATTENTION_MAPS:
    print("\n" + "=" * 110 + f"\nC) ATTENTION MAPS  (first {DIAG_ROWS} test events; learned-attention models only)\n" + "=" * 110)
    print(f"{'variant':<{W}}{'entropy (1=uniform)':>21}{'mean max weight':>17}{'share on adjacent stations':>28}")
    for v in order:
        ams = [r["attention"] for r in ALL_GA[v] if r.get("attention")]
        if ams:
            print(f"{v:<{W}}{np.mean([a['entropy'] for a in ams]):>21.3f}{np.mean([a['max_weight_mean'] for a in ams]):>17.3f}"
                  f"{np.mean([a['adjacent_share'] for a in ams]):>28.3f}")
    print("(uniform attention would give entropy 1.000, mean max weight 0.333, adjacent share 0.556 on the full graph)")
    for v in order:
        ams = [r["attention"] for r in ALL_GA[v] if r.get("attention")]
        if not ams:
            continue
        M = np.nanmean([np.asarray(a["matrix"], float) for a in ams], 0)
        print(f"\n  {v}: mean attention weight, receiver station (row) <- sender station (column)")
        print("            " + "".join(f"{'st ' + str(s):>9}" for s in STATIONS))
        for i, s in enumerate(STATIONS):
            print(f"   st {s:<5}" + "".join(f"{'-':>9}" if np.isnan(M[i, j]) else f"{M[i, j]:>9.3f}" for j in range(N_NODES)))

# ==============================================================================
# 10. SAVE
# ==============================================================================
out = dict(variants=ALL_GA, reliance={str(s): v for s, v in REL.items()}, seeds=SEEDS, specs=VARIANTS, stations=list(STATIONS))
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")
print("HOW TO READ: uniform_attn ~ full -> learned attention adds little; the gain comes from the messages.\n"
      "             z_msg_only vs z_attn_only -> where the quantum term earns its keep (attention logit or message).\n"
      "             no_quantum / no_message_passing -> value of the quantum block / of letting stations exchange information.\n"
      "             chain_graph ~ full -> long-range station pairs are not needed.  Gaps below ~0.01 R2 are single-seed noise.")

GRAPH-ATTENTION ABLATION (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['full', 'uniform_attn', 'z_msg_only', 'z_attn_only', 'no_quantum', 'no_message_passing', 'chain_graph']
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001 | nodes = stations [0, 2, 3, 4]

##############################################################################################################
# VARIANT: full   |   144 params
#   attention logit = quantum + physics prior | message = h_j + z_ij | graph = full
##############################################################################################################
[full | seed 42] ep  1/50  train 0.1842  val loss 0.1540  R2_log 0.5052  MAE_log 0.3613  MAE_pT 15.023 GeV   18.9s *best
[full | seed 42] ep  2/50  train 0.1359  val loss 0.1007  R2_log 0.7002  MAE_log 0.3057  MAE_pT 13.826 GeV    5.4s *best
[full | seed 42] ep  3/50  train 0.0806  val loss 0.0707  R2_log 0.7977  MAE_log 0.2497  MAE_pT 12.681 GeV    4.8s *best
[full |

#Message Passing Ablation

In [ ]:
"""
================================================================================
CELL 13 -- MESSAGE-PASSING ABLATION for PI-QGAT                                                     [FAST JAX BACKEND]
Needs: ONLY Cell 0 (installs) and Cell 1 (data). Independent of every other cell.

PI-QGAT message passing (Cell 3), receiver station i, incoming edges j (3 per node):
    theta_ij = pi*tanh(pair_proj[h_i, h_j])            edge input: BOTH endpoints
    z_ij     = quantum block(theta_ij)                  4 numbers per edge
    a_ij     = softmax_j(mean_k z_ij,k + lambda*P_ij)   attention
    m_i      = sum_j a_ij (h_j + z_ij)                  MESSAGE = neighbour state + quantum term, attention-weighted SUM
    h'_i     = h_i + m_i                                RESIDUAL update, ONE round
    g        = mean_i h'_i                              MEAN readout  ->  head  ->  log1p(pT)

A) RETRAINING ABLATION  (Cell 3 recipe, same seed / batch order / 50 epochs, best-val checkpoint). One change per variant:
   full               : Cell 3 model (reference)
   msg_z_only         : message = z_ij only (neighbour state h_j dropped from the message)
   no_residual        : h'_i = m_i (the node forgets its own state h_i)
   sum_aggregation    : m_i = sum_j (h_j + z_ij)      (no attention / normalisation; prior unused -> 141 params)
   max_aggregation    : m_i = max_j (h_j + z_ij)      (element-wise max over neighbours; prior unused -> 141 params)
   readout_max        : g = max_i h'_i instead of the mean
   two_rounds         : 2 rounds of message passing with the SAME (shared) weights -> 144 params, 2x circuit calls
   z_sender_only      : theta_ij = pi*tanh(pair_proj[h_j])  -> the edge function sees only the sender, not the PAIR (128 params)
   no_message_passing : no graph, g = mean_i h_i (69 params)
   Reported per variant: test log1p(pT) R2/MAE/MSE/RMSE, pT MAE/MSE/RMSE (GeV), clamped rows, per-pT-class MAE_pT and
   MAE_log, convergence (val R2 at epochs 1/2/5/10/25/50, epochs-to-2%, best epoch), training time, delta vs full.
B) MESSAGE DIAGNOSTICS (trained models, first 65536 test events): size of the neighbour part vs the quantum part of the
   message, and how much the node state changes in the update (|m| / |h|).

RUN TIME: 9 trainings, ~4 min each on CPU (two_rounds ~8 min) ~= 35-40 min for one seed. Trim VARIANTS_TO_RUN for a quick look.
================================================================================
"""
SEEDS = [SEED]
VARIANTS = {   # msg: "h+z"|"z", residual, aggr: "attn"|"sum"|"max", readout: "mean"|"max", rounds: 0/1/2, pair_in: "both"|"sender"
    "full":               dict(msg="h+z", residual=True,  aggr="attn", readout="mean", rounds=1, pair_in="both"),
    "msg_z_only":         dict(msg="z",   residual=True,  aggr="attn", readout="mean", rounds=1, pair_in="both"),
    "no_residual":        dict(msg="h+z", residual=False, aggr="attn", readout="mean", rounds=1, pair_in="both"),
    "sum_aggregation":    dict(msg="h+z", residual=True,  aggr="sum",  readout="mean", rounds=1, pair_in="both"),
    "max_aggregation":    dict(msg="h+z", residual=True,  aggr="max",  readout="mean", rounds=1, pair_in="both"),
    "readout_max":        dict(msg="h+z", residual=True,  aggr="attn", readout="max",  rounds=1, pair_in="both"),
    "two_rounds":         dict(msg="h+z", residual=True,  aggr="attn", readout="mean", rounds=2, pair_in="both"),
    "z_sender_only":      dict(msg="h+z", residual=True,  aggr="attn", readout="mean", rounds=1, pair_in="sender"),
    "no_message_passing": dict(msg="h+z", residual=True,  aggr="attn", readout="mean", rounds=0, pair_in="both"),
}
VARIANTS_TO_RUN = list(VARIANTS)
RUN_MESSAGE_DIAGNOSTICS = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_message_passing_ablation.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_QUBITS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_QUBITS == 4 and N_EDGES == 12
for v in VARIANTS_TO_RUN:
    assert v in VARIANTS, f"unknown variant {v}"

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RING_PAIRS = [(0, 1), (1, 2), (2, 3), (3, 0)]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]

print("=" * 110)
print(f"MESSAGE-PASSING ABLATION (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | variants {VARIANTS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE}")
print("=" * 110)

# ==============================================================================
# 2. CLOSED-FORM 4-QUBIT CIRCUIT  (identical to Cell 3; verified there against PennyLane)
# ==============================================================================
def circuit_core(theta, W0, W1, a1):
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0, o1, g1 = W0[:, 0], W0[:, 1], W1[:, 0], W1[:, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    ca, sa = jnp.cos(a1), jnp.sin(a1)
    z_next, z_prev = jnp.roll(z, -1, axis=1), jnp.roll(z, 1, axis=1)
    fr, fi = ca, -z_next * sa
    gr, gi = jnp.roll(ca, 1, axis=1), -z_prev * jnp.roll(sa, 1, axis=1)
    hr, hi = fr * gr - fi * gi, fr * gi + fi * gr
    wr, wi = x * hr + y * hi, x * hi - y * hr
    x, y = wr, -wi
    x, z = x * ct + z * st, z * ct - x * st
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

# ==============================================================================
# 3. PI-QGAT MODEL WITH SWITCHABLE MESSAGE PASSING  ("full" == Cell 3 exactly)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def uses_prior(spec):
    return spec["rounds"] > 0 and spec["aggr"] == "attn"

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(spec, seed):
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> "full" starts identical to Cell 3
    P = {"node": _init_lin(ks[0], N_FEATS, N_QUBITS),
         "head1": _init_lin(ks[3], N_QUBITS, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1)}
    if uses_prior(spec):
        P["phys"] = _init_lin(ks[2], 2, 1)
    if spec["rounds"] > 0:
        P["pair"] = _init_lin(ks[1], (2 if spec["pair_in"] == "both" else 1) * N_QUBITS, N_QUBITS)
        P["gate1"] = _init_lin(ks[4], N_QUBITS, len(RING_PAIRS))
        P["qw"] = (jax.random.uniform(ks[5], (2, N_QUBITS, 2), jnp.float32) * 2 - 1) * math.pi
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def prior_term(P, phi, ba):
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    return LAMBDA_PHYSICS_ATTN * Pp

def mp_round(P, h, prior, spec):
    """one round of message passing; returns (new h, neighbour part of message, quantum part of message, aggregated message)."""
    B = h.shape[0]
    h_i, h_j = h[:, RECV], h[:, SEND]
    pin = jnp.concatenate([h_i, h_j], -1) if spec["pair_in"] == "both" else h_j
    tf = (math.pi * jnp.tanh(_lin(P["pair"], pin))).reshape(-1, N_QUBITS)
    a1 = jax.nn.sigmoid(_lin(P["gate1"], tf)) * math.pi
    z = circuit_core(tf, P["qw"][0], P["qw"][1], a1).reshape(B, N_EDGES, N_QUBITS)
    nb = h_j if spec["msg"] == "h+z" else jnp.zeros_like(h_j)
    msg = (nb + z).reshape(B, N_NODES, N_IN, N_QUBITS)
    if spec["aggr"] == "attn":
        alpha = jax.nn.softmax((z.mean(-1) + prior).reshape(B, N_NODES, N_IN), -1)
        m = (alpha[..., None] * msg).sum(2)
    elif spec["aggr"] == "sum":
        m = msg.sum(2)
    else:
        m = msg.max(2)
    h_new = h + m if spec["residual"] else m
    return h_new, nb, z, m

def forward_parts(P, x, phi, ba, spec):
    h = _lin(P["node"], x)
    h0 = h
    prior = prior_term(P, phi, ba) if uses_prior(spec) else 0.0
    nb = z = m = None
    for _ in range(spec["rounds"]):
        h, nb, z, m = mp_round(P, h, prior, spec)
    g = h.mean(1) if spec["readout"] == "mean" else h.max(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    y = y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y
    return y, h0, nb, z, m

def forward(P, x, phi, ba, spec):
    return forward_parts(P, x, phi, ba, spec)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

class Compiled:
    def __init__(self, name):
        self.name, self.spec = name, dict(VARIANTS[name])
        spec = self.spec
        P0 = init_params(spec, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k == "qw" else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        unravel = self.unravel

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(lambda ww: _sl1(forward(unravel(ww), x, p_, b_, spec), y_))(w)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd_vec * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        @jax.jit
        def pred(w, x, phi, ba):
            return forward(unravel(w), x, phi, ba, spec)

        self.epoch, self.one_step, self.pred = epoch, one_step, pred
        if spec["rounds"] > 0:
            self.parts = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, spec)[1:])

# ==============================================================================
# 4. DATA + METRICS  (metrics identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([C.pred(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 5. TRAIN ONE (variant, seed)  -- Cell 3 loop incl. gamma escape and identical batch order
# ==============================================================================
def train_one(C, seed):
    w, _ = ravel_pytree(init_params(C.spec, seed)); n_par = int(w.shape[0])
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    has_q = C.spec["rounds"] > 0
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE and has_q:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{C.name} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}", flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes)

def evaluate_test(C, w):
    pred = predict_log(C, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    return m, bin_stats(pT_test64, pt_pred, pred, y_test)

def message_diag(C, w):
    h0, nb, z, m = C.parts(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS])     # last-round values
    nrm = lambda a: float(np.sqrt((np.asarray(a, np.float64) ** 2).sum(-1)).mean())
    return dict(norm_neighbour=nrm(nb), norm_quantum=nrm(z), norm_message=nrm(m), norm_node=nrm(h0),
                update_ratio=nrm(m) / max(nrm(h0), 1e-12))

# ==============================================================================
# 6. A) RETRAINING ABLATION
# ==============================================================================
ALL_MP = {}
for name in VARIANTS_TO_RUN:
    C = Compiled(name); spec = C.spec
    n_total = int(ravel_pytree(init_params(spec, 0))[0].shape[0])
    if spec["rounds"] == 0:
        desc = "no message passing: g = mean_i h_i"
    else:
        desc = (f"message = {'h_j + z_ij' if spec['msg'] == 'h+z' else 'z_ij'} | aggregation = "
                f"{ {'attn': 'attention-weighted sum', 'sum': 'plain sum', 'max': 'element-wise max'}[spec['aggr']] } | "
                f"update = {'h_i + m_i' if spec['residual'] else 'm_i'} | rounds = {spec['rounds']} | readout = {spec['readout']} | "
                f"edge input = {'[h_i, h_j]' if spec['pair_in'] == 'both' else '[h_j] (sender only)'}")
    print("\n" + "#" * 110 + f"\n# VARIANT: {name}   |   {n_total} params\n#   {desc}\n" + "#" * 110)
    res = []
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins = evaluate_test(C, tr["w"])
        diag = message_diag(C, tr["w"]) if (RUN_MESSAGE_DIAGNOSTICS and spec["rounds"] > 0) else None
        print(f"   >> {name} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  MSE {m['mse_log']:.4f}  "
              f"RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | "
              f"train {tr['train_total']:.0f}s")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_mp_{name}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        spec=spec, test=m, bins=bins, hist=tr["hist"], diag=diag))
    ALL_MP[name] = res

# ==============================================================================
# 7. A) REPORT
# ==============================================================================
REF = ALL_MP.get("full")
if REF is None:
    _qp = os.path.join(RESULTS_DIR, "piqgat_results_quantum.json")
    if os.path.exists(_qp):
        REF = json.load(open(_qp))["quantum"]; print(f"\n(full-model reference loaded from {_qp})")
ref_by_seed = {r["seed"]: r for r in REF} if REF else {}

def delta(rs, key):
    d = [r["test"][key] - ref_by_seed[r["seed"]]["test"][key] for r in rs if r["seed"] in ref_by_seed]
    return d if d else [float("nan")]

def ep2(r):
    best_v = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * best_v), np.nan)

order = (["full"] if "full" in ALL_MP else []) + [v for v in VARIANTS_TO_RUN if v != "full"]
W = 20
print("\n" + "=" * 162 + "\nA) MESSAGE-PASSING RETRAINING ABLATION  (test set; delta = variant minus full; for R2 negative = variant worse, "
      "for errors positive = variant worse)\n" + "=" * 162)
print(f"{'variant':<{W}}{'params':>7}{'R2_log':>18}{'dR2_log':>10}{'MAE_log':>18}{'dMAE_log':>10}{'MSE_log':>10}{'RMSE_log':>10}"
      f"{'MAE_pT':>17}{'dMAE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}{'ep->2%':>8}{'best ep':>8}")
for v in order:
    rs = ALL_MP[v]; g = lambda k: [r["test"][k] for r in rs]
    dd = lambda k, f: "-" if v == "full" else f.format(np.nanmean(delta(rs, k)))
    print(f"{v:<{W}}{rs[0]['n_par']:>7}{ms(g('r2_log')):>18}{dd('r2_log', '{:+.4f}'):>10}{ms(g('mae_log')):>18}{dd('mae_log', '{:+.4f}'):>10}"
          f"{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}{ms(g('mae_pt'), '{:.3f}'):>17}{dd('mae_pt', '{:+.3f}'):>10}"
          f"{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}{np.nanmean([ep2(r) for r in rs]):>8.1f}{np.mean([r['best_ep'] for r in rs]):>8.1f}")

print("\nCONVERGENCE (val R2_log at epoch, seed-mean) and TRAINING TIME:")
eps = [1, 2, 5, 10, 25, EPOCHS]
print(f"{'variant':<{W}}" + "".join(f"{'ep ' + str(e):>10}" for e in eps) + f"{'sec/epoch':>11}{'total s':>9}")
for v in order:
    rs = ALL_MP[v]
    print(f"{v:<{W}}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps)
          + f"{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}")

short = {"full": "full", "msg_z_only": "msg_z_only", "no_residual": "no_resid", "sum_aggregation": "sum_aggr",
         "max_aggregation": "max_aggr", "readout_max": "readout_max", "two_rounds": "2_rounds", "z_sender_only": "z_sender",
         "no_message_passing": "no_MP"}
for title, key, fmt in [("PER pT CLASS MAE_pT (GeV)", "mae_pt", "{:>12.3f}"), ("PER pT CLASS MAE_log", "mae_log", "{:>12.4f}")]:
    print(f"\n{title}, test, seed-mean:")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{short.get(v, v):>12}" for v in order))
    for b0 in ALL_MP[order[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            fmt.format(np.mean([[b for b in r['bins'] if b['lo'] == b0['lo']][0][key] for r in ALL_MP[v]])) for v in order))

# ==============================================================================
# 8. B) MESSAGE DIAGNOSTICS
# ==============================================================================
if RUN_MESSAGE_DIAGNOSTICS:
    print("\n" + "=" * 110 + f"\nB) MESSAGE DIAGNOSTICS  (first {DIAG_ROWS} test events, last message-passing round; mean L2 norms)\n" + "=" * 110)
    print(f"{'variant':<{W}}{'|h_i| (node)':>14}{'|h_j| part':>12}{'|z| part':>11}{'|m_i| (aggr.)':>15}{'|m|/|h| update':>16}")
    for v in order:
        ds = [r["diag"] for r in ALL_MP[v] if r.get("diag")]
        if not ds:
            continue
        g = lambda k: np.mean([d[k] for d in ds])
        print(f"{v:<{W}}{g('norm_node'):>14.3f}{g('norm_neighbour'):>12.3f}{g('norm_quantum'):>11.3f}{g('norm_message'):>15.3f}{g('update_ratio'):>16.3f}")
    print("(|h_j| part = 0 when the message has no neighbour state; |m|/|h| = how strongly message passing changes a node)")

# ==============================================================================
# 9. SAVE
# ==============================================================================
out = dict(variants=ALL_MP, seeds=SEEDS, specs=VARIANTS)
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")
print("HOW TO READ: msg_z_only ~ full -> the neighbour state in the message is not needed, the quantum term carries it.\n"
      "             z_sender_only << full -> the edge function must see BOTH stations (a pairwise comparison).\n"
      "             sum/max aggregation, readout_max, no_residual, two_rounds -> sensitivity to standard GNN design choices.\n"
      "             Gaps below ~0.01 R2 are single-seed noise.")

MESSAGE-PASSING ABLATION (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | variants ['full', 'msg_z_only', 'no_residual', 'sum_aggregation', 'max_aggregation', 'readout_max', 'two_rounds', 'z_sender_only', 'no_message_passing']
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001

##############################################################################################################
# VARIANT: full   |   144 params
#   message = h_j + z_ij | aggregation = attention-weighted sum | update = h_i + m_i | rounds = 1 | readout = mean | edge input = [h_i, h_j]
##############################################################################################################
[full | seed 42] ep  1/50  train 0.1842  val loss 0.1540  R2_log 0.5052  MAE_log 0.3613  MAE_pT 15.023 GeV   16.7s *best
[full | seed 42] ep  2/50  train 0.1359  val loss 0.1007  R2_log 0.7002  MAE_log 0.3057  MAE_pT 13.826 GeV    6.9s *best
[full | seed 42] ep  3/50  train 0.0806  val loss 0.0707  R2_log

In [ ]:
"""
================================================================================
CELL 14 -- QUANTUM CIRCUIT EXPERIMENTS for PI-QGAT  (22 qubits | 23 depth | 24 entanglement | 25 initialization)  [JAX]
Needs: Cell 0 (installs, incl. PennyLane) and Cell 1 (data). Independent of every other cell.

THE CIRCUIT (per edge; Cell 3 = n=4 qubits, depth L=2, ring IsingZZ, uniform(-pi,pi) init):
    RY(theta_k)  ->  for l = 1..L:  [RZ(gamma_lk) RY(omega_lk)] on every qubit
                                    if l < L:  entangler_l  ->  RY(theta_k) data re-upload
                 ->  <Z_k> on every qubit
    theta = pi*tanh(pair_proj[h_i, h_j]) (n angles), entangler angles a_l = pi*sigmoid(gate_l(theta)) (data dependent).
    The model scales with n: node_proj 7->n, pair_proj 2n->n, quantum output z has n values, head n->6->1
    (PI-QGAT ties the node dimension to the number of qubits).
    depth L = number of variational layers; L=1 has NO entangler, L=2 is Cell 3, L=3/4 add entangler+re-upload blocks.

SECTIONS (one change at a time from the Cell 3 reference, which is trained once and shared):
  22 NUMBER OF QUBITS : n = 2, 3, 4, 5, 6                          (depth 2, ring ZZ, uniform init)
  23 CIRCUIT DEPTH    : L = 1, 2, 3, 4                             (4 qubits, ring ZZ, uniform init)
  24 ENTANGLEMENT     : none | chain ZZ | ring ZZ | all-to-all ZZ | ring ZZ with FIXED (trained, data-independent)
                        angles | CNOT ring                          (4 qubits, depth 2, uniform init)
  25 INITIALISATION   : uniform(-pi,pi) | small uniform(-0.1,0.1) | normal(0,1) | zeros   (circuit angles only)

SIMULATORS (both exact; verified against PennyLane default.qubit below, and against each other):
  closed form   : one diagonal ZZ entangling layer acting on a product state -> each qubit's Bloch vector picks up a
                  phase factor from its ZZ neighbours (generalises Cell 3's formula to any n and any ZZ pair set). Fast.
  state vector  : full 2^n complex amplitudes; used for depth >= 3 and for CNOT entanglers (not diagonal).

REPORTED per configuration: test log1p(pT) R2/MAE/MSE/RMSE, pT MAE/MSE/RMSE (GeV), clamped rows, per-pT-class MAE_pT,
  convergence (val R2 at epochs 1/2/5/10/25/50, epochs-to-2%, best epoch), sec/epoch, inference us/row, params, and two
  circuit diagnostics: gradient norm of the circuit angles at initialisation (barren-plateau indicator) and the spread
  (std) of the circuit outputs <Z> on test edges (a near-constant circuit has std ~ 0).

RUN TIME (CPU, 50 epochs): closed-form runs ~4 min each, state-vector runs ~10-20 min each. All four sections =
  16 distinct trainings ~= 1.5-2 h. Run one section at a time with SECTIONS_TO_RUN, or lower QC_EPOCHS for a quick look.
================================================================================
"""
SEEDS = [SEED]
SECTIONS = {
    "22_qubits":       [dict(n=k, L=2, ent="ring", init="uniform_pi") for k in (2, 3, 4, 5, 6)],
    "23_depth":        [dict(n=4, L=k, ent="ring", init="uniform_pi") for k in (1, 2, 3, 4)],
    "24_entanglement": [dict(n=4, L=2, ent=e, init="uniform_pi") for e in ("none", "chain", "ring", "all_to_all", "ring_fixed", "cnot_ring")],
    "25_init":         [dict(n=4, L=2, ent="ring", init=i) for i in ("uniform_pi", "small", "normal", "zeros")],
}
SECTIONS_TO_RUN = list(SECTIONS)                 # e.g. ["22_qubits"]
QC_EPOCHS = 50                                   # Cell 3 uses 50
VERIFY_WITH_PENNYLANE = True
DIAG_ROWS = 65536
RESULTS_FILE = "piqgat_quantum_circuit_experiments.json"

# ==============================================================================
# 0. IMPORTS
# ==============================================================================
import os, math, time, json, warnings, subprocess, sys, itertools
warnings.filterwarnings("ignore")
import numpy as np

try:
    import jax
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=False)
    import jax
import jax.numpy as jnp
try:
    jax.config.update("jax_compilation_cache_dir", "/content/jax_cache" if os.path.isdir("/content") else "./jax_cache")
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
except Exception:
    pass
from jax.flatten_util import ravel_pytree

for _n in ["SEED", "X_train", "X_val", "X_test", "phi_train", "phi_val", "phi_test", "ba_train", "ba_val", "ba_test",
           "y_train", "y_val", "y_test", "pT_test", "edge_recv", "edge_send", "r_mean", "r_std",
           "Y_MEAN", "Y_STD", "PT_CLAMP_LO", "PHI_IS_RADIANS", "N_NODES", "N_FEATS", "N_EDGES"]:
    assert _n in globals(), f"'{_n}' missing -- run Cell 0 and Cell 1 first"
assert N_NODES == 4 and N_FEATS == 7 and N_EDGES == 12
for s in SECTIONS_TO_RUN:
    assert s in SECTIONS, f"unknown section {s}"

# ==============================================================================
# 1. HYPER-PARAMETERS  (identical to Cell 3)
# ==============================================================================
EPOCHS = QC_EPOCHS
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
CLASSICAL_WEIGHT_DECAY = 3e-5
QUANTUM_WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
SMOOTH_L1_BETA = 1.0
STANDARDIZE_OUTPUT = True
LAMBDA_PHYSICS_ATTN = 0.1
BOUND_PHYSICS_PRIOR = True
PHYSICS_BOUND_SCALE = 6.0
HEAD_HIDDEN = 6
GAMMA_ESCAPE = True
GAMMA_STUCK_TOL = 1e-3
GAMMA_ESCAPE_KICK = 0.3
EVAL_BATCH_SIZE = 65536
PT_BIN_EDGES = [0, 2, 4, 6, 10, 20, 50, 100, float("inf")]
RESULTS_DIR = "/content" if os.path.isdir("/content") else "."
RUN_DEVICE = jax.config.jax_default_device or jax.devices()[0]
REF_CFG = dict(n=4, L=2, ent="ring", init="uniform_pi")

def cfg_key(c):
    return f"q{c['n']}_d{c['L']}_{c['ent']}_{c['init']}"

def cfg_label(c):
    return f"{c['n']} qubits, depth {c['L']}, {c['ent'] if c['L'] > 1 else 'no entangler'}, init {c['init']}"

print("=" * 110)
print(f"QUANTUM CIRCUIT EXPERIMENTS (PI-QGAT) | JAX {jax.__version__} on {RUN_DEVICE} | seeds {SEEDS} | sections {SECTIONS_TO_RUN}")
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)} | epochs {EPOCHS}, batch {BATCH_SIZE}, lr {LEARNING_RATE} | "
      f"reference = Cell 3 circuit ({cfg_label(REF_CFG)})")
print("=" * 110)

# ==============================================================================
# 2. CIRCUIT FAMILY: entangler pairs, closed form, state vector
# ==============================================================================
def entangler_pairs(ent, n):
    if ent in ("ring", "ring_fixed", "cnot_ring"):
        return [(0, 1)] if n == 2 else [(k, (k + 1) % n) for k in range(n)]
    if ent == "chain":
        return [(k, k + 1) for k in range(n - 1)]
    if ent == "all_to_all":
        return list(itertools.combinations(range(n), 2))
    return []                                                            # "none"

def backend_of(c):
    return "closed" if (c["L"] <= 2 and c["ent"] != "cnot_ring") else "statevector"

def closed_form(theta, W, a, pairs):
    """theta (N,n), W (L<=2,n,2) = (omega, gamma), a (N,P) ZZ angles or None -> <Z_k> (N,n). Exact for one diagonal ZZ layer."""
    n = theta.shape[1]
    st, ct = jnp.sin(theta), jnp.cos(theta)
    o0, g0 = W[0, :, 0], W[0, :, 1]
    x = st * jnp.cos(g0); y = st * jnp.sin(g0); z = ct
    co0, so0 = jnp.cos(o0), jnp.sin(o0)
    x, z = x * co0 + z * so0, z * co0 - x * so0
    if W.shape[0] == 1:
        return z
    if a is not None and len(pairs):
        ca, sa = jnp.cos(a), jnp.sin(a)
        hr = [jnp.ones_like(z[:, 0]) for _ in range(n)]
        hi = [jnp.zeros_like(z[:, 0]) for _ in range(n)]
        for p, (u, v) in enumerate(pairs):                              # qubit q's coherence x - i y gets (cos a - i <Z_other> sin a)
            for q, other in ((u, v), (v, u)):
                fr, fi = ca[:, p], -z[:, other] * sa[:, p]
                hr[q], hi[q] = hr[q] * fr - hi[q] * fi, hr[q] * fi + hi[q] * fr
        hr, hi = jnp.stack(hr, 1), jnp.stack(hi, 1)
        x, y = x * hr + y * hi, -(x * hi - y * hr)
    x, z = x * ct + z * st, z * ct - x * st                             # data re-upload RY(theta)
    o1, g1 = W[1, :, 0], W[1, :, 1]
    x = x * jnp.cos(g1) - y * jnp.sin(g1)
    return z * jnp.cos(o1) - x * jnp.sin(o1)

_SV_CACHE = {}
def _sv_tables(n, pairs, cnot):
    key = (n, tuple(pairs), cnot)
    if key not in _SV_CACHE:
        idx = np.arange(2 ** n)
        bit = lambda q: (idx >> (n - 1 - q)) & 1
        zs = np.stack([1 - 2 * bit(q) for q in range(n)], 1).astype(np.float32)                      # (2^n, n)
        zz = (np.stack([(1 - 2 * bit(u)) * (1 - 2 * bit(v)) for u, v in pairs]).astype(np.float32)
              if (pairs and not cnot) else np.zeros((1, 2 ** n), np.float32))                          # (P, 2^n)
        perms = [idx ^ (bit(c) << (n - 1 - t)) for c, t in pairs] if cnot else []
        _SV_CACHE[key] = (jnp.asarray(zs), jnp.asarray(zz), [jnp.asarray(p) for p in perms])
    return _SV_CACHE[key]

def _apply_1q(psi, U, k, n):
    N = psi.shape[0]
    r = psi.reshape(N, 2 ** k, 2, 2 ** (n - k - 1))
    out = jnp.einsum("nij,najb->naib" if U.ndim == 3 else "ij,najb->naib", U, r)
    return out.reshape(N, 2 ** n)

def statevector(theta, W, a_list, pairs, cnot):
    """exact 2^n-amplitude simulation of the same circuit family, any depth, ZZ or CNOT entanglers -> <Z_k> (N,n)."""
    N, n = theta.shape
    zs, zz, perms = _sv_tables(n, pairs, cnot)
    c, s = jnp.cos(theta / 2), jnp.sin(theta / 2)
    v = jnp.stack([c, s], -1)
    psi = v[:, 0, :]
    for k in range(1, n):
        psi = (psi[:, :, None] * v[:, k, None, :]).reshape(N, -1)
    psi = psi.astype(jnp.complex64)
    ry_data = jnp.stack([jnp.stack([c, -s], -1), jnp.stack([s, c], -1)], -2).astype(jnp.complex64)      # (N, n, 2, 2)
    L = W.shape[0]
    for l in range(L):
        for k in range(n):
            om, ga = W[l, k, 0] / 2, W[l, k, 1] / 2
            ry = jnp.stack([jnp.stack([jnp.cos(om), -jnp.sin(om)]), jnp.stack([jnp.sin(om), jnp.cos(om)])]).astype(jnp.complex64)
            rz = jnp.diag(jnp.stack([jnp.exp(-1j * ga), jnp.exp(1j * ga)])).astype(jnp.complex64)
            psi = _apply_1q(psi, ry @ rz, k, n)
        if l < L - 1:
            if cnot:
                for pm in perms:
                    psi = psi[:, pm]
            elif len(pairs):
                psi = psi * jnp.exp(-0.5j * (a_list[l] @ zz).astype(jnp.complex64))
            for k in range(n):
                psi = _apply_1q(psi, ry_data[:, k], k, n)
    return (psi.real ** 2 + psi.imag ** 2) @ zs

# ==============================================================================
# 3. PI-QGAT MODEL WITH A CONFIGURABLE CIRCUIT  (reference config == Cell 3 model)
# ==============================================================================
RECV, SEND = edge_recv.numpy(), edge_send.numpy()
N_IN = N_NODES - 1
RM = jnp.asarray(r_mean.numpy().astype(np.float32)); RS = jnp.asarray(r_std.numpy().astype(np.float32))

def _init_lin(key, n_in, n_out):
    k1, k2 = jax.random.split(key); b = 1.0 / math.sqrt(n_in)
    return {"W": jax.random.uniform(k1, (n_out, n_in), jnp.float32, -b, b),
            "b": jax.random.uniform(k2, (n_out,), jnp.float32, -b, b)}

def init_params(c, seed):
    n, L, ent = c["n"], c["L"], c["ent"]
    pairs = entangler_pairs(ent, n)
    ks = jax.random.split(jax.random.PRNGKey(seed), 8)      # same key layout as Cell 3 -> reference starts identical to Cell 3
    P = {"node": _init_lin(ks[0], N_FEATS, n), "phys": _init_lin(ks[2], 2, 1),
         "head1": _init_lin(ks[3], n, HEAD_HIDDEN), "head2": _init_lin(jax.random.fold_in(ks[3], 1), HEAD_HIDDEN, 1),
         "pair": _init_lin(ks[1], 2 * n, n)}
    if L >= 2 and ent in ("ring", "chain", "all_to_all"):
        for l in range(L - 1):                                           # one data-dependent angle generator per entangler
            P["gate1" if l == 0 else f"gate{l + 1}"] = _init_lin(ks[4] if l == 0 else jax.random.fold_in(ks[4], l), n, len(pairs))
    if L >= 2 and ent == "ring_fixed":
        P["ent_c"] = jax.random.uniform(ks[6], (L - 1, len(pairs)), jnp.float32, -1.0, 1.0)
    u = jax.random.uniform(ks[5], (L, n, 2), jnp.float32)
    P["qw"] = {"uniform_pi": (u * 2 - 1) * math.pi, "small": (u * 2 - 1) * 0.1,
               "normal": jax.random.normal(ks[5], (L, n, 2), jnp.float32), "zeros": jnp.zeros((L, n, 2), jnp.float32)}[c["init"]]
    return P

_lin = lambda p, x: x @ p["W"].T + p["b"]

def circuit(P, tf, c):
    n, L, ent = c["n"], c["L"], c["ent"]
    pairs = entangler_pairs(ent, n)
    a_list = [None] * max(L - 1, 0)
    if L >= 2 and ent in ("ring", "chain", "all_to_all"):
        a_list = [jax.nn.sigmoid(_lin(P["gate1" if l == 0 else f"gate{l + 1}"], tf)) * math.pi for l in range(L - 1)]
    elif L >= 2 and ent == "ring_fixed":
        a_list = [jnp.broadcast_to(jax.nn.sigmoid(P["ent_c"][l]) * math.pi, (tf.shape[0], len(pairs))) for l in range(L - 1)]
    if backend_of(c) == "closed":
        return closed_form(tf, P["qw"], a_list[0] if a_list else None, pairs)
    return statevector(tf, P["qw"], a_list, pairs, ent == "cnot_ring")

def forward_parts(P, x, phi, ba, c):
    n = c["n"]; B = x.shape[0]
    h = _lin(P["node"], x)
    h_i, h_j = h[:, RECV], h[:, SEND]
    d = phi[:, SEND] - phi[:, RECV]
    dphi = jnp.arctan2(jnp.sin(d), jnp.cos(d)) if PHI_IS_RADIANS else d
    r = jnp.stack([jnp.abs(ba[:, SEND] - ba[:, RECV]), jnp.abs(dphi)], -1)
    Pp = _lin(P["phys"], (r - RM) / RS)[..., 0]
    if BOUND_PHYSICS_PRIOR:
        Pp = PHYSICS_BOUND_SCALE * jnp.tanh(Pp / PHYSICS_BOUND_SCALE)
    tf = (math.pi * jnp.tanh(_lin(P["pair"], jnp.concatenate([h_i, h_j], -1)))).reshape(-1, n)
    z = circuit(P, tf, c).reshape(B, N_EDGES, n)
    s = z.mean(-1) + LAMBDA_PHYSICS_ATTN * Pp
    alpha = jax.nn.softmax(s.reshape(B, N_NODES, N_IN), -1)
    m = (alpha[..., None] * (h_j + z).reshape(B, N_NODES, N_IN, n)).sum(2)
    g = (h + m).mean(1)
    y = _lin(P["head2"], jnp.tanh(_lin(P["head1"], g)))[:, 0]
    return (y * Y_STD + Y_MEAN if STANDARDIZE_OUTPUT else y), z

def forward(P, x, phi, ba, c):
    return forward_parts(P, x, phi, ba, c)[0]

def _sl1(p, y):
    d = jnp.abs(p - y)
    return jnp.mean(jnp.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA))

QUANTUM_KEYS = ("qw", "ent_c")

class Compiled:
    def __init__(self, c):
        self.c = dict(c); c = self.c
        P0 = init_params(c, 0)
        _, self.unravel = ravel_pytree(P0)
        wd = {k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, QUANTUM_WEIGHT_DECAY if k in QUANTUM_KEYS else CLASSICAL_WEIGHT_DECAY), v)
              for k, v in P0.items()}
        wd_vec = ravel_pytree(wd)[0]
        self.qmask = ravel_pytree({k: jax.tree_util.tree_map(lambda a, k=k: jnp.full_like(a, 1.0 if k == "qw" else 0.0), v)
                                   for k, v in P0.items()})[0]
        unravel = self.unravel
        loss_fn = lambda ww, x, p_, b_, y_: _sl1(forward(unravel(ww), x, p_, b_, c), y_)

        def step(carry, batch):
            w, mm, vv, t = carry; x, p_, b_, y_ = batch
            loss, g = jax.value_and_grad(loss_fn)(w, x, p_, b_, y_)
            g = g * jnp.minimum(GRAD_CLIP_NORM / (jnp.sqrt(jnp.sum(g * g)) + 1e-6), 1.0)
            g = g + wd_vec * w
            t = t + 1.0
            mm = 0.9 * mm + 0.1 * g; vv = 0.999 * vv + 0.001 * g * g
            w = w - LEARNING_RATE * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8)
            return (w, mm, vv, t), loss

        @jax.jit
        def epoch(w, mm, vv, t, Xb, pb, bb, yb):
            (w, mm, vv, t), ls = jax.lax.scan(step, (w, mm, vv, t), (Xb, pb, bb, yb))
            return w, mm, vv, t, ls.mean()

        @jax.jit
        def one_step(w, mm, vv, t, x, p_, b_, y_):
            (w, mm, vv, t), l = step((w, mm, vv, t), (x, p_, b_, y_))
            return w, mm, vv, t, l

        self.epoch, self.one_step = epoch, one_step
        self.pred = jax.jit(lambda w, x, phi, ba: forward(unravel(w), x, phi, ba, c))
        self.zout = jax.jit(lambda w, x, phi, ba: forward_parts(unravel(w), x, phi, ba, c)[1])
        self.grad = jax.jit(jax.grad(loss_fn))

# ==============================================================================
# 4. VERIFICATION AGAINST PENNYLANE
# ==============================================================================
def pennylane_reference(c, th, W, a_list):
    import pennylane as qml
    n, L = c["n"], c["L"]; pairs = entangler_pairs(c["ent"], n)
    dev = qml.device("default.qubit", wires=n)
    @qml.qnode(dev)
    def qc():
        for k in range(n): qml.RY(th[:, k], wires=k)
        for l in range(L):
            for k in range(n): qml.RZ(W[l, k, 1], wires=k); qml.RY(W[l, k, 0], wires=k)
            if l < L - 1:
                if c["ent"] == "cnot_ring":
                    for u, v in pairs: qml.CNOT(wires=[u, v])
                else:
                    for p, (u, v) in enumerate(pairs): qml.IsingZZ(a_list[l][:, p], wires=[u, v])
                for k in range(n): qml.RY(th[:, k], wires=k)
        return [qml.expval(qml.PauliZ(k)) for k in range(n)]
    return np.stack([np.asarray(o) for o in qc()], -1)

def verify(configs):
    print("-" * 110 + "\nVERIFY circuits against PennyLane default.qubit (float64) -- every distinct (qubits, depth, entangler) used\n" + "-" * 110)
    seen = set(); rs = np.random.default_rng(7)
    for c in configs:
        k = (c["n"], c["L"], c["ent"])
        if k in seen:
            continue
        seen.add(k)
        n, L = c["n"], c["L"]; pairs = entangler_pairs(c["ent"], n)
        th = rs.normal(size=(32, n)) * 1.5; W = rs.normal(size=(L, n, 2)) * 2
        a_list = [rs.uniform(0, math.pi, (32, len(pairs))) for _ in range(max(L - 1, 0))] if (L >= 2 and c["ent"] != "cnot_ring") else [None] * max(L - 1, 0)
        ref = pennylane_reference(c, th, W, a_list)
        thj, Wj = jnp.asarray(th, jnp.float32), jnp.asarray(W, jnp.float32)
        alj = [jnp.asarray(a, jnp.float32) if a is not None else None for a in a_list]
        out = {}
        if backend_of(c) == "closed":
            out["closed form"] = np.asarray(closed_form(thj, Wj, alj[0] if alj else None, pairs if (L >= 2 and c["ent"] != "none") else []))
        sv_pairs = pairs if c["ent"] != "none" else []
        out["state vector"] = np.asarray(statevector(thj, Wj, alj, sv_pairs, c["ent"] == "cnot_ring"))
        msg = "   ".join(f"max|PL - {nm}| = {np.abs(ref - o).max():.1e}" for nm, o in out.items())
        print(f"  {n} qubits, depth {L}, {c['ent']:<11} {msg}")
        for nm, o in out.items():
            assert np.abs(ref - o).max() < 5e-5, f"{nm} disagrees with PennyLane for {c}"
    print("  -> PASS\n")

# ==============================================================================
# 5. DATA + METRICS  (metrics identical to Cell 3)
# ==============================================================================
_f32 = lambda a: jnp.asarray(a, dtype=jnp.float32)
Xtr, ptr, btr, ytr = _f32(X_train), _f32(phi_train), _f32(ba_train), _f32(y_train)
Xva, pva, bva = _f32(X_val), _f32(phi_val), _f32(ba_val)
Xte, pte, bte = _f32(X_test), _f32(phi_test), _f32(ba_test)
N_TR = Xtr.shape[0]; NB = N_TR // BATCH_SIZE; N_TAIL = N_TR - NB * BATCH_SIZE
pT_val = np.expm1(y_val.astype(np.float64)); pT_test64 = pT_test.astype(np.float64)

def predict_log(C, w, X, P_, B_, bs=EVAL_BATCH_SIZE):
    out = jnp.concatenate([C.pred(w, X[i:i + bs], P_[i:i + bs], B_[i:i + bs]) for i in range(0, X.shape[0], bs)])
    out.block_until_ready()
    return np.asarray(out)

def split_metrics(pred_log, y_log, pt_true):
    pl, yl = pred_log.astype(np.float64), y_log.astype(np.float64)
    e = pl - yl
    raw = np.expm1(pl); n_neg = int((raw < PT_CLAMP_LO).sum())
    pt_pred = np.clip(raw, PT_CLAMP_LO, None)
    et = pt_pred - pt_true
    m = dict(r2_log=1 - (e ** 2).sum() / ((yl - yl.mean()) ** 2).sum(), mae_log=np.abs(e).mean(), mse_log=(e ** 2).mean(),
             rmse_log=math.sqrt((e ** 2).mean()), mae_pt=np.abs(et).mean(), mse_pt=(et ** 2).mean(),
             rmse_pt=math.sqrt((et ** 2).mean()), n_clamped=n_neg)
    d = np.abs(e); m["loss"] = float(np.mean(np.where(d < SMOOTH_L1_BETA, 0.5 * d * d / SMOOTH_L1_BETA, d - 0.5 * SMOOTH_L1_BETA)))
    return {k: float(v) for k, v in m.items()}, pt_pred

def bin_stats(pt_true, pt_pred, pred_log, y_log):
    rows = []; sse_tot = float(((pt_pred - pt_true) ** 2).sum())
    for lo, hi in zip(PT_BIN_EDGES[:-1], PT_BIN_EDGES[1:]):
        msk = (pt_true >= lo) & (pt_true < hi); n = int(msk.sum())
        if n == 0:
            continue
        res = pt_pred[msk] - pt_true[msk]; rel = res / np.maximum(pt_true[msk], 1e-8)
        rows.append(dict(lo=lo, hi=hi, n=n, frac=100.0 * n / len(pt_true),
                         mae_log=float(np.abs(pred_log[msk].astype(np.float64) - y_log[msk]).mean()),
                         mae_pt=float(np.abs(res).mean()), mse_pt=float((res ** 2).mean()), rmse_pt=float(math.sqrt((res ** 2).mean())),
                         med_rel=float(np.median(np.abs(rel))), bias=float(rel.mean()), sse_share=100.0 * float((res ** 2).sum()) / sse_tot))
    return rows

def _label(lo, hi):
    return f"[{lo:g}, {'inf' if hi == float('inf') else format(hi, 'g')})"

def ms(vals, f="{:.4f}"):
    v = np.asarray(vals, dtype=float)
    return (f + " +- " + f).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)

# ==============================================================================
# 6. TRAIN / EVALUATE ONE CONFIGURATION
# ==============================================================================
def init_grad_norm(C, w, seed, n_batches=8):
    rng = np.random.default_rng(10_000 + seed); idx = rng.permutation(N_TR)[:n_batches * BATCH_SIZE].reshape(n_batches, BATCH_SIZE)
    gq, ga = [], []
    for b in idx:
        b = jnp.asarray(b)
        g = C.grad(w, Xtr[b], ptr[b], btr[b], ytr[b])
        gq.append(float(jnp.sqrt(jnp.sum((g * C.qmask) ** 2)))); ga.append(float(jnp.sqrt(jnp.sum(g * g))))
    return float(np.mean(gq)), float(np.mean(ga))

def train_one(C, seed):
    w, _ = ravel_pytree(init_params(C.c, seed)); n_par = int(w.shape[0])
    g_q0, g_all0 = init_grad_norm(C, w, seed)
    mm, vv, t = jnp.zeros_like(w), jnp.zeros_like(w), jnp.float32(0.0)
    rng = np.random.default_rng(seed)
    nfull = NB * BATCH_SIZE
    shp = lambda a: a[:nfull].reshape((NB, BATCH_SIZE) + a.shape[1:])
    hist, best, best_w, best_ep, escapes = [], float("inf"), w, 0, 0
    tag = cfg_key(C.c)
    t_start = time.time()
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        perm = jnp.asarray(rng.permutation(N_TR))
        Xp, pp, bp, yp = Xtr[perm], ptr[perm], btr[perm], ytr[perm]
        w, mm, vv, t, tl = C.epoch(w, mm, vv, t, shp(Xp), shp(pp), shp(bp), shp(yp))
        if N_TAIL:
            w, mm, vv, t, _ = C.one_step(w, mm, vv, t, Xp[nfull:], pp[nfull:], bp[nfull:], yp[nfull:])
        w.block_until_ready(); t_tr = time.time() - t0
        if GAMMA_ESCAPE:
            P = C.unravel(w); qw = np.array(P["qw"]); stuck = np.abs(np.sin(qw[..., 1])) < GAMMA_STUCK_TOL
            if stuck.any():
                kk = rng.random(qw[..., 1].shape) * 2 - 1; kk = np.where(np.abs(kk) < 0.1, 0.5, kk)
                qw[..., 1] = np.where(stuck, qw[..., 1] + GAMMA_ESCAPE_KICK * kk, qw[..., 1])
                P["qw"] = jnp.asarray(qw, dtype=jnp.float32); w = ravel_pytree(P)[0]; escapes += int(stuck.sum())
        vm, _ = split_metrics(predict_log(C, w, Xva, pva, bva), y_val, pT_val)
        sec = time.time() - t0
        hist.append(dict(ep=ep, train_loss=float(tl), val_loss=vm["loss"], val_r2_log=vm["r2_log"], val_mae_log=vm["mae_log"],
                         val_mae_pt=vm["mae_pt"], sec=sec, train_sec=t_tr))
        flag = ""
        if vm["loss"] < best - 1e-6:
            best, best_w, best_ep = vm["loss"], jnp.array(w), ep; flag = " *best"
        print(f"[{tag} | seed {seed}] ep {ep:2d}/{EPOCHS}  train {float(tl):.4f}  val loss {vm['loss']:.4f}  "
              f"R2_log {vm['r2_log']:.4f}  MAE_log {vm['mae_log']:.4f}  MAE_pT {vm['mae_pt']:.3f} GeV  {sec:5.1f}s{flag}", flush=True)
    return dict(w=best_w, hist=hist, best_ep=best_ep, n_par=n_par, train_total=time.time() - t_start, escapes=escapes,
                init_grad_qw=g_q0, init_grad_all=g_all0)

def evaluate(C, w):
    pred = predict_log(C, w, Xte, pte, bte)
    m, pt_pred = split_metrics(pred, y_test, pT_test64)
    bins = bin_stats(pT_test64, pt_pred, pred, y_test)
    z = np.asarray(C.zout(w, Xte[:DIAG_ROWS], pte[:DIAG_ROWS], bte[:DIAG_ROWS]), np.float64).reshape(-1, C.c["n"])
    full = lambda: predict_log(C, w, Xte, pte, bte)
    full(); t0 = time.perf_counter(); full(); us = 1e6 * (time.perf_counter() - t0) / Xte.shape[0]
    return m, bins, dict(z_std=float(z.std(0).mean()), z_absmean=float(np.abs(z).mean()), us_per_row=us)

# ==============================================================================
# 7. RUN ALL REQUESTED SECTIONS  (each distinct configuration trained once, shared across sections)
# ==============================================================================
CONFIGS = []
for s in SECTIONS_TO_RUN:
    for c in SECTIONS[s]:
        if cfg_key(c) not in [cfg_key(x) for x in CONFIGS]:
            CONFIGS.append(c)
if VERIFY_WITH_PENNYLANE:
    verify(CONFIGS)

RUNS = {}
for c in CONFIGS:
    C = Compiled(c); key = cfg_key(c)
    n_total = int(ravel_pytree(init_params(c, 0))[0].shape[0])
    print("\n" + "#" * 110 + f"\n# CONFIG: {cfg_label(c)}   |   {n_total} params   |   simulator: {backend_of(c)}"
          + ("   (= Cell 3 reference)" if key == cfg_key(REF_CFG) else "") + "\n" + "#" * 110)
    res = []
    for seed in SEEDS:
        tr = train_one(C, seed)
        m, bins, diag = evaluate(C, tr["w"])
        print(f"   >> {key} | best epoch {tr['best_ep']} | TEST log: R2 {m['r2_log']:.4f}  MAE {m['mae_log']:.4f}  MSE {m['mse_log']:.4f}  "
              f"RMSE {m['rmse_log']:.4f} | pT: MAE {m['mae_pt']:.3f} GeV  RMSE {m['rmse_pt']:.3f} | clamped {m['n_clamped']:.0f} | "
              f"train {tr['train_total']:.0f}s | init |grad qw| {tr['init_grad_qw']:.2e} | <Z> std {diag['z_std']:.3f} | {diag['us_per_row']:.2f} us/row")
        np.savez(os.path.join(RESULTS_DIR, f"piqgat_qc_{key}_seed{seed}.npz"), w=np.asarray(tr["w"]))
        res.append(dict(seed=seed, best_ep=tr["best_ep"], n_par=tr["n_par"], train_total_s=tr["train_total"], escapes=tr["escapes"],
                        init_grad_qw=tr["init_grad_qw"], init_grad_all=tr["init_grad_all"], cfg=c, backend=backend_of(c),
                        test=m, bins=bins, hist=tr["hist"], diag=diag))
    RUNS[key] = res

# ==============================================================================
# 8. REPORT PER SECTION
# ==============================================================================
def ep2(r):
    best_v = min(x["val_loss"] for x in r["hist"])
    return next((x["ep"] for x in r["hist"] if x["val_loss"] <= 1.02 * best_v), np.nan)

REF = RUNS.get(cfg_key(REF_CFG))
ref_by_seed = {r["seed"]: r for r in REF} if REF else {}
def delta(rs, key):
    d = [r["test"][key] - ref_by_seed[r["seed"]]["test"][key] for r in rs if r["seed"] in ref_by_seed]
    return d if d else [float("nan")]

TITLES = {"22_qubits": ("22. NUMBER OF QUBITS", lambda c: f"{c['n']} qubits"),
          "23_depth": ("23. CIRCUIT DEPTH (variational layers)", lambda c: f"depth {c['L']}"),
          "24_entanglement": ("24. ENTANGLEMENT STRATEGY", lambda c: c["ent"]),
          "25_init": ("25. CIRCUIT-PARAMETER INITIALISATION", lambda c: c["init"])}
W = 14
for s in SECTIONS_TO_RUN:
    title, lab = TITLES[s]
    keys = [cfg_key(c) for c in SECTIONS[s]]
    print("\n" + "=" * 160 + f"\n{title}   (test set; delta vs the Cell 3 reference: {cfg_label(REF_CFG)})\n" + "=" * 160)
    print(f"{'setting':<{W}}{'params':>7}{'sim':>7}{'R2_log':>18}{'dR2_log':>10}{'MAE_log':>18}{'dMAE_log':>10}{'MSE_log':>10}{'RMSE_log':>10}"
          f"{'MAE_pT':>17}{'dMAE_pT':>10}{'RMSE_pT':>10}{'clamped':>9}")
    for c, k in zip(SECTIONS[s], keys):
        rs = RUNS[k]; g = lambda kk: [r["test"][kk] for r in rs]
        isref = k == cfg_key(REF_CFG)
        dd = lambda kk, f: "ref" if isref else f.format(np.nanmean(delta(rs, kk)))
        print(f"{lab(c):<{W}}{rs[0]['n_par']:>7}{('cf' if rs[0]['backend'] == 'closed' else 'sv'):>7}{ms(g('r2_log')):>18}{dd('r2_log', '{:+.4f}'):>10}"
              f"{ms(g('mae_log')):>18}{dd('mae_log', '{:+.4f}'):>10}{np.mean(g('mse_log')):>10.4f}{np.mean(g('rmse_log')):>10.4f}"
              f"{ms(g('mae_pt'), '{:.3f}'):>17}{dd('mae_pt', '{:+.3f}'):>10}{np.mean(g('rmse_pt')):>10.3f}{np.mean(g('n_clamped')):>9.0f}")
    print(f"\n  cost and circuit diagnostics:")
    print(f"{'setting':<{W}}{'sec/epoch':>11}{'total s':>9}{'us/row':>9}{'ep->2%':>8}{'best ep':>8}{'init |grad qw|':>16}{'init |grad all|':>17}"
          f"{'<Z> std':>9}{'mean|<Z>|':>11}{'gamma kicks':>13}")
    for c, k in zip(SECTIONS[s], keys):
        rs = RUNS[k]
        print(f"{lab(c):<{W}}{np.mean([np.median([h['sec'] for h in r['hist']]) for r in rs]):>11.2f}{np.mean([r['train_total_s'] for r in rs]):>9.0f}"
              f"{np.mean([r['diag']['us_per_row'] for r in rs]):>9.2f}{np.nanmean([ep2(r) for r in rs]):>8.1f}{np.mean([r['best_ep'] for r in rs]):>8.1f}"
              f"{np.mean([r['init_grad_qw'] for r in rs]):>16.2e}{np.mean([r['init_grad_all'] for r in rs]):>17.2e}"
              f"{np.mean([r['diag']['z_std'] for r in rs]):>9.3f}{np.mean([r['diag']['z_absmean'] for r in rs]):>11.3f}"
              f"{np.mean([r['escapes'] for r in rs]):>13.0f}")
    print(f"\n  convergence (val R2_log at epoch, seed-mean):")
    eps = [e for e in (1, 2, 5, 10, 25, EPOCHS) if e <= EPOCHS]
    print(f"{'setting':<{W}}" + "".join(f"{'ep ' + str(e):>10}" for e in eps))
    for c, k in zip(SECTIONS[s], keys):
        rs = RUNS[k]
        print(f"{lab(c):<{W}}" + "".join(f"{np.mean([r['hist'][e-1]['val_r2_log'] for r in rs]):>10.4f}" for e in eps))
    print(f"\n  per pT class MAE_pT (GeV), test, seed-mean:")
    print(f"{'pT class (GeV)':>16}" + "".join(f"{lab(c):>14}" for c in SECTIONS[s]))
    for b0 in RUNS[keys[0]][0]["bins"]:
        print(f"{_label(b0['lo'], b0['hi']):>16}" + "".join(
            f"{np.mean([[b for b in r['bins'] if b['lo'] == b0['lo']][0]['mae_pt'] for r in RUNS[k]]):>14.3f}" for k in keys))

# ==============================================================================
# 9. SAVE
# ==============================================================================
out = dict(runs=RUNS, sections={s: [cfg_key(c) for c in SECTIONS[s]] for s in SECTIONS_TO_RUN}, seeds=SEEDS, epochs=EPOCHS,
           reference=cfg_key(REF_CFG))
_p = os.path.join(RESULTS_DIR, RESULTS_FILE)
json.dump(out, open(_p, "w"), default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))
print(f"\nresults saved -> {_p}")
print("HOW TO READ: params change with qubits (node dim = n) -- compare accuracy AND params.  init |grad qw| shrinking with qubits/depth\n"
      "             = barren-plateau tendency.  <Z> std ~ 0 = the circuit output is nearly constant (not expressive on this data).\n"
      "             'none' ~ 'ring' -> entanglement adds nothing here; ring_fixed vs ring -> value of data-dependent entangling angles.\n"
      "             Gaps below ~0.01 R2 are single-seed noise.")

QUANTUM CIRCUIT EXPERIMENTS (PI-QGAT) | JAX 0.11.1 on cpu:0 | seeds [42] | sections ['22_qubits', '23_depth', '24_entanglement', '25_init']
Train 235871 | Val 235871 | Test 707614 | epochs 50, batch 64, lr 0.001 | reference = Cell 3 circuit (4 qubits, depth 2, ring, init uniform_pi)
--------------------------------------------------------------------------------------------------------------
VERIFY circuits against PennyLane default.qubit (float64) -- every distinct (qubits, depth, entangler) used
--------------------------------------------------------------------------------------------------------------
  2 qubits, depth 2, ring        max|PL - closed form| = 2.1e-07   max|PL - state vector| = 3.5e-07
  3 qubits, depth 2, ring        max|PL - closed form| = 2.1e-07   max|PL - state vector| = 3.3e-07
  4 qubits, depth 2, ring        max|PL - closed form| = 1.8e-07   max|PL - state vector| = 5.2e-07
  5 qubits, depth 2, ring        max|PL - closed form| = 2.2e-07   max|PL - state vect